In [ ]:
import cv2 as cv
import numpy as np
from scipy.spatial import KDTree
from collections import deque
import random

# Ruta al video
video_path = "../../../data/raw/Prueba2.mp4"

# Abrir el video
cap = cv.VideoCapture(video_path)

if not cap.isOpened():
    raise IOError(f"No se pudo abrir el video: {video_path}")

# Obtener información del video
total_frames = int(cap.get(cv.CAP_PROP_FRAME_COUNT))
fps = cap.get(cv.CAP_PROP_FPS)

print(f"Frames totales: {total_frames}")
print(f"FPS: {fps:.2f}")

window_name = "Video"
sobel_window = "Sobel"
canny_window = "Canny"
hough_filtered_window = "Saddle Points"
contour_window = "Todos los contornos"
contour_filtered_window = "Contornos con puntos de silla"
approx_window = "Polígonos aproximados"
quadrilaterals_window = "Cuadriláteros detectados"
mesh_window = "Malla más grande"
estimated_board_window = "Tablero estimado (mejor warp)"
warp_window = "Mejor Warp"

cv.namedWindow(window_name, cv.WINDOW_NORMAL)
cv.namedWindow(sobel_window, cv.WINDOW_NORMAL)
cv.namedWindow(canny_window, cv.WINDOW_NORMAL)
cv.namedWindow(hough_filtered_window, cv.WINDOW_NORMAL)
cv.namedWindow(contour_window, cv.WINDOW_NORMAL)
cv.namedWindow(contour_filtered_window, cv.WINDOW_NORMAL)
cv.namedWindow(approx_window, cv.WINDOW_NORMAL)
cv.namedWindow(quadrilaterals_window, cv.WINDOW_NORMAL)
cv.namedWindow(mesh_window, cv.WINDOW_NORMAL)
cv.namedWindow(estimated_board_window, cv.WINDOW_NORMAL)
cv.namedWindow(warp_window, cv.WINDOW_NORMAL)

# Variable para detectar cambios del slider
current_frame = 0

def on_trackbar(pos):
    global current_frame
    current_frame = pos

# Crear la trackbar
cv.createTrackbar(
    "Frame",
    window_name,
    0,
    total_frames - 1,
    on_trackbar
)

last_frame = -1

# Tolerancia fija
TOLERANCE = 5

# Área mínima para considerar un cuadrilátero
MIN_AREA = 10

# Distancia máxima para considerar que dos vértices son adyacentes
ADJACENT_THRESHOLD = 40

# Lista para almacenar todos los cuadriláteros detectados
all_quadrilaterals = []

def find_largest_mesh(quadrilaterals, threshold=ADJACENT_THRESHOLD):
    """
    Encuentra la malla más grande de cuadriláteros adyacentes.
    Retorna la lista de cuadriláteros que forman la malla más grande.
    """
    if len(quadrilaterals) == 0:
        return []
    
    def get_vertices(polygon):
        vertices = []
        for point in polygon:
            vertices.append((int(point[0][0]), int(point[0][1])))
        return vertices
    
    def are_adjacent(poly1, poly2, threshold):
        vertices1 = get_vertices(poly1)
        vertices2 = get_vertices(poly2)
        
        close_points = 0
        for v1 in vertices1:
            for v2 in vertices2:
                dist = np.sqrt((v1[0] - v2[0])**2 + (v1[1] - v2[1])**2)
                if dist <= threshold:
                    close_points += 1
                    break
        
        return close_points >= 2
    
    n = len(quadrilaterals)
    adjacency = [[] for _ in range(n)]
    
    for i in range(n):
        for j in range(i+1, n):
            if are_adjacent(quadrilaterals[i], quadrilaterals[j], threshold):
                adjacency[i].append(j)
                adjacency[j].append(i)
    
    visited = [False] * n
    largest_mesh = []
    
    for i in range(n):
        if not visited[i]:
            queue = deque([i])
            visited[i] = True
            current_mesh = []
            
            while queue:
                current = queue.popleft()
                current_mesh.append(current)
                
                for neighbor in adjacency[current]:
                    if not visited[neighbor]:
                        visited[neighbor] = True
                        queue.append(neighbor)
            
            if len(current_mesh) > len(largest_mesh):
                largest_mesh = current_mesh
    
    return [quadrilaterals[idx] for idx in largest_mesh]

def get_vertices_as_points(polygon):
    """Convierte un polígono a lista de puntos (x,y) en orden"""
    vertices = []
    for point in polygon:
        vertices.append((int(point[0][0]), int(point[0][1])))
    return vertices

def fit_polygon_to_mesh(mesh_quadrilaterals):
    """
    Ajusta un polígono de 4 lados que contenga todos los cuadriláteros de la malla.
    Retorna el polígono ajustado y las dimensiones de la grilla (rows, cols).
    """
    if len(mesh_quadrilaterals) == 0:
        return None, 0, 0
    
    # Obtener todos los vértices de todos los cuadriláteros
    all_vertices = []
    for quad in mesh_quadrilaterals:
        vertices = get_vertices_as_points(quad)
        all_vertices.extend(vertices)
    
    # Convertir a array numpy
    points = np.array(all_vertices, dtype=np.float32)
    
    # Encontrar el rectángulo mínimo que contiene todos los puntos (orientado)
    # Usamos minAreaRect para obtener el rectángulo orientado
    rect = cv.minAreaRect(points)
    box = cv.boxPoints(rect)
    box = np.array(box, dtype=np.float32)
    
    # Ordenar los puntos en sentido antihorario comenzando desde el superior-izquierdo
    center = np.mean(box, axis=0)
    
    # Calcular ángulos desde el centro
    angles = np.arctan2(box[:, 1] - center[1], box[:, 0] - center[0])
    
    # Ordenar por ángulo en sentido antihorario (de -pi a pi)
    sorted_indices = np.argsort(angles)
    box = box[sorted_indices]
    
    # Reordenar para que empiece por el superior-izquierdo
    min_sum_idx = np.argmin(box[:, 0] + box[:, 1])
    box = np.roll(box, -min_sum_idx, axis=0)
    
    # Verificar que esté en sentido antihorario
    area = 0
    for i in range(4):
        j = (i + 1) % 4
        area += box[i, 0] * box[j, 1]
        area -= box[j, 0] * box[i, 1]
    
    # Si el área es positiva (sentido horario), invertir
    if area > 0:
        box = box[::-1]
        min_sum_idx = np.argmin(box[:, 0] + box[:, 1])
        box = np.roll(box, -min_sum_idx, axis=0)
    
    # Calcular el número de celdas en la malla
    # Tomar el primer cuadrilátero como referencia
    ref_quad = mesh_quadrilaterals[0]
    ref_vertices = get_vertices_as_points(ref_quad)
    
    # Calcular tamaño promedio de una celda
    widths = []
    heights = []
    for i in range(4):
        j = (i + 1) % 4
        dist = np.sqrt((ref_vertices[i][0] - ref_vertices[j][0])**2 + 
                       (ref_vertices[i][1] - ref_vertices[j][1])**2)
        # Determinar si es ancho o alto basado en la orientación
        if abs(ref_vertices[i][0] - ref_vertices[j][0]) > abs(ref_vertices[i][1] - ref_vertices[j][1]):
            widths.append(dist)
        else:
            heights.append(dist)
    
    avg_cell_width = np.mean(widths) if widths else 50
    avg_cell_height = np.mean(heights) if heights else 50
    
    # Calcular dimensiones del rectángulo
    rect_width = np.sqrt((box[0][0] - box[1][0])**2 + (box[0][1] - box[1][1])**2)
    rect_height = np.sqrt((box[0][0] - box[3][0])**2 + (box[0][1] - box[3][1])**2)
    
    # Estimar número de celdas
    num_cols = max(2, int(rect_width / avg_cell_width + 0.5))
    num_rows = max(2, int(rect_height / avg_cell_height + 0.5))
    
    # Asegurar que no exceda 8
    num_cols = min(num_cols, 8)
    num_rows = min(num_rows, 8)
    
    return box, num_rows, num_cols

def expand_polygon_with_perspective(polygon, row_start, col_start, grid_rows, grid_cols, target_rows=8, target_cols=8):
    """
    Expande un polígono que contiene la malla para que ocupe todo el tablero de 8x8.
    """
    # Obtener vértices del polígono
    p1 = polygon[0]  # top-left
    p2 = polygon[1]  # top-right
    p3 = polygon[2]  # bottom-right
    p4 = polygon[3]  # bottom-left
    
    # Calcular vectores de los lados
    top_vec = p2 - p1
    bottom_vec = p3 - p4
    left_vec = p4 - p1
    right_vec = p3 - p2
    
    # Calcular cuántas celdas expandir en cada dirección
    cells_up = row_start
    cells_down = target_rows - grid_rows - row_start
    cells_left = col_start
    cells_right = target_cols - grid_cols - col_start
    
    # Extrapolar los vértices del tablero completo
    # Esquina superior-izquierda
    top_left = p1 - (cells_up * left_vec / (cells_up + cells_down + 1)) - (cells_left * top_vec / (cells_left + cells_right + 1))
    
    # Esquina superior-derecha
    top_right = p2 - (cells_up * right_vec / (cells_up + cells_down + 1)) + (cells_right * top_vec / (cells_left + cells_right + 1))
    
    # Esquina inferior-izquierda
    bottom_left = p4 + (cells_down * left_vec / (cells_up + cells_down + 1)) - (cells_left * bottom_vec / (cells_left + cells_right + 1))
    
    # Esquina inferior-derecha
    bottom_right = p3 + (cells_down * right_vec / (cells_up + cells_down + 1)) + (cells_right * bottom_vec / (cells_left + cells_right + 1))
    
    # Convertir a enteros
    top_left = (int(top_left[0]), int(top_left[1]))
    top_right = (int(top_right[0]), int(top_right[1]))
    bottom_right = (int(bottom_right[0]), int(bottom_right[1]))
    bottom_left = (int(bottom_left[0]), int(bottom_left[1]))
    
    return np.array([top_left, top_right, bottom_right, bottom_left], dtype=np.float32)

def apply_warp(frame, src_points, dst_size=(800, 800)):
    """Aplica warp perspective a la imagen"""
    dst_points = np.array([
        [0, 0],
        [dst_size[0]-1, 0],
        [dst_size[0]-1, dst_size[1]-1],
        [0, dst_size[1]-1]
    ], dtype=np.float32)
    
    M = cv.getPerspectiveTransform(src_points, dst_points)
    warped = cv.warpPerspective(frame, M, dst_size)
    return warped

def score_warp(warped_img):
    """Puntúa qué tan bien se asemeja el warp a un tablero ideal."""
    gray = cv.cvtColor(warped_img, cv.COLOR_BGR2GRAY)
    h, w = gray.shape
    
    edges = cv.Canny(gray, 50, 150)
    cell_h = h // 8
    cell_w = w // 8
    
    score = 0
    total_checks = 0
    
    # Verificar líneas de la grilla
    grid_edge_score = 0
    for i in range(9):
        y = i * cell_h
        if y < h:
            edge_count = np.sum(edges[y, :] > 0)
            expected_edges = w * 0.2
            if edge_count > 0:
                line_score = min(edge_count / expected_edges, 1.0)
                grid_edge_score += line_score
                total_checks += 1
        
        x = i * cell_w
        if x < w:
            edge_count = np.sum(edges[:, x] > 0)
            expected_edges = h * 0.2
            if edge_count > 0:
                line_score = min(edge_count / expected_edges, 1.0)
                grid_edge_score += line_score
                total_checks += 1
    
    if total_checks > 0:
        grid_edge_score = grid_edge_score / total_checks
        score += grid_edge_score * 30
    
    # Uniformidad de celdas
    uniformity_score = 0
    for row in range(8):
        for col in range(8):
            y1 = row * cell_h
            y2 = (row + 1) * cell_h
            x1 = col * cell_w
            x2 = (col + 1) * cell_w
            
            cell_region = gray[y1:y2, x1:x2]
            if cell_region.size > 0:
                variance = np.var(cell_region)
                var_score = max(0, 1 - variance / 5000)
                uniformity_score += var_score
    
    uniformity_score = uniformity_score / 64
    score += uniformity_score * 20
    
    # Simetría
    symmetry_score = 0
    for row in range(7):
        for col in range(8):
            y1 = row * cell_h
            y2 = (row + 1) * cell_h
            y3 = (row + 2) * cell_h
            
            if y3 < h:
                cell_top = gray[y1:y2, col*cell_w:(col+1)*cell_w]
                cell_bottom = gray[y2:y3, col*cell_w:(col+1)*cell_w]
                
                if cell_top.size > 0 and cell_bottom.size > 0:
                    mean_top = np.mean(cell_top)
                    mean_bottom = np.mean(cell_bottom)
                    diff = abs(mean_top - mean_bottom)
                    diff_score = min(diff / 50, 1.0)
                    symmetry_score += diff_score
    
    if (7 * 8) > 0:
        symmetry_score = symmetry_score / (7 * 8)
        score += symmetry_score * 25
    
    # Bordes externos
    perimeter_score = 0
    edge_count = np.sum(edges[0:5, :] > 0)
    perimeter_score += min(edge_count / (w * 0.1), 1.0)
    edge_count = np.sum(edges[h-5:h, :] > 0)
    perimeter_score += min(edge_count / (w * 0.1), 1.0)
    edge_count = np.sum(edges[:, 0:5] > 0)
    perimeter_score += min(edge_count / (h * 0.1), 1.0)
    edge_count = np.sum(edges[:, w-5:w] > 0)
    perimeter_score += min(edge_count / (h * 0.1), 1.0)
    
    perimeter_score = perimeter_score / 4
    score += perimeter_score * 25
    
    final_score = min(score, 100)
    return final_score

def find_best_warp_and_board(frame, largest_mesh):
    """Encuentra el mejor warp usando el polígono ajustado a la malla."""
    
    # Ajustar polígono a la malla
    mesh_polygon, num_rows, num_cols = fit_polygon_to_mesh(largest_mesh)
    
    if mesh_polygon is None:
        return None, -float('inf'), None, None
    
    print(f"Malla: {num_rows}x{num_cols} celdas")
    
    best_warp = None
    best_score = -float('inf')
    best_cell_info = None
    best_board_points = None
    
    # Probar diferentes posiciones iniciales para el polígono en la grilla 8x8
    for row_start in range(0, 8 - num_rows + 1):
        for col_start in range(0, 8 - num_cols + 1):
            # Expandir el polígono para que ocupe todo el tablero 8x8
            src_points = expand_polygon_with_perspective(
                mesh_polygon, row_start, col_start, num_rows, num_cols
            )
            
            # Verificar que los puntos estén dentro de la imagen
            h, w = frame.shape[:2]
            valid = True
            for pt in src_points:
                if pt[0] < 0 or pt[0] >= w or pt[1] < 0 or pt[1] >= h:
                    valid = False
                    break
            
            if not valid:
                continue
            
            # Aplicar warp
            warped = apply_warp(frame, src_points)
            
            # Calcular score
            score = score_warp(warped)
            
            if score > best_score:
                best_score = score
                best_warp = warped
                best_cell_info = (row_start, col_start, num_rows, num_cols, src_points)
                best_board_points = src_points
    
    return best_warp, best_score, best_cell_info, best_board_points

while True:

    # Solo leer un nuevo frame si cambió la posición
    if current_frame != last_frame:
        cap.set(cv.CAP_PROP_POS_FRAMES, current_frame)
        ret, frame = cap.read()

        if ret:
            frame_vis = frame.copy()
            gray = cv.cvtColor(frame, cv.COLOR_BGR2GRAY)

            # ========== CÁLCULO DE PUNTOS DE SILLA ==========
            Ixx = cv.Sobel(gray, cv.CV_32F, 2, 0, ksize=3)
            Iyy = cv.Sobel(gray, cv.CV_32F, 0, 2, ksize=3)
            Ixy = cv.Sobel(gray, cv.CV_32F, 1, 1, ksize=3)

            response = -(Ixx*Iyy - Ixy*Ixy)
            response = cv.GaussianBlur(response, (15,15), 0)

            mx = cv.dilate(response, np.ones((7,7), np.uint8))
            mask = np.zeros_like(gray)
            th = 0.15 * response.max()

            pts = np.where((response == mx) & (response > th))
            points = np.column_stack((pts[1], pts[0]))
            mask[pts] = 255

            cv.imshow("Saddle mask", mask)

            # ========== SOBEL Y CANNY ==========
            sobelx = cv.Sobel(gray, cv.CV_64F, 1, 0, ksize=3)
            sobely = cv.Sobel(gray, cv.CV_64F, 0, 1, ksize=3)
            sobel_magnitude = np.sqrt(sobelx**2 + sobely**2)
            sobel_magnitude = np.uint8(np.clip(sobel_magnitude, 0, 255))

            cv.imshow(sobel_window, sobel_magnitude)

            edges = cv.Canny(sobel_magnitude, 7000, 7050, apertureSize=5)
            cv.imshow(canny_window, edges)

            # ========== PASO 1: ENCONTRAR TODOS LOS CONTORNOS ==========
            contours, hierarchy = cv.findContours(edges, cv.RETR_EXTERNAL, cv.CHAIN_APPROX_SIMPLE)

            contour_img = frame.copy()
            cv.drawContours(contour_img, contours, -1, (0, 255, 0), 2)
            cv.imshow(contour_window, contour_img)

            # ========== PASO 2: APROXIMAR POLÍGONOS ==========
            all_polygons = []
            contour_approx_img = frame.copy()
            
            for contour in contours:
                epsilon = 0.01 * cv.arcLength(contour, True)
                approx = cv.approxPolyDP(contour, epsilon, True)
                all_polygons.append(approx)
                cv.drawContours(contour_approx_img, [approx], -1, (255, 0, 0), 3)
                if len(approx) == 4:
                    for point in approx:
                        cv.circle(contour_approx_img, tuple(point[0]), 5, (0, 255, 255), -1)

            cv.imshow(approx_window, contour_approx_img)

            # ========== PASO 3: FILTRAR POLÍGONOS POR PUNTOS DE SILLA ==========
            saddle_polygons = []
            saddle_points = [(int(p[0]), int(p[1])) for p in points]
            
            for polygon in all_polygons:
                contains_saddle = False
                for point in saddle_points:
                    distance = cv.pointPolygonTest(polygon, point, True)
                    if distance >= -TOLERANCE:
                        contains_saddle = True
                        break
                if contains_saddle:
                    saddle_polygons.append(polygon)

            contour_filtered_img = frame.copy()
            cv.drawContours(contour_filtered_img, saddle_polygons, -1, (0, 0, 255), 3)
            cv.putText(contour_filtered_img, f"Tolerance: {TOLERANCE}px", 
                      (10, 30), cv.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
            cv.imshow(contour_filtered_window, contour_filtered_img)

            # ========== DETECCIÓN DE CUADRILÁTEROS ==========
            quadrilaterals_img = frame.copy()
            quadrilaterals = []
            
            for polygon in saddle_polygons:
                if len(polygon) == 4:
                    area = cv.contourArea(polygon)
                    if cv.isContourConvex(polygon) and area >= MIN_AREA:
                        quadrilaterals.append(polygon)
                        cv.drawContours(quadrilaterals_img, [polygon], -1, (0, 0, 255), 3)
                        for point in polygon:
                            cv.circle(quadrilaterals_img, tuple(point[0]), 6, (0, 255, 255), -1)
                        cv.putText(quadrilaterals_img, f"Area: {int(area)}", 
                                  (int(polygon[0][0][0]), int(polygon[0][0][1]) - 20), 
                                  cv.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2)
            
            all_quadrilaterals.extend(quadrilaterals)
            cv.imshow(quadrilaterals_window, quadrilaterals_img)

            # ========== ENCONTRAR LA MALLA MÁS GRANDE ==========
            mesh_img = frame.copy()
            largest_mesh = []
            
            if len(quadrilaterals) > 0:
                largest_mesh = find_largest_mesh(quadrilaterals, ADJACENT_THRESHOLD)
                
                if len(largest_mesh) > 0:
                    for polygon in largest_mesh:
                        pts = polygon.reshape(-1, 2).astype(np.int32)
                        cv.fillPoly(mesh_img, [pts], (128, 128, 128))
                        cv.drawContours(mesh_img, [polygon], -1, (0, 0, 255), 2)
                    
                    cv.putText(mesh_img, f"Mesh size: {len(largest_mesh)} quadrilaterals", 
                              (10, 30), cv.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
            
            cv.imshow(mesh_window, mesh_img)

            # ========== ENCONTRAR EL MEJOR WARP USANDO LA MALLA COMPLETA ==========
            warp_result_img = np.zeros((800, 800, 3), dtype=np.uint8)
            estimated_board_img = frame.copy()
            
            if len(largest_mesh) >= 3:
                best_warp, best_score, best_info, best_board_points = find_best_warp_and_board(frame, largest_mesh)
                
                # ===== MOSTRAR TABLERO ESTIMADO =====
                if best_board_points is not None:
                    pts = best_board_points.astype(np.int32)
                    cv.polylines(estimated_board_img, [pts], True, (0, 255, 0), 4)
                    for point in pts:
                        cv.circle(estimated_board_img, tuple(point), 10, (0, 255, 255), -1)
                    
                    if best_info is not None:
                        row_start, col_start, num_rows, num_cols, src_points = best_info
                        cv.putText(estimated_board_img, f"Malla: {num_rows}x{num_cols} - Pos: ({row_start},{col_start})", 
                                  (10, 30), cv.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
                        cv.putText(estimated_board_img, f"Score: {best_score:.1f}/100", 
                                  (10, 60), cv.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
                
                # ===== MOSTRAR WARP =====
                if best_warp is not None:
                    warp_result_img = best_warp
                    row_start, col_start, num_rows, num_cols, src_points = best_info
                    
                    cv.putText(warp_result_img, f"Score: {best_score:.2f}/100", 
                              (10, 30), cv.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
                    cv.putText(warp_result_img, f"Malla: {num_rows}x{num_cols} - Pos: ({row_start},{col_start})", 
                              (10, 60), cv.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
                    
                    # Dibujar la grilla 8x8
                    h, w = warp_result_img.shape[:2]
                    cell_h = h // 8
                    cell_w = w // 8
                    for i in range(9):
                        y = i * cell_h
                        cv.line(warp_result_img, (0, y), (w, y), (0, 255, 255), 1)
                        x = i * cell_w
                        cv.line(warp_result_img, (x, 0), (x, h), (0, 255, 255), 1)
                    
                    # Mostrar puntuación con color
                    if best_score >= 70:
                        color = (0, 255, 0)
                    elif best_score >= 50:
                        color = (0, 255, 255)
                    else:
                        color = (0, 0, 255)
                    
                    cv.rectangle(warp_result_img, (w-200, 10), (w-10, 50), (0, 0, 0), -1)
                    cv.putText(warp_result_img, f"{best_score:.1f}%", 
                              (w-180, 40), cv.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)
            
            cv.imshow(estimated_board_window, estimated_board_img)
            cv.imshow(warp_window, warp_result_img)

            # Mostrar puntos de silla
            saddle_points_img = frame.copy()
            if len(points) > 0:
                for point in points:
                    cv.circle(saddle_points_img, (int(point[0]), int(point[1])), 4, (0, 255, 0), -1)

            cv.imshow(hough_filtered_window, saddle_points_img)
            cv.imshow(window_name, frame_vis)

            last_frame = current_frame

    key = cv.waitKey(20) & 0xFF

    if key == 27:      # ESC para salir
        break

cap.release()
cv.destroyAllWindows()

Frames totales: 430
FPS: 29.97


In [3]:
import cv2 as cv
import numpy as np
from scipy.fft import fft2, fftshift
from scipy.stats import entropy
import copy
import matplotlib.pyplot as plt
from IPython.display import display, clear_output

# Ruta al video
video_path = "../../../data/raw/Prueba2.mp4"

# Abrir el video
cap = cv.VideoCapture(video_path)

if not cap.isOpened():
    raise IOError(f"No se pudo abrir el video: {video_path}")

# Obtener información del video
total_frames = int(cap.get(cv.CAP_PROP_FRAME_COUNT))
fps = cap.get(cv.CAP_PROP_FPS)

print(f"Frames totales: {total_frames}")
print(f"FPS: {fps:.2f}")

# Crear ventanas
window_name = "Video - Marcar 4 puntos para warp"
warp1_window = "Warp 1 Result"
warp2_window = "Warp 2 Result"
fft_original_window = "FFT Original"
fft_warp1_window = "FFT Warp 1 (Cuadrante Superior)"
fft_warp2_window = "FFT Warp 2 (Cuadrante Superior)"
fft_warp1_info_window = "FFT Warp 1 Info"
fft_warp2_info_window = "FFT Warp 2 Info"
comparison_window = "Comparación Warp 1 vs Warp 2"
lines_warp1_window = "Líneas detectadas Warp 1"
lines_warp2_window = "Líneas detectadas Warp 2"
entropy_warp1_window = "Entropía Warp 1"
entropy_warp2_window = "Entropía Warp 2"

cv.namedWindow(window_name, cv.WINDOW_NORMAL)
cv.namedWindow(warp1_window, cv.WINDOW_NORMAL)
cv.namedWindow(warp2_window, cv.WINDOW_NORMAL)
cv.namedWindow(fft_original_window, cv.WINDOW_NORMAL)
cv.namedWindow(fft_warp1_window, cv.WINDOW_NORMAL)
cv.namedWindow(fft_warp2_window, cv.WINDOW_NORMAL)
cv.namedWindow(fft_warp1_info_window, cv.WINDOW_NORMAL)
cv.namedWindow(fft_warp2_info_window, cv.WINDOW_NORMAL)
cv.namedWindow(comparison_window, cv.WINDOW_NORMAL)
cv.namedWindow(lines_warp1_window, cv.WINDOW_NORMAL)
cv.namedWindow(lines_warp2_window, cv.WINDOW_NORMAL)
cv.namedWindow(entropy_warp1_window, cv.WINDOW_NORMAL)
cv.namedWindow(entropy_warp2_window, cv.WINDOW_NORMAL)

# Variables globales
points1 = []
points2 = []
current_frame = 0
last_frame = -1
points1_marked = False
points2_marked = False
warp1_result = None
warp2_result = None
selected_points1 = []
selected_points2 = []
frame_actual = None
modo_actual = 1

# ========== FUNCIONES DE ENTROPÍA Y SEGMENTACIÓN ==========
def calcular_entropia_seccion(seccion):
    """Calcula la entropía de una sección de imagen"""
    if seccion is None or seccion.size == 0:
        return 0
    
    # Convertir a grises si es color
    if len(seccion.shape) == 3:
        gray = cv.cvtColor(seccion, cv.COLOR_BGR2GRAY)
    else:
        gray = seccion
    
    # Aplanar y normalizar
    flat = gray.flatten().astype(np.float32)
    # Normalizar a [0, 1]
    flat = flat / 255.0
    
    # Calcular histograma (64 bins para mayor precisión)
    hist, _ = np.histogram(flat, bins=64, range=(0, 1))
    prob = hist / (np.sum(hist) + 1e-10)
    prob_eps = prob + 1e-10
    entropia_val = -np.sum(prob_eps * np.log2(prob_eps))
    
    return entropia_val

def segmentar_y_calcular_entropias(imagen, n_secciones=8):
    """
    Segmenta la imagen en n_secciones filas y n_secciones columnas,
    y calcula la entropía de cada sección.
    Retorna las entropías y las imágenes segmentadas.
    """
    if imagen is None:
        return None, None, None, None, None, None
    
    h, w = imagen.shape[:2]
    alto_seccion = h // n_secciones
    ancho_seccion = w // n_secciones
    
    entropias_filas = []
    entropias_columnas = []
    
    # Crear imágenes para visualización
    img_filas = imagen.copy()
    img_columnas = imagen.copy()
    
    # Calcular entropías por filas
    for i in range(n_secciones):
        y_inicio = i * alto_seccion
        y_fin = (i + 1) * alto_seccion
        seccion = imagen[y_inicio:y_fin, :]
        entropia = calcular_entropia_seccion(seccion)
        entropias_filas.append(entropia)
        
        # Dibujar línea divisoria y entropía en la imagen
        cv.line(img_filas, (0, y_fin-1), (w, y_fin-1), (0, 255, 255), 2)
        cv.putText(img_filas, f'E{i+1}: {entropia:.3f}', 
                  (10, y_inicio + 30), cv.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)
    
    # Calcular entropías por columnas
    for i in range(n_secciones):
        x_inicio = i * ancho_seccion
        x_fin = (i + 1) * ancho_seccion
        seccion = imagen[:, x_inicio:x_fin]
        entropia = calcular_entropia_seccion(seccion)
        entropias_columnas.append(entropia)
        
        # Dibujar línea divisoria y entropía en la imagen
        cv.line(img_columnas, (x_fin-1, 0), (x_fin-1, h), (0, 255, 255), 2)
        cv.putText(img_columnas, f'E{i+1}: {entropia:.3f}', 
                  (x_inicio + 10, 30), cv.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)
    
    # Calcular estadísticas
    mean_filas = np.mean(entropias_filas)
    std_filas = np.std(entropias_filas)
    var_filas = np.var(entropias_filas)
    
    mean_columnas = np.mean(entropias_columnas)
    std_columnas = np.std(entropias_columnas)
    var_columnas = np.var(entropias_columnas)
    
    estadisticas = {
        'filas': {
            'entropias': entropias_filas,
            'mean': mean_filas,
            'std': std_filas,
            'var': var_filas
        },
        'columnas': {
            'entropias': entropias_columnas,
            'mean': mean_columnas,
            'std': std_columnas,
            'var': var_columnas
        }
    }
    
    return entropias_filas, entropias_columnas, img_filas, img_columnas, estadisticas

def calcular_score_warp(entropias_filas, entropias_columnas):
    """
    Calcula un score basado en las entropías.
    Ideal: todas las secciones tienen entropía alta (cerca de 8 para imágenes de 8 bits)
    """
    if entropias_filas is None or entropias_columnas is None:
        return 0
    
    # Promedio de entropías
    promedio_filas = np.mean(entropias_filas)
    promedio_columnas = np.mean(entropias_columnas)
    
    # Desviación estándar (queremos baja variación)
    std_filas = np.std(entropias_filas)
    std_columnas = np.std(entropias_columnas)
    
    # Score: mayor promedio y menor desviación = mejor
    # Para imágenes de 8 bits, entropía máxima ≈ 8 (log2(256))
    score_promedio = (promedio_filas + promedio_columnas) / 2
    score_std = 1 - (std_filas + std_columnas) / 2  # Normalizado
    
    # Score combinado (0-100)
    score = (score_promedio / 8) * 70 + score_std * 30
    score = min(max(score, 0), 100)
    
    return score

# ========== FUNCIONES DE FFT ==========
def calcular_fft(imagen, mostrar_cuadrante_superior=True):
    if imagen is None or imagen.size == 0:
        return None
    
    if len(imagen.shape) == 3:
        gray = cv.cvtColor(imagen, cv.COLOR_BGR2GRAY)
    else:
        gray = imagen
    
    f_transform = fft2(gray.astype(np.float32))
    f_shift = fftshift(f_transform)
    magnitude = np.abs(f_shift)
    
    magnitude_flat = magnitude.flatten()
    magnitude_norm = magnitude_flat / (np.sum(magnitude_flat) + 1e-10)
    
    energia_total = np.sum(magnitude_flat ** 2)
    
    magnitude_norm_eps = magnitude_norm + 1e-10
    entropia = -np.sum(magnitude_norm_eps * np.log2(magnitude_norm_eps + 1e-10))
    
    center = magnitude.shape[0] // 2, magnitude.shape[1] // 2
    radius = 20
    y, x = np.ogrid[:magnitude.shape[0], :magnitude.shape[1]]
    mask = (x - center[1])**2 + (y - center[0])**2 <= radius**2
    energia_centro = np.sum(magnitude[mask] ** 2)
    energia_alta = energia_total - energia_centro
    ratio_energia = energia_centro / (energia_alta + 1e-10)
    
    magnitude_log = np.log(magnitude + 1)
    magnitude_norm_vis = cv.normalize(magnitude_log, None, 0, 255, cv.NORM_MINMAX)
    magnitude_uint8 = np.uint8(magnitude_norm_vis)
    magnitude_enhanced = cv.equalizeHist(magnitude_uint8)
    fft_color = cv.applyColorMap(magnitude_enhanced, cv.COLORMAP_JET)
    
    if mostrar_cuadrante_superior:
        h, w = fft_color.shape[:2]
        fft_color = fft_color[0:h//2, :]
        magnitude_cuadrante = magnitude[0:h//2, :]
        magnitude_flat_cuadrante = magnitude_cuadrante.flatten()
        energia_total_cuadrante = np.sum(magnitude_flat_cuadrante ** 2)
        
        center_cuadrante = magnitude_cuadrante.shape[0] // 2, magnitude_cuadrante.shape[1] // 2
        radius = 20
        y, x = np.ogrid[:magnitude_cuadrante.shape[0], :magnitude_cuadrante.shape[1]]
        mask_cuadrante = (x - center_cuadrante[1])**2 + (y - center_cuadrante[0])**2 <= radius**2
        energia_centro_cuadrante = np.sum(magnitude_cuadrante[mask_cuadrante] ** 2)
        energia_alta_cuadrante = energia_total_cuadrante - energia_centro_cuadrante
        ratio_energia_cuadrante = energia_centro_cuadrante / (energia_alta_cuadrante + 1e-10)
        
        return {
            'entropia': entropia,
            'energia_total': energia_total_cuadrante,
            'energia_centro': energia_centro_cuadrante,
            'energia_alta': energia_alta_cuadrante,
            'ratio_energia': ratio_energia_cuadrante,
            'fft_color': fft_color,
            'magnitude': magnitude_cuadrante,
            'magnitude_flat': magnitude_flat_cuadrante,
            'magnitude_enhanced': magnitude_enhanced,
            'es_cuadrante': True
        }
    
    return {
        'entropia': entropia,
        'energia_total': energia_total,
        'energia_centro': energia_centro,
        'energia_alta': energia_alta,
        'ratio_energia': ratio_energia,
        'fft_color': fft_color,
        'magnitude': magnitude,
        'magnitude_flat': magnitude_flat,
        'magnitude_enhanced': magnitude_enhanced,
        'es_cuadrante': False
    }

def detectar_lineas_fft(magnitude_enhanced, umbral_canny=230, umbral_hough=250):
    """
    Detecta líneas rectas en la FFT usando la transformada de Hough.
    Retorna la imagen con líneas dibujadas y el número de líneas detectadas.
    """
    if magnitude_enhanced is None:
        return None, 0
    
    # Asegurar que la imagen esté en uint8
    img = magnitude_enhanced.astype(np.uint8)
    
    # Aplicar Canny para detectar bordes
    edges = cv.Canny(img, umbral_canny, umbral_canny * 2)
    
    # Detectar líneas con Hough
    lines = cv.HoughLinesP(edges, 1, np.pi/180, umbral_hough, 
                          minLineLength=30, maxLineGap=10)
    
    # Crear copia de la FFT para dibujar líneas
    img_lines = cv.cvtColor(img, cv.COLOR_GRAY2BGR)
    
    num_lineas = 0
    if lines is not None:
        num_lineas = len(lines)
        for line in lines:
            x1, y1, x2, y2 = line[0]
            cv.line(img_lines, (x1, y1), (x2, y2), (0, 0, 255), 2)
    
    return img_lines, num_lineas

def plotear_histograma(magnitude_flat, titulo="Histograma FFT"):
    """Plotea el histograma en Jupyter"""
    if magnitude_flat is None or len(magnitude_flat) == 0:
        return
    
    log_magnitude = np.log10(magnitude_flat + 1e-10)
    
    plt.figure(figsize=(10, 5))
    plt.hist(log_magnitude, bins=50, color='blue', alpha=0.7, edgecolor='black')
    plt.title(titulo, fontsize=14, fontweight='bold')
    plt.xlabel('Log10(Magnitud)', fontsize=12)
    plt.ylabel('Frecuencia', fontsize=12)
    plt.grid(True, alpha=0.3)
    
    mean_val = np.mean(log_magnitude)
    std_val = np.std(log_magnitude)
    median_val = np.median(log_magnitude)
    
    plt.axvline(mean_val, color='red', linestyle='--', linewidth=2, label=f'Media: {mean_val:.2f}')
    plt.axvline(median_val, color='green', linestyle='--', linewidth=2, label=f'Mediana: {median_val:.2f}')
    plt.axvline(mean_val - std_val, color='orange', linestyle=':', linewidth=1, alpha=0.7)
    plt.axvline(mean_val + std_val, color='orange', linestyle=':', linewidth=1, alpha=0.7, label=f'±1σ: {std_val:.2f}')
    plt.legend()
    plt.tight_layout()
    plt.show()

def plotear_entropias(entropias_filas, entropias_columnas, estadisticas, titulo="Entropías"):
    """Plotea las entropías en Jupyter con estadísticas"""
    n = len(entropias_filas)
    x = np.arange(n)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Entropías por filas
    bars1 = ax1.bar(x, entropias_filas, color='blue', alpha=0.7)
    ax1.set_xlabel('Fila')
    ax1.set_ylabel('Entropía (bits)')
    ax1.set_title(f'{titulo} - Filas')
    ax1.set_xticks(x)
    ax1.set_xticklabels([f'F{i+1}' for i in range(n)])
    ax1.grid(True, alpha=0.3)
    
    # Línea de media
    mean_f = estadisticas['filas']['mean']
    std_f = estadisticas['filas']['std']
    ax1.axhline(y=mean_f, color='red', linestyle='--', linewidth=2, 
                label=f'Media: {mean_f:.3f}')
    ax1.axhline(y=mean_f + std_f, color='orange', linestyle=':', linewidth=1, alpha=0.7)
    ax1.axhline(y=mean_f - std_f, color='orange', linestyle=':', linewidth=1, alpha=0.7)
    
    # Agregar valores en las barras
    for i, (bar, val) in enumerate(zip(bars1, entropias_filas)):
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height + 0.02,
                f'{val:.3f}', ha='center', va='bottom', fontsize=8)
    
    # Texto con estadísticas
    stats_text = f'Media: {mean_f:.3f}\nStd: {std_f:.3f}\nVar: {estadisticas["filas"]["var"]:.3f}'
    ax1.text(0.02, 0.98, stats_text, transform=ax1.transAxes, 
            verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    ax1.legend(loc='lower right')
    
    # Entropías por columnas
    bars2 = ax2.bar(x, entropias_columnas, color='green', alpha=0.7)
    ax2.set_xlabel('Columna')
    ax2.set_ylabel('Entropía (bits)')
    ax2.set_title(f'{titulo} - Columnas')
    ax2.set_xticks(x)
    ax2.set_xticklabels([f'C{i+1}' for i in range(n)])
    ax2.grid(True, alpha=0.3)
    
    # Línea de media
    mean_c = estadisticas['columnas']['mean']
    std_c = estadisticas['columnas']['std']
    ax2.axhline(y=mean_c, color='red', linestyle='--', linewidth=2, 
                label=f'Media: {mean_c:.3f}')
    ax2.axhline(y=mean_c + std_c, color='orange', linestyle=':', linewidth=1, alpha=0.7)
    ax2.axhline(y=mean_c - std_c, color='orange', linestyle=':', linewidth=1, alpha=0.7)
    
    # Agregar valores en las barras
    for i, (bar, val) in enumerate(zip(bars2, entropias_columnas)):
        height = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2., height + 0.02,
                f'{val:.3f}', ha='center', va='bottom', fontsize=8)
    
    # Texto con estadísticas
    stats_text = f'Media: {mean_c:.3f}\nStd: {std_c:.3f}\nVar: {estadisticas["columnas"]["var"]:.3f}'
    ax2.text(0.02, 0.98, stats_text, transform=ax2.transAxes, 
            verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    ax2.legend(loc='lower right')
    
    plt.tight_layout()
    plt.show()

def mostrar_fft(imagen, window_name):
    stats = calcular_fft(imagen, mostrar_cuadrante_superior=True)
    if stats:
        cv.imshow(window_name, stats['fft_color'])
        return stats
    return None

def crear_overlay_fft(stats, titulo="FFT"):
    if stats is None:
        return None
    
    img_fft = stats['fft_color'].copy()
    h, w = img_fft.shape[:2]
    
    overlay = img_fft.copy()
    cv.rectangle(overlay, (10, 10), (w-10, 250), (0, 0, 0), -1)
    cv.addWeighted(overlay, 0.7, img_fft, 0.3, 0, img_fft)
    
    y_pos = 35
    cv.putText(img_fft, titulo, (20, y_pos), cv.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
    
    y_pos += 40
    cv.putText(img_fft, f"ENTROPIA: {stats['entropia']:.3f} bits", 
              (20, y_pos), cv.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
    
    y_pos += 30
    cv.putText(img_fft, f"ENERGIA TOTAL: {stats['energia_total']:.2e}", 
              (20, y_pos), cv.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)
    
    y_pos += 30
    cv.putText(img_fft, f"ENERGIA BAJA: {stats['energia_centro']:.2e}", 
              (20, y_pos), cv.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)
    
    y_pos += 30
    cv.putText(img_fft, f"ENERGIA ALTA: {stats['energia_alta']:.2e}", 
              (20, y_pos), cv.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 255), 2)
    
    y_pos += 30
    cv.putText(img_fft, f"RATIO BAJA/ALTA: {stats['ratio_energia']:.3f}", 
              (20, y_pos), cv.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
    
    if stats['ratio_energia'] < 5 and stats['entropia'] > 10:
        calidad = "EXCELENTE"
        color = (0, 255, 0)
    elif stats['ratio_energia'] < 15 and stats['entropia'] > 8:
        calidad = "BUENO"
        color = (0, 255, 255)
    elif stats['ratio_energia'] < 30 and stats['entropia'] > 5:
        calidad = "REGULAR"
        color = (0, 128, 255)
    else:
        calidad = "MALO"
        color = (0, 0, 255)
    
    y_pos += 40
    cv.putText(img_fft, "CUADRANTE SUPERIOR (No reflejado)", 
              (20, y_pos), cv.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
    
    y_pos += 30
    cv.putText(img_fft, f"CALIDAD: {calidad}", 
              (20, y_pos), cv.FONT_HERSHEY_SIMPLEX, 0.8, color, 3)
    
    return img_fft

# ========== FUNCIONES PARA EL WARP ==========
def aplicar_warp(frame, src_points, dst_size=(800, 800)):
    dst_points = np.array([
        [0, 0],
        [dst_size[0]-1, 0],
        [dst_size[0]-1, dst_size[1]-1],
        [0, dst_size[1]-1]
    ], dtype=np.float32)
    
    src_points = np.array(src_points, dtype=np.float32)
    M = cv.getPerspectiveTransform(src_points, dst_points)
    warped = cv.warpPerspective(frame, M, dst_size)
    return warped

def crear_comparacion(stats1, stats2, warp1, warp2, score1=0, score2=0, 
                      stats_entropia1=None, stats_entropia2=None):
    if warp1 is None or warp2 is None:
        return None
    
    h, w = warp1.shape[:2]
    comparacion = np.zeros((h, w*2 + 10, 3), dtype=np.uint8)
    
    comparacion[0:h, 0:w] = warp1
    comparacion[0:h, w+10:w*2+10] = warp2
    
    cv.line(comparacion, (w+5, 0), (w+5, h), (255, 255, 255), 2)
    
    cv.putText(comparacion, "WARP 1", (10, 30), cv.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
    cv.putText(comparacion, "WARP 2", (w+20, 30), cv.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
    
    # Estadísticas Warp 1
    if stats1:
        cv.putText(comparacion, f"Entropia FFT: {stats1['entropia']:.2f}", 
                  (10, 60), cv.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 0), 1)
        cv.putText(comparacion, f"Ratio: {stats1['ratio_energia']:.2f}", 
                  (10, 80), cv.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 0), 1)
        cv.putText(comparacion, f"Score: {score1:.2f}", 
                  (10, 100), cv.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255), 1)
    
    # Estadísticas de entropía de segmentación Warp 1
    if stats_entropia1:
        cv.putText(comparacion, f"Mean Filas: {stats_entropia1['filas']['mean']:.3f}", 
                  (10, 120), cv.FONT_HERSHEY_SIMPLEX, 0.4, (200, 200, 200), 1)
        cv.putText(comparacion, f"Std Filas: {stats_entropia1['filas']['std']:.3f}", 
                  (10, 135), cv.FONT_HERSHEY_SIMPLEX, 0.4, (200, 200, 200), 1)
        cv.putText(comparacion, f"Mean Cols: {stats_entropia1['columnas']['mean']:.3f}", 
                  (10, 150), cv.FONT_HERSHEY_SIMPLEX, 0.4, (200, 200, 200), 1)
        cv.putText(comparacion, f"Std Cols: {stats_entropia1['columnas']['std']:.3f}", 
                  (10, 165), cv.FONT_HERSHEY_SIMPLEX, 0.4, (200, 200, 200), 1)
    
    # Estadísticas Warp 2
    if stats2:
        cv.putText(comparacion, f"Entropia FFT: {stats2['entropia']:.2f}", 
                  (w+20, 60), cv.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 0), 1)
        cv.putText(comparacion, f"Ratio: {stats2['ratio_energia']:.2f}", 
                  (w+20, 80), cv.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 0), 1)
        cv.putText(comparacion, f"Score: {score2:.2f}", 
                  (w+20, 100), cv.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255), 1)
    
    # Estadísticas de entropía de segmentación Warp 2
    if stats_entropia2:
        cv.putText(comparacion, f"Mean Filas: {stats_entropia2['filas']['mean']:.3f}", 
                  (w+20, 120), cv.FONT_HERSHEY_SIMPLEX, 0.4, (200, 200, 200), 1)
        cv.putText(comparacion, f"Std Filas: {stats_entropia2['filas']['std']:.3f}", 
                  (w+20, 135), cv.FONT_HERSHEY_SIMPLEX, 0.4, (200, 200, 200), 1)
        cv.putText(comparacion, f"Mean Cols: {stats_entropia2['columnas']['mean']:.3f}", 
                  (w+20, 150), cv.FONT_HERSHEY_SIMPLEX, 0.4, (200, 200, 200), 1)
        cv.putText(comparacion, f"Std Cols: {stats_entropia2['columnas']['std']:.3f}", 
                  (w+20, 165), cv.FONT_HERSHEY_SIMPLEX, 0.4, (200, 200, 200), 1)
    
    return comparacion

# ========== FUNCIONES DE MOUSE ==========
def mouse_callback(event, x, y, flags, param):
    global points1, points2, points1_marked, points2_marked
    global selected_points1, selected_points2, frame_actual, modo_actual
    
    if event == cv.EVENT_LBUTTONDOWN:
        if frame_actual is None:
            return
            
        if modo_actual == 1 and len(points1) < 4:
            points1.append((x, y))
            print(f"Warp 1 - Punto {len(points1)}: ({x}, {y})")
            
            if len(points1) == 4:
                points1_marked = True
                selected_points1 = points1.copy()
                print(f"\nWarp 1 - Puntos seleccionados: {selected_points1}")
                print("Presiona 'w' para aplicar el warp 1")
                print("Presiona '2' para cambiar a Warp 2\n")
            
        elif modo_actual == 2 and len(points2) < 4:
            points2.append((x, y))
            print(f"Warp 2 - Punto {len(points2)}: ({x}, {y})")
            
            if len(points2) == 4:
                points2_marked = True
                selected_points2 = points2.copy()
                print(f"\nWarp 2 - Puntos seleccionados: {selected_points2}")
                print("Presiona 'w' para aplicar el warp 2")
                print("Presiona '1' para cambiar a Warp 1\n")
        
        actualizar_frame(frame_actual)

# ========== FUNCIÓN PARA ACTUALIZAR FRAME ==========
def actualizar_frame(frame):
    global points1, points2, frame_actual, modo_actual
    
    frame_actual = frame.copy()
    frame_display = frame.copy()
    
    if modo_actual == 1:
        cv.putText(frame_display, "MODO: WARP 1 (Presiona '2' para cambiar)", 
                  (10, 30), cv.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)
    else:
        cv.putText(frame_display, "MODO: WARP 2 (Presiona '1' para cambiar)", 
                  (10, 30), cv.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)
    
    for i, pt in enumerate(points1):
        color = (0, 255, 0)
        cv.circle(frame_display, pt, 8, color, -1)
        cv.putText(frame_display, f"1-{i+1}", (pt[0]+10, pt[1]-10), 
                  cv.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
    
    if len(points1) == 4:
        for i in range(4):
            cv.line(frame_display, points1[i], points1[(i+1)%4], (0, 255, 0), 2)
        cv.putText(frame_display, "Warp 1 listo", (10, 60), 
                  cv.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
    
    for i, pt in enumerate(points2):
        color = (255, 0, 0)
        cv.circle(frame_display, pt, 8, color, -1)
        cv.putText(frame_display, f"2-{i+1}", (pt[0]+10, pt[1]-10), 
                  cv.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
    
    if len(points2) == 4:
        for i in range(4):
            cv.line(frame_display, points2[i], points2[(i+1)%4], (255, 0, 0), 2)
        cv.putText(frame_display, "Warp 2 listo", (10, 80), 
                  cv.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2)
    
    cv.imshow(window_name, frame_display)
    
    stats_original = calcular_fft(frame, mostrar_cuadrante_superior=True)
    if stats_original:
        cv.imshow(fft_original_window, stats_original['fft_color'])

# ========== CONFIGURACIÓN ==========
cv.setMouseCallback(window_name, mouse_callback)

def on_trackbar(pos):
    global current_frame
    current_frame = pos

cv.createTrackbar(
    "Frame",
    window_name,
    0,
    total_frames - 1,
    on_trackbar
)

# ========== LOOP PRINCIPAL ==========
print("\n=== INSTRUCCIONES ===")
print("1. Usa el slider para navegar entre frames")
print("2. Presiona '1' o '2' para cambiar entre Warp 1 y Warp 2")
print("3. Haz click en 4 puntos para marcar el tablero (esquinas)")
print("4. Presiona 'w' para aplicar el warp activo")
print("5. Presiona 'r' para reiniciar TODAS las selecciones")
print("6. Presiona 'ESC' para salir")
print("=====================\n")

# Variables para almacenar scores y entropías
warp1_score = 0
warp2_score = 0
warp1_estadisticas = None
warp2_estadisticas = None

while True:
    if current_frame != last_frame:
        cap.set(cv.CAP_PROP_POS_FRAMES, current_frame)
        ret, frame = cap.read()

        if ret:
            actualizar_frame(frame)
            last_frame = current_frame

    key = cv.waitKey(20) & 0xFF
    
    if key == ord('1'):
        modo_actual = 1
        print("\nCambiado a Warp 1")
        if frame_actual is not None:
            actualizar_frame(frame_actual)
    
    elif key == ord('2'):
        modo_actual = 2
        print("\nCambiado a Warp 2")
        if frame_actual is not None:
            actualizar_frame(frame_actual)
    
    elif key == ord('w'):
        if modo_actual == 1 and len(points1) == 4 and frame_actual is not None:
            print("\nAplicando Warp 1...")
            warp1_result = aplicar_warp(frame_actual, points1)
            cv.imshow(warp1_window, warp1_result)
            
            # ===== CALCULAR ENTROPÍAS POR SEGMENTACIÓN =====
            entropias_filas, entropias_columnas, img_filas, img_columnas, estadisticas = segmentar_y_calcular_entropias(warp1_result)
            warp1_estadisticas = estadisticas
            
            # Mostrar imágenes con entropías
            if img_filas is not None:
                cv.imshow(entropy_warp1_window + " Filas", img_filas)
            if img_columnas is not None:
                cv.imshow(entropy_warp1_window + " Columnas", img_columnas)
            
            # Calcular score
            warp1_score = calcular_score_warp(entropias_filas, entropias_columnas)
            print(f"Score Warp 1 (Segmentación): {warp1_score:.2f}/100")
            
            # Mostrar estadísticas en consola
            print(f"\n--- ESTADÍSTICAS DE ENTROPÍA WARP 1 ---")
            print(f"Filas - Media: {estadisticas['filas']['mean']:.4f}")
            print(f"Filas - Std: {estadisticas['filas']['std']:.4f}")
            print(f"Filas - Varianza: {estadisticas['filas']['var']:.4f}")
            print(f"Columnas - Media: {estadisticas['columnas']['mean']:.4f}")
            print(f"Columnas - Std: {estadisticas['columnas']['std']:.4f}")
            print(f"Columnas - Varianza: {estadisticas['columnas']['var']:.4f}")
            
            # Plotear entropías en Jupyter
            clear_output(wait=True)
            plotear_entropias(entropias_filas, entropias_columnas, estadisticas, "WARP 1")
            
            # ===== FFT =====
            stats_warp1 = mostrar_fft(warp1_result, fft_warp1_window)
            
            if stats_warp1:
                # Detectar líneas en la FFT
                img_lines, num_lineas = detectar_lineas_fft(stats_warp1['magnitude_enhanced'])
                if img_lines is not None:
                    cv.imshow(lines_warp1_window, img_lines)
                    stats_warp1['lineas'] = num_lineas
                    print(f"Líneas detectadas en Warp 1: {num_lineas}")
                
                print(f"\n--- ESTADÍSTICAS FFT WARP 1 ---")
                print(f"Entropía FFT: {stats_warp1['entropia']:.3f} bits")
                print(f"Energía Total: {stats_warp1['energia_total']:.2e}")
                print(f"Ratio Baja/Alta: {stats_warp1['ratio_energia']:.3f}")
                print(f"Score Segmentación: {warp1_score:.2f}/100")
                
                fft_with_overlay1 = crear_overlay_fft(stats_warp1, "FFT WARP 1 (CUADRANTE SUPERIOR)")
                if fft_with_overlay1 is not None:
                    cv.imshow(fft_warp1_info_window, fft_with_overlay1)
            
            # Comparación si ambos warps existen
            if warp1_result is not None and warp2_result is not None:
                stats1 = calcular_fft(warp1_result, mostrar_cuadrante_superior=True)
                stats2 = calcular_fft(warp2_result, mostrar_cuadrante_superior=True)
                comparacion = crear_comparacion(stats1, stats2, warp1_result, warp2_result, 
                                               warp1_score, warp2_score,
                                               warp1_estadisticas, warp2_estadisticas)
                if comparacion is not None:
                    cv.imshow(comparison_window, comparacion)
        
        elif modo_actual == 2 and len(points2) == 4 and frame_actual is not None:
            print("\nAplicando Warp 2...")
            warp2_result = aplicar_warp(frame_actual, points2)
            cv.imshow(warp2_window, warp2_result)
            
            # ===== CALCULAR ENTROPÍAS POR SEGMENTACIÓN =====
            entropias_filas, entropias_columnas, img_filas, img_columnas, estadisticas = segmentar_y_calcular_entropias(warp2_result)
            warp2_estadisticas = estadisticas
            
            # Mostrar imágenes con entropías
            if img_filas is not None:
                cv.imshow(entropy_warp2_window + " Filas", img_filas)
            if img_columnas is not None:
                cv.imshow(entropy_warp2_window + " Columnas", img_columnas)
            
            # Calcular score
            warp2_score = calcular_score_warp(entropias_filas, entropias_columnas)
            print(f"Score Warp 2 (Segmentación): {warp2_score:.2f}/100")
            
            # Mostrar estadísticas en consola
            print(f"\n--- ESTADÍSTICAS DE ENTROPÍA WARP 2 ---")
            print(f"Filas - Media: {estadisticas['filas']['mean']:.4f}")
            print(f"Filas - Std: {estadisticas['filas']['std']:.4f}")
            print(f"Filas - Varianza: {estadisticas['filas']['var']:.4f}")
            print(f"Columnas - Media: {estadisticas['columnas']['mean']:.4f}")
            print(f"Columnas - Std: {estadisticas['columnas']['std']:.4f}")
            print(f"Columnas - Varianza: {estadisticas['columnas']['var']:.4f}")
            
            # Plotear entropías en Jupyter
            clear_output(wait=True)
            plotear_entropias(entropias_filas, entropias_columnas, estadisticas, "WARP 2")
            
            # ===== FFT =====
            stats_warp2 = mostrar_fft(warp2_result, fft_warp2_window)
            
            if stats_warp2:
                # Detectar líneas en la FFT
                img_lines, num_lineas = detectar_lineas_fft(stats_warp2['magnitude_enhanced'])
                if img_lines is not None:
                    cv.imshow(lines_warp2_window, img_lines)
                    stats_warp2['lineas'] = num_lineas
                    print(f"Líneas detectadas en Warp 2: {num_lineas}")
                
                print(f"\n--- ESTADÍSTICAS FFT WARP 2 ---")
                print(f"Entropía FFT: {stats_warp2['entropia']:.3f} bits")
                print(f"Energía Total: {stats_warp2['energia_total']:.2e}")
                print(f"Ratio Baja/Alta: {stats_warp2['ratio_energia']:.3f}")
                print(f"Score Segmentación: {warp2_score:.2f}/100")
                
                fft_with_overlay2 = crear_overlay_fft(stats_warp2, "FFT WARP 2 (CUADRANTE SUPERIOR)")
                if fft_with_overlay2 is not None:
                    cv.imshow(fft_warp2_info_window, fft_with_overlay2)
            
            # Comparación si ambos warps existen
            if warp1_result is not None and warp2_result is not None:
                stats1 = calcular_fft(warp1_result, mostrar_cuadrante_superior=True)
                stats2 = calcular_fft(warp2_result, mostrar_cuadrante_superior=True)
                comparacion = crear_comparacion(stats1, stats2, warp1_result, warp2_result,
                                               warp1_score, warp2_score,
                                               warp1_estadisticas, warp2_estadisticas)
                if comparacion is not None:
                    cv.imshow(comparison_window, comparacion)
        else:
            if modo_actual == 1:
                print("❌ Necesitas marcar 4 puntos para Warp 1 primero")
            else:
                print("❌ Necesitas marcar 4 puntos para Warp 2 primero")
    
    elif key == ord('r'):
        points1 = []
        points2 = []
        points1_marked = False
        points2_marked = False
        warp1_result = None
        warp2_result = None
        warp1_score = 0
        warp2_score = 0
        warp1_estadisticas = None
        warp2_estadisticas = None
        print("\nTODAS las selecciones reiniciadas.\n")
        
        cv.imshow(warp1_window, np.zeros((800, 800, 3), dtype=np.uint8))
        cv.imshow(warp2_window, np.zeros((800, 800, 3), dtype=np.uint8))
        cv.imshow(fft_warp1_window, np.zeros((400, 400, 3), dtype=np.uint8))
        cv.imshow(fft_warp2_window, np.zeros((400, 400, 3), dtype=np.uint8))
        cv.imshow(fft_warp1_info_window, np.zeros((400, 400, 3), dtype=np.uint8))
        cv.imshow(fft_warp2_info_window, np.zeros((400, 400, 3), dtype=np.uint8))
        cv.imshow(comparison_window, np.zeros((400, 400, 3), dtype=np.uint8))
        cv.imshow(lines_warp1_window, np.zeros((400, 400, 3), dtype=np.uint8))
        cv.imshow(lines_warp2_window, np.zeros((400, 400, 3), dtype=np.uint8))
        cv.imshow(entropy_warp1_window + " Filas", np.zeros((400, 400, 3), dtype=np.uint8))
        cv.imshow(entropy_warp1_window + " Columnas", np.zeros((400, 400, 3), dtype=np.uint8))
        cv.imshow(entropy_warp2_window + " Filas", np.zeros((400, 400, 3), dtype=np.uint8))
        cv.imshow(entropy_warp2_window + " Columnas", np.zeros((400, 400, 3), dtype=np.uint8))
        
        if frame_actual is not None:
            actualizar_frame(frame_actual)
    
    elif key == 27:
        break

cap.release()
cv.destroyAllWindows()

KeyboardInterrupt: 

In [8]:
import cv2 as cv
import numpy as np
from scipy.spatial import KDTree
from collections import deque
import random

# Ruta al video
video_path = "../../../data/raw/Prueba2.mp4"

# Abrir el video
cap = cv.VideoCapture(video_path)

if not cap.isOpened():
    raise IOError(f"No se pudo abrir el video: {video_path}")

# Obtener información del video
total_frames = int(cap.get(cv.CAP_PROP_FRAME_COUNT))
fps = cap.get(cv.CAP_PROP_FPS)

print(f"Frames totales: {total_frames}")
print(f"FPS: {fps:.2f}")

window_name = "Video"
sobel_window = "Sobel"
canny_window = "Canny"
hough_filtered_window = "Saddle Points"
contour_window = "Todos los contornos"
contour_filtered_window = "Contornos con puntos de silla"
approx_window = "Polígonos aproximados"
quadrilaterals_window = "Cuadriláteros detectados"
mesh_window = "Malla más grande"
estimated_board_window = "Tablero estimado (mejor warp)"
warp_window = "Mejor Warp"

# Ventanas adicionales para visualización de entropías
entropy_filas_window = "Entropía por Filas"
entropy_columnas_window = "Entropía por Columnas"

cv.namedWindow(window_name, cv.WINDOW_NORMAL)
cv.namedWindow(sobel_window, cv.WINDOW_NORMAL)
cv.namedWindow(canny_window, cv.WINDOW_NORMAL)
cv.namedWindow(hough_filtered_window, cv.WINDOW_NORMAL)
cv.namedWindow(contour_window, cv.WINDOW_NORMAL)
cv.namedWindow(contour_filtered_window, cv.WINDOW_NORMAL)
cv.namedWindow(approx_window, cv.WINDOW_NORMAL)
cv.namedWindow(quadrilaterals_window, cv.WINDOW_NORMAL)
cv.namedWindow(mesh_window, cv.WINDOW_NORMAL)
cv.namedWindow(estimated_board_window, cv.WINDOW_NORMAL)
cv.namedWindow(warp_window, cv.WINDOW_NORMAL)
cv.namedWindow(entropy_filas_window, cv.WINDOW_NORMAL)
cv.namedWindow(entropy_columnas_window, cv.WINDOW_NORMAL)

# Variable para detectar cambios del slider
current_frame = 0

def on_trackbar(pos):
    global current_frame
    current_frame = pos

# Crear la trackbar
cv.createTrackbar(
    "Frame",
    window_name,
    0,
    total_frames - 1,
    on_trackbar
)

last_frame = -1

# Tolerancia fija
TOLERANCE = 5

# Área mínima para considerar un cuadrilátero
MIN_AREA = 10

# Distancia máxima para considerar que dos vértices son adyacentes
ADJACENT_THRESHOLD = 40

# Factor de aprendizaje para el desplazamiento iterativo (0-1)
LEARNING_RATE = 0.1

# Almacenar los vértices del tablero del frame anterior
previous_board_vertices = None

# Lista para almacenar todos los cuadriláteros detectados
all_quadrilaterals = []

# ========== FUNCIÓN PARA EXPANDIR CUADRILÁTERO 1px ==========
def expand_quadrilateral_1px(quadrilateral):
    """
    Expande un cuadrilátero 1 píxel hacia afuera en todos los lados.
    Desplaza cada vértice 1px lejos del centro del cuadrilátero.
    """
    if quadrilateral is None or len(quadrilateral) == 0:
        return quadrilateral
    
    # Obtener vértices como puntos
    vertices = []
    for point in quadrilateral:
        vertices.append(np.array(point[0], dtype=np.float32))
    
    # Calcular centro del cuadrilátero
    center = np.mean(vertices, axis=0)
    
    # Expandir cada vértice 1px alejándolo del centro
    expanded_vertices = []
    for v in vertices:
        # Vector desde el centro al vértice
        vec = v - center
        # Normalizar y escalar a 1px
        norm = np.linalg.norm(vec)
        if norm > 0:
            vec_unit = vec / norm
            new_v = v + vec_unit  # Desplazar 1px hacia afuera
        else:
            new_v = v
        expanded_vertices.append([new_v.astype(np.int32)])
    
    return np.array(expanded_vertices, dtype=np.int32)

def expand_quadrilateral_npx(quadrilateral, n=1):
    """
    Expande un cuadrilátero n píxeles hacia afuera en todos los lados.
    """
    if quadrilateral is None or len(quadrilateral) == 0:
        return quadrilateral
    
    # Obtener vértices como puntos
    vertices = []
    for point in quadrilateral:
        vertices.append(np.array(point[0], dtype=np.float32))
    
    # Calcular centro del cuadrilátero
    center = np.mean(vertices, axis=0)
    
    # Expandir cada vértice npx alejándolo del centro
    expanded_vertices = []
    for v in vertices:
        vec = v - center
        norm = np.linalg.norm(vec)
        if norm > 0:
            vec_unit = vec / norm
            new_v = v + vec_unit * n
        else:
            new_v = v
        expanded_vertices.append([new_v.astype(np.int32)])
    
    return np.array(expanded_vertices, dtype=np.int32)

# ========== FUNCIONES DE ENTROPÍA Y SCORE ==========
def calcular_entropia_seccion(seccion):
    """Calcula la entropía de una sección de imagen"""
    if seccion is None or seccion.size == 0:
        return 0
    
    # Convertir a grises si es color
    if len(seccion.shape) == 3:
        gray = cv.cvtColor(seccion, cv.COLOR_BGR2GRAY)
    else:
        gray = seccion
    
    # Aplanar y normalizar
    flat = gray.flatten().astype(np.float32)
    flat = flat / 255.0
    
    # Calcular histograma (64 bins para mayor precisión)
    hist, _ = np.histogram(flat, bins=64, range=(0, 1))
    prob = hist / (np.sum(hist) + 1e-10)
    prob_eps = prob + 1e-10
    entropia_val = -np.sum(prob_eps * np.log2(prob_eps))
    
    return entropia_val

def calcular_entropias_warp(imagen, n_secciones=8):
    """
    Calcula las entropías de las filas y columnas de un warp.
    Retorna las entropías y las imágenes con visualización.
    """
    if imagen is None:
        return None, None, None, None
    
    h, w = imagen.shape[:2]
    alto_seccion = h // n_secciones
    ancho_seccion = w // n_secciones
    
    entropias_filas = []
    entropias_columnas = []
    
    # Crear imágenes para visualización
    img_filas = imagen.copy()
    img_columnas = imagen.copy()
    
    # Calcular entropías por filas
    for i in range(n_secciones):
        y_inicio = i * alto_seccion
        y_fin = (i + 1) * alto_seccion
        seccion = imagen[y_inicio:y_fin, :]
        entropia = calcular_entropia_seccion(seccion)
        entropias_filas.append(entropia)
        
        # Dibujar línea divisoria y entropía
        cv.line(img_filas, (0, y_fin-1), (w, y_fin-1), (0, 255, 255), 2)
        cv.putText(img_filas, f'E{i+1}: {entropia:.3f}', 
                  (10, y_inicio + 30), cv.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)
    
    # Calcular entropías por columnas
    for i in range(n_secciones):
        x_inicio = i * ancho_seccion
        x_fin = (i + 1) * ancho_seccion
        seccion = imagen[:, x_inicio:x_fin]
        entropia = calcular_entropia_seccion(seccion)
        entropias_columnas.append(entropia)
        
        # Dibujar línea divisoria y entropía
        cv.line(img_columnas, (x_fin-1, 0), (x_fin-1, h), (0, 255, 255), 2)
        cv.putText(img_columnas, f'E{i+1}: {entropia:.3f}', 
                  (x_inicio + 10, 30), cv.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)
    
    # Calcular estadísticas
    std_filas = np.std(entropias_filas)
    std_columnas = np.std(entropias_columnas)
    mean_filas = np.mean(entropias_filas)
    mean_columnas = np.mean(entropias_columnas)
    
    # Score: minimizar la suma de stds (menor variación = mejor)
    suma_std = std_filas + std_columnas
    score = max(0, 100 - (suma_std / 8) * 100)
    score = min(score, 100)
    
    return {
        'entropias_filas': entropias_filas,
        'entropias_columnas': entropias_columnas,
        'std_filas': std_filas,
        'std_columnas': std_columnas,
        'mean_filas': mean_filas,
        'mean_columnas': mean_columnas,
        'suma_std': suma_std,
        'score': score,
        'img_filas': img_filas,
        'img_columnas': img_columnas
    }

def score_warp_entropy(warped):
    """
    Calcula el score de un warp basado en la desviación estándar de las entropías.
    Score = 100 - (suma_std / 8) * 100
    Menor std = mejor score
    """
    if warped is None:
        return 0, None
    
    stats = calcular_entropias_warp(warped)
    if stats is None:
        return 0, None
    
    return stats['score'], stats

# ========== FUNCIONES DE PROCESAMIENTO ==========
def find_largest_mesh(quadrilaterals, threshold=ADJACENT_THRESHOLD):
    """
    Encuentra la malla más grande de cuadriláteros adyacentes.
    Retorna la lista de cuadriláteros que forman la malla más grande.
    """
    if len(quadrilaterals) == 0:
        return []
    
    def get_vertices(polygon):
        vertices = []
        for point in polygon:
            vertices.append((int(point[0][0]), int(point[0][1])))
        return vertices
    
    def are_adjacent(poly1, poly2, threshold):
        vertices1 = get_vertices(poly1)
        vertices2 = get_vertices(poly2)
        
        close_points = 0
        for v1 in vertices1:
            for v2 in vertices2:
                dist = np.sqrt((v1[0] - v2[0])**2 + (v1[1] - v2[1])**2)
                if dist <= threshold:
                    close_points += 1
                    break
        
        return close_points >= 2
    
    n = len(quadrilaterals)
    adjacency = [[] for _ in range(n)]
    
    for i in range(n):
        for j in range(i+1, n):
            if are_adjacent(quadrilaterals[i], quadrilaterals[j], threshold):
                adjacency[i].append(j)
                adjacency[j].append(i)
    
    visited = [False] * n
    largest_mesh = []
    
    for i in range(n):
        if not visited[i]:
            queue = deque([i])
            visited[i] = True
            current_mesh = []
            
            while queue:
                current = queue.popleft()
                current_mesh.append(current)
                
                for neighbor in adjacency[current]:
                    if not visited[neighbor]:
                        visited[neighbor] = True
                        queue.append(neighbor)
            
            if len(current_mesh) > len(largest_mesh):
                largest_mesh = current_mesh
    
    return [quadrilaterals[idx] for idx in largest_mesh]

def get_vertices_as_points(polygon):
    """Convierte un polígono a lista de puntos (x,y) en orden"""
    vertices = []
    for point in polygon:
        vertices.append((int(point[0][0]), int(point[0][1])))
    return vertices

def fit_polygon_to_mesh(mesh_quadrilaterals):
    """
    Ajusta un polígono de 4 lados que contenga todos los cuadriláteros de la malla.
    Retorna el polígono ajustado y las dimensiones de la grilla (rows, cols).
    """
    if len(mesh_quadrilaterals) == 0:
        return None, 0, 0
    
    # Obtener todos los vértices de todos los cuadriláteros
    all_vertices = []
    for quad in mesh_quadrilaterals:
        vertices = get_vertices_as_points(quad)
        all_vertices.extend(vertices)
    
    # Convertir a array numpy
    points = np.array(all_vertices, dtype=np.float32)
    
    # Encontrar el rectángulo mínimo que contiene todos los puntos (orientado)
    rect = cv.minAreaRect(points)
    box = cv.boxPoints(rect)
    box = np.array(box, dtype=np.float32)
    
    # Ordenar los puntos en sentido antihorario comenzando desde el superior-izquierdo
    center = np.mean(box, axis=0)
    angles = np.arctan2(box[:, 1] - center[1], box[:, 0] - center[0])
    sorted_indices = np.argsort(angles)
    box = box[sorted_indices]
    
    min_sum_idx = np.argmin(box[:, 0] + box[:, 1])
    box = np.roll(box, -min_sum_idx, axis=0)
    
    # Verificar que esté en sentido antihorario
    area = 0
    for i in range(4):
        j = (i + 1) % 4
        area += box[i, 0] * box[j, 1]
        area -= box[j, 0] * box[i, 1]
    
    if area > 0:
        box = box[::-1]
        min_sum_idx = np.argmin(box[:, 0] + box[:, 1])
        box = np.roll(box, -min_sum_idx, axis=0)
    
    # Calcular el número de celdas en la malla
    ref_quad = mesh_quadrilaterals[0]
    ref_vertices = get_vertices_as_points(ref_quad)
    
    widths = []
    heights = []
    for i in range(4):
        j = (i + 1) % 4
        dist = np.sqrt((ref_vertices[i][0] - ref_vertices[j][0])**2 + 
                       (ref_vertices[i][1] - ref_vertices[j][1])**2)
        if abs(ref_vertices[i][0] - ref_vertices[j][0]) > abs(ref_vertices[i][1] - ref_vertices[j][1]):
            widths.append(dist)
        else:
            heights.append(dist)
    
    avg_cell_width = np.mean(widths) if widths else 50
    avg_cell_height = np.mean(heights) if heights else 50
    
    rect_width = np.sqrt((box[0][0] - box[1][0])**2 + (box[0][1] - box[1][1])**2)
    rect_height = np.sqrt((box[0][0] - box[3][0])**2 + (box[0][1] - box[3][1])**2)
    
    num_cols = max(2, int(rect_width / avg_cell_width + 0.5))
    num_rows = max(2, int(rect_height / avg_cell_height + 0.5))
    
    num_cols = min(num_cols, 8)
    num_rows = min(num_rows, 8)
    
    return box, num_rows, num_cols

def expand_polygon_with_perspective(polygon, row_start, col_start, grid_rows, grid_cols, target_rows=8, target_cols=8):
    """
    Expande un polígono que contiene la malla para que ocupe todo el tablero de 8x8.
    """
    p1 = polygon[0]  # top-left
    p2 = polygon[1]  # top-right
    p3 = polygon[2]  # bottom-right
    p4 = polygon[3]  # bottom-left
    
    top_vec = p2 - p1
    bottom_vec = p3 - p4
    left_vec = p4 - p1
    right_vec = p3 - p2
    
    cells_up = row_start
    cells_down = target_rows - grid_rows - row_start
    cells_left = col_start
    cells_right = target_cols - grid_cols - col_start
    
    top_left = p1 - (cells_up * left_vec / (cells_up + cells_down + 1)) - (cells_left * top_vec / (cells_left + cells_right + 1))
    top_right = p2 - (cells_up * right_vec / (cells_up + cells_down + 1)) + (cells_right * top_vec / (cells_left + cells_right + 1))
    bottom_left = p4 + (cells_down * left_vec / (cells_up + cells_down + 1)) - (cells_left * bottom_vec / (cells_left + cells_right + 1))
    bottom_right = p3 + (cells_down * right_vec / (cells_up + cells_down + 1)) + (cells_right * bottom_vec / (cells_left + cells_right + 1))
    
    top_left = (int(top_left[0]), int(top_left[1]))
    top_right = (int(top_right[0]), int(top_right[1]))
    bottom_right = (int(bottom_right[0]), int(bottom_right[1]))
    bottom_left = (int(bottom_left[0]), int(bottom_left[1]))
    
    return np.array([top_left, top_right, bottom_right, bottom_left], dtype=np.float32)

def apply_warp(frame, src_points, dst_size=(800, 800)):
    """Aplica warp perspective a la imagen"""
    dst_points = np.array([
        [0, 0],
        [dst_size[0]-1, 0],
        [dst_size[0]-1, dst_size[1]-1],
        [0, dst_size[1]-1]
    ], dtype=np.float32)
    
    M = cv.getPerspectiveTransform(src_points, dst_points)
    warped = cv.warpPerspective(frame, M, dst_size)
    return warped

def find_best_warp_and_board(frame, largest_mesh):
    """Encuentra el mejor warp usando el polígono ajustado a la malla."""
    
    # Ajustar polígono a la malla
    mesh_polygon, num_rows, num_cols = fit_polygon_to_mesh(largest_mesh)
    
    if mesh_polygon is None:
        return None, -float('inf'), None, None, None, None
    
    print(f"Malla: {num_rows}x{num_cols} celdas")
    
    best_warp = None
    best_score = -float('inf')
    best_cell_info = None
    best_board_points = None
    best_stats = None
    best_cell_original = None  # Guardar la celda que generó el mejor warp
    
    # Probar diferentes posiciones iniciales para el polígono en la grilla 8x8
    for row_start in range(0, 8 - num_rows + 1):
        for col_start in range(0, 8 - num_cols + 1):
            # Expandir el polígono para que ocupe todo el tablero 8x8
            src_points = expand_polygon_with_perspective(
                mesh_polygon, row_start, col_start, num_rows, num_cols
            )
            
            # Verificar que los puntos estén dentro de la imagen
            h, w = frame.shape[:2]
            valid = True
            for pt in src_points:
                if pt[0] < 0 or pt[0] >= w or pt[1] < 0 or pt[1] >= h:
                    valid = False
                    break
            
            if not valid:
                continue
            
            # Aplicar warp
            warped = apply_warp(frame, src_points)
            
            # Calcular score basado en entropía
            score, stats = score_warp_entropy(warped)
            
            if score > best_score:
                best_score = score
                best_warp = warped
                best_cell_info = (row_start, col_start, num_rows, num_cols, src_points)
                best_board_points = src_points
                best_stats = stats
                best_cell_original = largest_mesh[0] if len(largest_mesh) > 0 else None
    
    # Si no se encontró ningún warp válido, intentar con el primer warp disponible
    if best_warp is None and len(largest_mesh) > 0:
        # Intentar con la primera celda y posición (0,0)
        src_points = expand_polygon_with_perspective(
            mesh_polygon, 0, 0, num_rows, num_cols
        )
        warped = apply_warp(frame, src_points)
        score, stats = score_warp_entropy(warped)
        best_warp = warped
        best_score = score
        best_cell_info = (0, 0, num_rows, num_cols, src_points)
        best_board_points = src_points
        best_stats = stats
        best_cell_original = largest_mesh[0] if len(largest_mesh) > 0 else None
    
    return best_warp, best_score, best_cell_info, best_board_points, best_stats, best_cell_original

def iterative_update_vertices(current_vertices, new_vertices, learning_rate=LEARNING_RATE):
    """
    Actualiza los vértices actuales hacia los nuevos vértices usando un factor de aprendizaje.
    """
    if current_vertices is None or len(current_vertices) != len(new_vertices):
        return new_vertices
    
    updated_vertices = []
    for curr, new in zip(current_vertices, new_vertices):
        dx = new[0] - curr[0]
        dy = new[1] - curr[1]
        new_x = int(curr[0] + learning_rate * dx)
        new_y = int(curr[1] + learning_rate * dy)
        updated_vertices.append((new_x, new_y))
    
    return np.array(updated_vertices, dtype=np.float32)

def crear_grafico_entropias(entropias_filas, entropias_columnas, stats, titulo="Entropías"):
    """Crea un gráfico de barras con las entropías usando matplotlib"""
    import matplotlib.pyplot as plt
    from io import BytesIO
    
    n = len(entropias_filas)
    x = np.arange(n)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    
    bars1 = ax1.bar(x, entropias_filas, color='blue', alpha=0.7)
    ax1.set_xlabel('Fila')
    ax1.set_ylabel('Entropía (bits)')
    ax1.set_title(f'{titulo} - Filas\nStd: {stats["std_filas"]:.4f}, Mean: {stats["mean_filas"]:.4f}')
    ax1.set_xticks(x)
    ax1.set_xticklabels([f'F{i+1}' for i in range(n)])
    ax1.grid(True, alpha=0.3)
    ax1.axhline(y=stats['mean_filas'], color='red', linestyle='--', linewidth=2)
    ax1.axhline(y=stats['mean_filas'] + stats['std_filas'], color='orange', linestyle=':', linewidth=1, alpha=0.7)
    ax1.axhline(y=stats['mean_filas'] - stats['std_filas'], color='orange', linestyle=':', linewidth=1, alpha=0.7)
    
    bars2 = ax2.bar(x, entropias_columnas, color='green', alpha=0.7)
    ax2.set_xlabel('Columna')
    ax2.set_ylabel('Entropía (bits)')
    ax2.set_title(f'{titulo} - Columnas\nStd: {stats["std_columnas"]:.4f}, Mean: {stats["mean_columnas"]:.4f}')
    ax2.set_xticks(x)
    ax2.set_xticklabels([f'C{i+1}' for i in range(n)])
    ax2.grid(True, alpha=0.3)
    ax2.axhline(y=stats['mean_columnas'], color='red', linestyle='--', linewidth=2)
    ax2.axhline(y=stats['mean_columnas'] + stats['std_columnas'], color='orange', linestyle=':', linewidth=1, alpha=0.7)
    ax2.axhline(y=stats['mean_columnas'] - stats['std_columnas'], color='orange', linestyle=':', linewidth=1, alpha=0.7)
    
    plt.tight_layout()
    
    buf = BytesIO()
    plt.savefig(buf, format='png', bbox_inches='tight', dpi=80)
    buf.seek(0)
    img_array = np.frombuffer(buf.getvalue(), dtype=np.uint8)
    img_plot = cv.imdecode(img_array, cv.IMREAD_COLOR)
    plt.close()
    
    return img_plot

while True:

    # Solo leer un nuevo frame si cambió la posición
    if current_frame != last_frame:
        cap.set(cv.CAP_PROP_POS_FRAMES, current_frame)
        ret, frame = cap.read()

        if ret:
            frame_vis = frame.copy()
            gray = cv.cvtColor(frame, cv.COLOR_BGR2GRAY)

            # ========== CÁLCULO DE PUNTOS DE SILLA ==========
            Ixx = cv.Sobel(gray, cv.CV_32F, 2, 0, ksize=3)
            Iyy = cv.Sobel(gray, cv.CV_32F, 0, 2, ksize=3)
            Ixy = cv.Sobel(gray, cv.CV_32F, 1, 1, ksize=3)

            response = -(Ixx*Iyy - Ixy*Ixy)
            response = cv.GaussianBlur(response, (15,15), 0)

            mx = cv.dilate(response, np.ones((7,7), np.uint8))
            mask = np.zeros_like(gray)
            th = 0.15 * response.max()

            pts = np.where((response == mx) & (response > th))
            points = np.column_stack((pts[1], pts[0]))
            mask[pts] = 255

            cv.imshow("Saddle mask", mask)

            # ========== SOBEL Y CANNY ==========
            sobelx = cv.Sobel(gray, cv.CV_64F, 1, 0, ksize=3)
            sobely = cv.Sobel(gray, cv.CV_64F, 0, 1, ksize=3)
            sobel_magnitude = np.sqrt(sobelx**2 + sobely**2)
            sobel_magnitude = np.uint8(np.clip(sobel_magnitude, 0, 255))

            cv.imshow(sobel_window, sobel_magnitude)

            edges = cv.Canny(sobel_magnitude, 7000, 7050, apertureSize=5)
            cv.imshow(canny_window, edges)

            # ========== PASO 1: ENCONTRAR TODOS LOS CONTORNOS ==========
            contours, hierarchy = cv.findContours(edges, cv.RETR_EXTERNAL, cv.CHAIN_APPROX_SIMPLE)

            contour_img = frame.copy()
            cv.drawContours(contour_img, contours, -1, (0, 255, 0), 2)
            cv.imshow(contour_window, contour_img)

            # ========== PASO 2: APROXIMAR POLÍGONOS ==========
            all_polygons = []
            contour_approx_img = frame.copy()
            
            for contour in contours:
                epsilon = 0.01 * cv.arcLength(contour, True)
                approx = cv.approxPolyDP(contour, epsilon, True)
                all_polygons.append(approx)
                cv.drawContours(contour_approx_img, [approx], -1, (255, 0, 0), 3)
                if len(approx) == 4:
                    for point in approx:
                        cv.circle(contour_approx_img, tuple(point[0]), 5, (0, 255, 255), -1)

            cv.imshow(approx_window, contour_approx_img)

            # ========== PASO 3: FILTRAR POLÍGONOS POR PUNTOS DE SILLA ==========
            saddle_polygons = []
            saddle_points = [(int(p[0]), int(p[1])) for p in points]
            
            for polygon in all_polygons:
                contains_saddle = False
                for point in saddle_points:
                    distance = cv.pointPolygonTest(polygon, point, True)
                    if distance >= -TOLERANCE:
                        contains_saddle = True
                        break
                if contains_saddle:
                    saddle_polygons.append(polygon)

            contour_filtered_img = frame.copy()
            cv.drawContours(contour_filtered_img, saddle_polygons, -1, (0, 0, 255), 3)
            cv.putText(contour_filtered_img, f"Tolerance: {TOLERANCE}px", 
                      (10, 30), cv.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
            cv.imshow(contour_filtered_window, contour_filtered_img)

            # ========== DETECCIÓN DE CUADRILÁTEROS ==========
            quadrilaterals_img = frame.copy()
            quadrilaterals = []
            
            for polygon in saddle_polygons:
                if len(polygon) == 4:
                    area = cv.contourArea(polygon)
                    if cv.isContourConvex(polygon) and area >= MIN_AREA:
                        quadrilaterals.append(polygon)
                        cv.drawContours(quadrilaterals_img, [polygon], -1, (0, 0, 255), 3)
                        for point in polygon:
                            cv.circle(quadrilaterals_img, tuple(point[0]), 6, (0, 255, 255), -1)
                        cv.putText(quadrilaterals_img, f"Area: {int(area)}", 
                                  (int(polygon[0][0][0]), int(polygon[0][0][1]) - 20), 
                                  cv.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2)
            
            all_quadrilaterals.extend(quadrilaterals)
            cv.imshow(quadrilaterals_window, quadrilaterals_img)

            # ========== ENCONTRAR LA MALLA MÁS GRANDE ==========
            mesh_img = frame.copy()
            largest_mesh = []
            
            if len(quadrilaterals) > 0:
                largest_mesh = find_largest_mesh(quadrilaterals, ADJACENT_THRESHOLD)
                
                if len(largest_mesh) > 0:
                    for polygon in largest_mesh:
                        pts = polygon.reshape(-1, 2).astype(np.int32)
                        cv.fillPoly(mesh_img, [pts], (128, 128, 128))
                        cv.drawContours(mesh_img, [polygon], -1, (0, 0, 255), 1)
                    
                    cv.putText(mesh_img, f"Mesh size: {len(largest_mesh)} quadrilaterals", 
                              (10, 30), cv.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
            
            cv.imshow(mesh_window, mesh_img)

            # ========== EXPANDIR CUADRILÁTEROS 1px Y MOSTRAR ==========
            if len(largest_mesh) > 0:
                mesh_expanded_img = frame.copy()
                mesh_expanded = []
                
                for polygon in largest_mesh:
                    # Expandir el cuadrilátero 1px hacia afuera
                    expanded_polygon = expand_quadrilateral_npx(polygon, 2)
                    mesh_expanded.append(expanded_polygon)
                    
                    # Dibujar en la imagen
                    pts = expanded_polygon.reshape(-1, 2).astype(np.int32)
                    cv.fillPoly(mesh_expanded_img, [pts], (128, 128, 128))
                    cv.drawContours(mesh_expanded_img, [expanded_polygon], -1, (0, 255, 0), 1)  # Verde
                
                cv.putText(mesh_expanded_img, f"Mesh expandida 10px: {len(mesh_expanded)} quadrilaterals", 
                          (10, 30), cv.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
                cv.imshow("Malla expandida 10px", mesh_expanded_img)
            else:
                cv.imshow("Malla expandida 10px", np.zeros_like(frame))

            # ========== ENCONTRAR EL MEJOR WARP USANDO LA MALLA EXPANDIDA ==========
            warp_result_img = np.zeros((800, 800, 3), dtype=np.uint8)
            estimated_board_img = frame.copy()
            
            if len(largest_mesh) >= 3:
                # Expandir todos los cuadriláteros de la malla 1px antes de buscar el warp
                expanded_mesh = []
                for polygon in largest_mesh:
                    expanded_polygon = expand_quadrilateral_npx(polygon, 2)
                    expanded_mesh.append(expanded_polygon)
                
                best_warp, best_score, best_info, best_board_points, best_stats, best_cell_original = find_best_warp_and_board(
                    frame, expanded_mesh
                )
                
                # ===== APLICAR ACTUALIZACIÓN ITERATIVA DE VÉRTICES =====
                if best_board_points is not None:
                    new_vertices = [(int(pt[0]), int(pt[1])) for pt in best_board_points]
                    
                    if previous_board_vertices is not None:
                        if len(previous_board_vertices) == 4 and len(new_vertices) == 4:
                            updated_vertices = iterative_update_vertices(
                                previous_board_vertices, 
                                new_vertices,
                                LEARNING_RATE
                            )
                            
                            print(f"Diferencia: {np.mean(np.abs(np.array(new_vertices) - np.array(previous_board_vertices)))}")
                            best_board_points = updated_vertices
                    
                    previous_board_vertices = [(int(pt[0]), int(pt[1])) for pt in best_board_points]
                
                # ===== MOSTRAR TABLERO ESTIMADO =====
                if best_board_points is not None:
                    pts = best_board_points.astype(np.int32)
                    cv.polylines(estimated_board_img, [pts], True, (0, 255, 0), 4)
                    for point in pts:
                        cv.circle(estimated_board_img, tuple(point), 10, (0, 255, 255), -1)
                    
                    # ===== MOSTRAR LA CELDA QUE GENERÓ EL WARP (EN ROJO) =====
                    if best_cell_original is not None:
                        cell_pts = best_cell_original.reshape(-1, 2).astype(np.int32)
                        cv.polylines(estimated_board_img, [cell_pts], True, (0, 0, 255), 1)
                        for point in cell_pts:
                            cv.circle(estimated_board_img, tuple(point), 6, (0, 0, 255), -1)
                        cv.putText(estimated_board_img, "Celda generadora", 
                                  (cell_pts[0][0] - 30, cell_pts[0][1] - 20), 
                                  cv.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1)
                    
                    if best_info is not None:
                        row_start, col_start, num_rows, num_cols, src_points = best_info
                        cv.putText(estimated_board_img, f"Malla: {num_rows}x{num_cols} - Pos: ({row_start},{col_start})", 
                                  (10, 30), cv.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
                        cv.putText(estimated_board_img, f"Score: {best_score:.1f}/100", 
                                  (10, 60), cv.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
                        cv.putText(estimated_board_img, f"LR: {LEARNING_RATE}", 
                                  (10, 90), cv.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255), 2)
                        
                        if best_stats is not None:
                            cv.putText(estimated_board_img, f"Std Filas: {best_stats['std_filas']:.3f}", 
                                      (10, 120), cv.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255), 1)
                            cv.putText(estimated_board_img, f"Std Cols: {best_stats['std_columnas']:.3f}", 
                                      (10, 140), cv.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255), 1)
                            cv.putText(estimated_board_img, f"Suma Std: {best_stats['suma_std']:.3f}", 
                                      (10, 160), cv.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255), 1)
                
                # ===== MOSTRAR WARP =====
                if best_warp is not None:
                    warp_result_img = best_warp
                    row_start, col_start, num_rows, num_cols, src_points = best_info
                    
                    cv.putText(warp_result_img, f"Score: {best_score:.2f}/100", 
                              (10, 30), cv.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
                    cv.putText(warp_result_img, f"Malla: {num_rows}x{num_cols} - Pos: ({row_start},{col_start})", 
                              (10, 60), cv.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
                    
                    # Dibujar la grilla 8x8
                    h, w = warp_result_img.shape[:2]
                    cell_h = h // 8
                    cell_w = w // 8
                    for i in range(9):
                        y = i * cell_h
                        cv.line(warp_result_img, (0, y), (w, y), (0, 255, 255), 1)
                        x = i * cell_w
                        cv.line(warp_result_img, (x, 0), (x, h), (0, 255, 255), 1)
                    
                    if best_score >= 70:
                        color = (0, 255, 0)
                    elif best_score >= 50:
                        color = (0, 255, 255)
                    else:
                        color = (0, 0, 255)
                    
                    cv.rectangle(warp_result_img, (w-200, 10), (w-10, 50), (0, 0, 0), -1)
                    cv.putText(warp_result_img, f"{best_score:.1f}%", 
                              (w-180, 40), cv.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)
                    
                    # Mostrar imágenes de entropías
                    if best_stats is not None:
                        cv.imshow(entropy_filas_window, best_stats['img_filas'])
                        cv.imshow(entropy_columnas_window, best_stats['img_columnas'])
                        
                        try:
                            grafico = crear_grafico_entropias(
                                best_stats['entropias_filas'],
                                best_stats['entropias_columnas'],
                                best_stats,
                                f"WARP - Score: {best_score:.1f}"
                            )
                            if grafico is not None:
                                cv.imshow("Grafico Entropias", grafico)
                        except:
                            pass
            
            cv.imshow(estimated_board_window, estimated_board_img)
            cv.imshow(warp_window, warp_result_img)

            # Mostrar puntos de silla
            saddle_points_img = frame.copy()
            if len(points) > 0:
                for point in points:
                    cv.circle(saddle_points_img, (int(point[0]), int(point[1])), 4, (0, 255, 0), -1)

            cv.imshow(hough_filtered_window, saddle_points_img)
            cv.imshow(window_name, frame_vis)

            last_frame = current_frame

    key = cv.waitKey(20) & 0xFF

    if key == 27:      # ESC para salir
        break

cap.release()
cv.destroyAllWindows()

Frames totales: 430
FPS: 29.97


In [ ]:
# PRUEBA CON VIDEO CON RANSAC

import cv2 as cv
import numpy as np
from collections import deque
import random
from copy import copy
from numpy.random import default_rng
import itertools

rng = default_rng()

# Ruta al video
video_path = "../../../data/raw/Prueba2.mp4"

# Abrir el video
cap = cv.VideoCapture(video_path)

if not cap.isOpened():
    raise IOError(f"No se pudo abrir el video: {video_path}")

# Obtener información del video
total_frames = int(cap.get(cv.CAP_PROP_FRAME_COUNT))
fps = cap.get(cv.CAP_PROP_FPS)

print(f"Frames totales: {total_frames}")
print(f"FPS: {fps:.2f}")

# Crear ventanas
window_name = "Video"
sobel_window = "Sobel"
canny_window = "Canny"
canny_dilated_window = "Canny Dilatado"
hough_filtered_window = "Saddle Points"
contour_window = "Contornos"
approx_window = "Polígonos aproximados"
quadrilaterals_window = "Cuadriláteros detectados"
estimated_board_window = "Cuadrilátero con más puntos silla"
warp_window = "Warp final"
controls_window = "Controles"
ransac_window = "RANSAC - Rectas ajustadas"
polygons_binary_window = "Máscara binaria de polígonos"
intersecciones_window = "Intersecciones - Puntos y Cuadriláteros"

cv.namedWindow(window_name, cv.WINDOW_NORMAL)
cv.namedWindow(sobel_window, cv.WINDOW_NORMAL)
cv.namedWindow(canny_window, cv.WINDOW_NORMAL)
cv.namedWindow(canny_dilated_window, cv.WINDOW_NORMAL)
cv.namedWindow(hough_filtered_window, cv.WINDOW_NORMAL)
cv.namedWindow(contour_window, cv.WINDOW_NORMAL)
cv.namedWindow(approx_window, cv.WINDOW_NORMAL)
cv.namedWindow(quadrilaterals_window, cv.WINDOW_NORMAL)
cv.namedWindow(estimated_board_window, cv.WINDOW_NORMAL)
cv.namedWindow(warp_window, cv.WINDOW_NORMAL)
cv.namedWindow(controls_window, cv.WINDOW_NORMAL)
cv.namedWindow(ransac_window, cv.WINDOW_NORMAL)
cv.namedWindow(polygons_binary_window, cv.WINDOW_NORMAL)
cv.namedWindow(intersecciones_window, cv.WINDOW_NORMAL)

# Variable para detectar cambios del slider
current_frame = 0

def on_trackbar(pos):
    global current_frame
    current_frame = pos

# Crear la trackbar para el frame
cv.createTrackbar(
    "Frame",
    window_name,
    0,
    total_frames - 1,
    on_trackbar
)

# Variable para el offset
offset_pixels = 0

def on_offset_trackbar(pos):
    global offset_pixels
    offset_pixels = pos

# Crear trackbar para el offset (0-50 píxeles)
cv.createTrackbar(
    "Offset",
    controls_window,
    0,
    50,
    on_offset_trackbar
)

# Variables para RANSAC
RANSAC_TOLERANCIA_PX = 5
RANSAC_ITERACIONES = 50  # Solo 50 iteraciones para RANSAC
RANSAC_RADIO_BUSQUEDA = 30
RANSAC_MIN_INLIERS = 5
RANSAC_NUM_RECTAS = 8

# Porcentaje de tolerancia para reciclaje (10%)
RECYCLE_THRESHOLD_PERCENT = 10

def on_tol_px_trackbar(pos):
    global RANSAC_TOLERANCIA_PX
    RANSAC_TOLERANCIA_PX = max(1, pos)

cv.createTrackbar(
    "Tolerancia (px)",
    controls_window,
    5,
    50,
    on_tol_px_trackbar
)

def on_radio_busqueda_trackbar(pos):
    global RANSAC_RADIO_BUSQUEDA
    RANSAC_RADIO_BUSQUEDA = max(1, pos)

cv.createTrackbar(
    "Radio busqueda",
    controls_window,
    1,
    200,
    on_radio_busqueda_trackbar
)

def on_min_inliers_trackbar(pos):
    global RANSAC_MIN_INLIERS
    RANSAC_MIN_INLIERS = max(1, pos)

cv.createTrackbar(
    "Min Inliers",
    controls_window,
    1,
    100,
    on_min_inliers_trackbar
)

def on_num_rectas_trackbar(pos):
    global RANSAC_NUM_RECTAS
    RANSAC_NUM_RECTAS = max(4, pos)

cv.createTrackbar(
    "Num Rectas",
    controls_window,
    4,
    12,
    on_num_rectas_trackbar
)

def on_recycle_threshold_trackbar(pos):
    global RECYCLE_THRESHOLD_PERCENT
    RECYCLE_THRESHOLD_PERCENT = max(1, pos)

cv.createTrackbar(
    "Tolerancia Reciclaje %",
    controls_window,
    1,
    100,
    on_recycle_threshold_trackbar
)

last_frame = -1

# Parámetros
TOLERANCE = 5
MIN_AREA = 10
MIN_PUNTOS_SILLA = 50

# Variables para almacenar el último cuadrilátero válido
ultimo_cuadrilatero_valido = None
ultimo_warp_valido = None
ultimo_frame_warp = None
ultimos_puntos_ordenados = None
ultimo_num_puntos = 0
ultimo_area = 0

# Flag para saber si ya se ejecutó RANSAC
ransac_ejecutado = False

def get_vertices_as_points(polygon):
    vertices = []
    for point in polygon:
        vertices.append((int(point[0][0]), int(point[0][1])))
    return vertices

def apply_warp(frame, src_points, dst_size=(800, 800)):
    dst_points = np.array([
        [0, 0],
        [dst_size[0]-1, 0],
        [dst_size[0]-1, dst_size[1]-1],
        [0, dst_size[1]-1]
    ], dtype=np.float32)
    
    src_points = np.array(src_points, dtype=np.float32)
    M = cv.getPerspectiveTransform(src_points, dst_points)
    warped = cv.warpPerspective(frame, M, dst_size)
    return warped

def aplicar_offset_a_puntos(puntos, offset, centro):
    if offset == 0:
        return puntos
    
    puntos_con_offset = []
    for punto in puntos:
        vector = centro - punto
        norm = np.linalg.norm(vector)
        if norm > 0:
            nuevo_punto = punto + (vector / norm) * offset
        else:
            nuevo_punto = punto
        puntos_con_offset.append(nuevo_punto)
    
    return np.array(puntos_con_offset, dtype=np.float32)

def obtener_puntos_de_imagen_binaria(imagen_binaria):
    pts = np.argwhere(imagen_binaria > 0)
    puntos = []
    for p in pts:
        y, x = p
        puntos.append((x, y))
    return puntos

def encontrar_punto_cercano(puntos, punto_origen, radio_busqueda):
    puntos_cercanos = []
    for p in puntos:
        dist = np.sqrt((p[0] - punto_origen[0])**2 + (p[1] - punto_origen[1])**2)
        if dist <= radio_busqueda and dist > 0:
            puntos_cercanos.append(p)
    
    if len(puntos_cercanos) == 0:
        return None
    
    return random.choice(puntos_cercanos)

def ajustar_recta_ransac_simple(puntos, tolerancia_px, radio_busqueda, iteraciones):
    if len(puntos) < 2:
        return None, [], 0
    
    mejor_recta = None
    mejores_inliers = []
    max_inliers = 0
    
    puntos_array = np.array(puntos)
    
    for _ in range(iteraciones):
        if len(puntos_array) < 2:
            break
        
        idx1 = random.randint(0, len(puntos_array) - 1)
        p1 = puntos_array[idx1]
        
        p2 = encontrar_punto_cercano(puntos_array.tolist(), p1, radio_busqueda)
        if p2 is None:
            continue
        
        x1, y1 = p1[0], p1[1]
        x2, y2 = p2[0], p2[1]
        
        if abs(x2 - x1) < 1e-6:
            continue
        
        m = (y2 - y1) / (x2 - x1)
        b = y1 - m * x1
        recta = (m, b)
        
        inliers = []
        for punto in puntos_array:
            x, y_p = punto[0], punto[1]
            y_pred = m * x + b
            dist = abs(y_p - y_pred) / np.sqrt(m**2 + 1)
            if dist <= tolerancia_px:
                inliers.append(punto)
        
        if len(inliers) > max_inliers:
            max_inliers = len(inliers)
            mejores_inliers = inliers
            mejor_recta = recta
    
    if mejor_recta is not None and len(mejores_inliers) >= 2:
        xs = [p[0] for p in mejores_inliers]
        ys = [p[1] for p in mejores_inliers]
        x = np.array(xs)
        y = np.array(ys)
        A = np.vstack([x, np.ones(len(x))]).T
        m, b = np.linalg.lstsq(A, y, rcond=None)[0]
        mejor_recta = (m, b)
        
        inliers_finales = []
        m, b = mejor_recta
        for punto in puntos:
            x, y_p = punto[0], punto[1]
            y_pred = m * x + b
            dist = abs(y_p - y_pred) / np.sqrt(m**2 + 1)
            if dist <= tolerancia_px:
                inliers_finales.append(punto)
        return mejor_recta, inliers_finales, len(inliers_finales)
    
    return mejor_recta, mejores_inliers, max_inliers

def encontrar_mejores_rectas_ransac(imagen_binaria, tolerancia_px, radio_busqueda, iteraciones, min_inliers, num_rectas=8):
    if imagen_binaria is None:
        return None, None
    
    img_temp = imagen_binaria.copy()
    rectas_horizontal = []
    
    for i in range(num_rectas // 2 + 2):
        puntos = obtener_puntos_de_imagen_binaria(img_temp)
        if len(puntos) < min_inliers:
            break
        
        recta, inliers, num_inliers = ajustar_recta_ransac_simple(
            puntos, tolerancia_px, radio_busqueda, iteraciones
        )
        
        if recta is None or num_inliers < min_inliers:
            break
        
        m, b = recta
        rectas_horizontal.append({
            'm': m,
            'b': b,
            'num_inliers': num_inliers,
            'inliers': inliers,
            'origen': 'horizontal'
        })
        
        for punto in inliers:
            x, y = int(punto[0]), int(punto[1])
            if 0 <= y < img_temp.shape[0] and 0 <= x < img_temp.shape[1]:
                img_temp[y, x] = 0
    
    imagen_transpuesta = imagen_binaria.T
    img_temp_vert = imagen_transpuesta.copy()
    rectas_vertical = []
    
    for i in range(num_rectas // 2 + 2):
        puntos = obtener_puntos_de_imagen_binaria(img_temp_vert)
        if len(puntos) < min_inliers:
            break
        
        recta, inliers, num_inliers = ajustar_recta_ransac_simple(
            puntos, tolerancia_px, radio_busqueda, iteraciones
        )
        
        if recta is None or num_inliers < min_inliers:
            break
        
        m_orig, b_orig = recta
        if abs(m_orig) > 1e-6:
            m_conv = 1.0 / m_orig
            b_conv = -b_orig / m_orig
        else:
            m_conv = 0
            b_conv = b_orig
        
        rectas_vertical.append({
            'm': m_conv,
            'b': b_conv,
            'num_inliers': num_inliers,
            'inliers': inliers,
            'origen': 'vertical'
        })
        
        for punto in inliers:
            x, y = int(punto[0]), int(punto[1])
            if 0 <= y < img_temp_vert.shape[0] and 0 <= x < img_temp_vert.shape[1]:
                img_temp_vert[y, x] = 0
    
    return rectas_horizontal, rectas_vertical

def calcular_interseccion(m1, b1, m2, b2):
    if abs(m1 - m2) < 1e-6:
        return None
    x = (b2 - b1) / (m1 - m2)
    y = m1 * x + b1
    return (int(x), int(y))

def ordenar_vertices_horario(vertices):
    if len(vertices) != 4:
        return None
    
    center = np.mean(vertices, axis=0)
    angles = np.arctan2(vertices[:, 1] - center[1], vertices[:, 0] - center[0])
    sorted_indices = np.argsort(angles)
    vertices_ordenados = vertices[sorted_indices]
    
    vertices_ordenados = vertices_ordenados[::-1]
    
    min_sum_idx = np.argmin(vertices_ordenados[:, 0] + vertices_ordenados[:, 1])
    vertices_ordenados = np.roll(vertices_ordenados, -min_sum_idx, axis=0)
    
    return vertices_ordenados

def formar_cuadrilateros_desde_rectas(rectas_horizontal, rectas_vertical, saddle_points, frame_shape):
    h, w = frame_shape[:2]
    cuadrilateros = []
    
    for combo_h in itertools.combinations(range(len(rectas_horizontal)), 2):
        for combo_v in itertools.combinations(range(len(rectas_vertical)), 2):
            h1 = rectas_horizontal[combo_h[0]]
            h2 = rectas_horizontal[combo_h[1]]
            v1 = rectas_vertical[combo_v[0]]
            v2 = rectas_vertical[combo_v[1]]
            
            h1_m, h1_b = h1['m'], h1['b']
            h2_m, h2_b = h2['m'], h2['b']
            v1_m, v1_b = v1['m'], v1['b']
            v2_m, v2_b = v2['m'], v2['b']
            
            inter1 = calcular_interseccion(h1_m, h1_b, v1_m, v1_b)
            inter2 = calcular_interseccion(h1_m, h1_b, v2_m, v2_b)
            inter3 = calcular_interseccion(h2_m, h2_b, v2_m, v2_b)
            inter4 = calcular_interseccion(h2_m, h2_b, v1_m, v1_b)
            
            if None in [inter1, inter2, inter3, inter4]:
                continue
            
            vertices = [inter1, inter2, inter3, inter4]
            
            dentro = True
            for v in vertices:
                if v[0] < 0 or v[0] >= w or v[1] < 0 or v[1] >= h:
                    dentro = False
                    break
            
            if not dentro:
                continue
            
            vertices_array = np.array(vertices, dtype=np.float32)
            vertices_ordenados = ordenar_vertices_horario(vertices_array)
            
            if vertices_ordenados is None:
                continue
            
            area = cv.contourArea(vertices_ordenados.astype(np.int32))
            if area < 100:
                continue
            
            poly_contour = vertices_ordenados.astype(np.int32).reshape(-1, 1, 2)
            if not cv.isContourConvex(poly_contour):
                continue
            
            num_puntos = contar_puntos_silla_en_poligono(poly_contour, saddle_points)
            
            if num_puntos >= MIN_PUNTOS_SILLA:
                ratio = num_puntos / np.sqrt(area) if area > 0 else 0
                cuadrilateros.append({
                    'vertices': vertices_ordenados,
                    'num_puntos': num_puntos,
                    'area': area,
                    'ratio': ratio,
                    'h1': h1,
                    'h2': h2,
                    'v1': v1,
                    'v2': v2
                })
    
    cuadrilateros.sort(key=lambda x: x['ratio'], reverse=True)
    return cuadrilateros

def dibujar_intersecciones_y_cuadrilateros(frame, rectas_horizontal, rectas_vertical, saddle_points, cuadrilateros):
    img = frame.copy()
    h, w = img.shape[:2]
    
    for punto in saddle_points:
        cv.circle(img, (int(punto[0]), int(punto[1])), 2, (0, 255, 255), -1)
    
    for recta in rectas_horizontal:
        m, b = recta['m'], recta['b']
        x1 = 0
        y1 = int(m * x1 + b)
        x2 = w
        y2 = int(m * x2 + b)
        cv.line(img, (x1, y1), (x2, y2), (255, 150, 0), 1)
    
    for recta in rectas_vertical:
        m, b = recta['m'], recta['b']
        x1 = 0
        y1 = int(m * x1 + b)
        x2 = w
        y2 = int(m * x2 + b)
        cv.line(img, (x1, y1), (x2, y2), (0, 200, 100), 1)
    
    for h_rect in rectas_horizontal:
        h_m, h_b = h_rect['m'], h_rect['b']
        for v_rect in rectas_vertical:
            v_m, v_b = v_rect['m'], v_rect['b']
            inter = calcular_interseccion(h_m, h_b, v_m, v_b)
            if inter is not None:
                x, y = inter
                if 0 <= x < w and 0 <= y < h:
                    cv.circle(img, (x, y), 4, (0, 255, 0), -1)
    
    y_offset = 30
    cv.putText(img, f"Cuadrilateros validos (min {MIN_PUNTOS_SILLA} pts): {len(cuadrilateros)}", 
              (10, y_offset), cv.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
    y_offset += 25
    
    colores = [
        (255, 0, 0), (0, 255, 0), (0, 0, 255), (255, 255, 0),
        (255, 0, 255), (0, 255, 255), (128, 0, 255), (255, 128, 0)
    ]
    
    for idx, cuad in enumerate(cuadrilateros[:8]):
        vertices = cuad['vertices']
        color = colores[idx % len(colores)]
        pts = vertices.astype(np.int32)
        cv.polylines(img, [pts], True, color, 2)
        
        centro = np.mean(vertices, axis=0).astype(int)
        cv.putText(img, f"{cuad['ratio']:.4f}", (centro[0]-20, centro[1]), 
                  cv.FONT_HERSHEY_SIMPLEX, 0.4, color, 1)
        
        if idx < 6:
            texto = f"#{idx+1}: {cuad['num_puntos']}pts/{cuad['area']:.0f}px = {cuad['ratio']:.4f}"
            cv.putText(img, texto, (10, y_offset), 
                      cv.FONT_HERSHEY_SIMPLEX, 0.4, color, 1)
            y_offset += 18
    
    if len(cuadrilateros) > 0:
        mejor = cuadrilateros[0]
        mejor_vertices = mejor['vertices']
        for v in mejor_vertices:
            cv.circle(img, (int(v[0]), int(v[1])), 8, (0, 255, 255), -1)
            cv.circle(img, (int(v[0]), int(v[1])), 10, (255, 255, 255), 1)
        
        cv.putText(img, f"MEJOR: {mejor['num_puntos']}pts / {mejor['area']:.0f}px = {mejor['ratio']:.4f}", 
                  (10, y_offset + 10), cv.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255), 2)
    
    cv.imshow(intersecciones_window, img)
    return img

def dibujar_rectas_ransac(frame, rectas_horizontal, rectas_vertical, imagen_binaria):
    img = frame.copy()
    h, w = img.shape[:2]
    
    puntos = obtener_puntos_de_imagen_binaria(imagen_binaria)
    puntos_muestra = random.sample(puntos, min(len(puntos), 500)) if len(puntos) > 500 else puntos
    
    for punto in puntos_muestra:
        cv.circle(img, (int(punto[0]), int(punto[1])), 2, (0, 255, 0), -1)
    
    colores_h = [(255, 150, 0), (255, 100, 0), (200, 80, 0), (150, 50, 0)]
    colores_v = [(0, 200, 100), (0, 150, 80), (0, 100, 60), (0, 80, 50)]
    
    for i, recta in enumerate(rectas_horizontal):
        m, b = recta['m'], recta['b']
        num_inliers = recta['num_inliers']
        x1 = 0
        y1 = int(m * x1 + b)
        x2 = w
        y2 = int(m * x2 + b)
        color = colores_h[i % len(colores_h)]
        cv.line(img, (x1, y1), (x2, y2), color, 2)
        cv.putText(img, f"H{i+1}: {num_inliers}", (20, 30 + i * 25), 
                  cv.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)
    
    for i, recta in enumerate(rectas_vertical):
        m, b = recta['m'], recta['b']
        num_inliers = recta['num_inliers']
        x1 = 0
        y1 = int(m * x1 + b)
        x2 = w
        y2 = int(m * x2 + b)
        color = colores_v[i % len(colores_v)]
        cv.line(img, (x1, y1), (x2, y2), color, 2)
        cv.putText(img, f"V{i+1}: {num_inliers}", (200, 30 + i * 25), 
                  cv.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)
    
    cv.putText(img, f"Grupo Horizontal: {len(rectas_horizontal)}, Grupo Vertical: {len(rectas_vertical)}", 
              (10, h - 20), cv.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
    
    cv.imshow(ransac_window, img)
    return img

def contar_puntos_silla_en_poligono(polygon, saddle_points):
    if polygon is None or len(saddle_points) == 0:
        return 0
    
    count = 0
    for point in saddle_points:
        distance = cv.pointPolygonTest(polygon, point, True)
        if distance >= 0:
            count += 1
    
    return count

def encontrar_cuadrilatero_con_mas_puntos_silla(cuadrilateros, saddle_points):
    if len(cuadrilateros) == 0:
        return None, 0
    
    mejor_cuadrilatero = None
    max_puntos = 0
    
    for quad in cuadrilateros:
        num_puntos = contar_puntos_silla_en_poligono(quad, saddle_points)
        if num_puntos > max_puntos and num_puntos >= MIN_PUNTOS_SILLA:
            max_puntos = num_puntos
            mejor_cuadrilatero = quad
    
    return mejor_cuadrilatero, max_puntos

def ordenar_puntos_para_warp(puntos):
    center = np.mean(puntos, axis=0)
    angles = np.arctan2(puntos[:, 1] - center[1], puntos[:, 0] - center[0])
    sorted_indices = np.argsort(angles)
    puntos_ordenados = puntos[sorted_indices]
    
    puntos_ordenados = puntos_ordenados[::-1]
    
    min_sum_idx = np.argmin(puntos_ordenados[:, 0] + puntos_ordenados[:, 1])
    puntos_ordenados = np.roll(puntos_ordenados, -min_sum_idx, axis=0)
    
    return puntos_ordenados

def dibujar_cuadrilatero_y_warp(frame, src_points_ordenados, num_puntos, area, offset=0, es_ransac=False, es_reciclado=False):
    estimated_board_img = frame.copy()
    warp_result_img = np.zeros((800, 800, 3), dtype=np.uint8)
    
    centro = np.mean(src_points_ordenados, axis=0)
    
    if offset > 0:
        puntos_con_offset = aplicar_offset_a_puntos(src_points_ordenados, offset, centro)
    else:
        puntos_con_offset = src_points_ordenados
    
    pts = puntos_con_offset.astype(np.int32)
    cv.polylines(estimated_board_img, [pts], True, (0, 255, 0), 4)
    for point in pts:
        cv.circle(estimated_board_img, tuple(point), 10, (0, 255, 255), -1)
    
    if es_reciclado:
        metodo = "RECICLADO"
        color_texto = (0, 255, 255)
    elif es_ransac:
        metodo = "RANSAC"
        color_texto = (0, 255, 255)
    else:
        metodo = "Detección"
        color_texto = (255, 255, 0)
    
    cv.putText(estimated_board_img, f"Metodo: {metodo}", 
              (10, 30), cv.FONT_HERSHEY_SIMPLEX, 0.6, color_texto, 2)
    cv.putText(estimated_board_img, f"Puntos silla: {num_puntos}", 
              (10, 60), cv.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
    cv.putText(estimated_board_img, f"Area: {int(area)}", 
              (10, 90), cv.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
    cv.putText(estimated_board_img, f"Offset: {offset}px", 
              (10, 120), cv.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)
    
    if es_reciclado:
        cv.putText(estimated_board_img, "RECICLADO (tolerancia)", 
                  (10, 150), cv.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)
    
    warped = apply_warp(frame, puntos_con_offset)
    warp_result_img = warped
    
    if es_reciclado:
        cv.putText(warp_result_img, "RECICLADO", 
                  (10, 30), cv.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)
    
    cv.putText(warp_result_img, f"Puntos silla: {num_puntos}", 
              (10, 60), cv.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
    cv.putText(warp_result_img, f"Area: {int(area)}", 
              (10, 90), cv.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
    cv.putText(warp_result_img, f"Offset: {offset}px", 
              (10, 120), cv.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)
    
    h, w = warp_result_img.shape[:2]
    cell_h = h // 8
    cell_w = w // 8
    for i in range(9):
        y = i * cell_h
        cv.line(warp_result_img, (0, y), (w, y), (0, 255, 255), 1)
        x = i * cell_w
        cv.line(warp_result_img, (x, 0), (x, h), (0, 255, 255), 1)
    
    return estimated_board_img, warp_result_img

def mostrar_mensaje_ransac(frame, mensaje, color=(0, 0, 255)):
    img = frame.copy()
    h, w = img.shape[:2]
    cv.putText(img, mensaje, (w//2 - 150, h//2), 
              cv.FONT_HERSHEY_SIMPLEX, 1, color, 2)
    cv.imshow(ransac_window, img)
    return img

while True:
    if current_frame != last_frame:
        cap.set(cv.CAP_PROP_POS_FRAMES, current_frame)
        ret, frame = cap.read()

        if ret:
            frame_vis = frame.copy()
            gray = cv.cvtColor(frame, cv.COLOR_BGR2GRAY)

            # ========== PASO 1: SOBEL Y CANNY ==========
            sobelx = cv.Sobel(gray, cv.CV_64F, 1, 0, ksize=3)
            sobely = cv.Sobel(gray, cv.CV_64F, 0, 1, ksize=3)
            sobel_magnitude = np.sqrt(sobelx**2 + sobely**2)
            sobel_magnitude = np.uint8(np.clip(sobel_magnitude, 0, 255))

            cv.imshow(sobel_window, sobel_magnitude)

            edges = cv.Canny(sobel_magnitude, 7000, 7050, apertureSize=5)
            cv.imshow(canny_window, edges)

            # ========== PASO 2: DILATACIÓN ==========
            kernel = np.ones((3,3), np.uint8)
            edges_dilated = cv.dilate(edges, kernel, iterations=2)
            cv.imshow(canny_dilated_window, edges_dilated)

            # ========== PASO 3: BÚSQUEDA DE CUADRILÁTERO + PUNTOS SILLA ==========
            Ixx = cv.Sobel(gray, cv.CV_32F, 2, 0, ksize=3)
            Iyy = cv.Sobel(gray, cv.CV_32F, 0, 2, ksize=3)
            Ixy = cv.Sobel(gray, cv.CV_32F, 1, 1, ksize=3)

            response = -(Ixx*Iyy - Ixy*Ixy)
            response = cv.GaussianBlur(response, (15,15), 0)

            mx = cv.dilate(response, np.ones((7,7), np.uint8))
            th = 0.15 * response.max()

            pts = np.where((response == mx) & (response > th))
            points = np.column_stack((pts[1], pts[0]))
            saddle_points = [(int(p[0]), int(p[1])) for p in points]

            saddle_points_img = frame.copy()
            if len(points) > 0:
                for point in points:
                    cv.circle(saddle_points_img, (int(point[0]), int(point[1])), 4, (0, 255, 0), -1)
            cv.imshow(hough_filtered_window, saddle_points_img)

            contours, hierarchy = cv.findContours(edges_dilated, cv.RETR_EXTERNAL, cv.CHAIN_APPROX_SIMPLE)

            contour_img = frame.copy()
            cv.drawContours(contour_img, contours, -1, (0, 255, 0), 2)
            cv.imshow(contour_window, contour_img)

            # ========== CREAR IMAGEN BINARIA ==========
            polygons_binary = np.zeros_like(gray)
            all_polygons = []
            contour_approx_img = frame.copy()
            saddle_polygons = []
            
            for contour in contours:
                epsilon = 0.01 * cv.arcLength(contour, True)
                approx = cv.approxPolyDP(contour, epsilon, True)
                all_polygons.append(approx)
                cv.drawContours(contour_approx_img, [approx], -1, (255, 0, 0), 3)
                
                cv.drawContours(polygons_binary, [approx], -1, 255, 1)
                
                if len(approx) == 4:
                    contains_saddle = False
                    for point in saddle_points:
                        distance = cv.pointPolygonTest(approx, point, True)
                        if distance >= -TOLERANCE:
                            contains_saddle = True
                            break
                    if contains_saddle:
                        saddle_polygons.append(approx)

            cv.imshow(approx_window, contour_approx_img)
            cv.imshow(polygons_binary_window, polygons_binary)

            quadrilaterals_img = frame.copy()
            quadrilaterals = []
            
            for polygon in saddle_polygons:
                if len(polygon) == 4:
                    area = cv.contourArea(polygon)
                    if cv.isContourConvex(polygon) and area >= MIN_AREA:
                        quadrilaterals.append(polygon)
                        cv.drawContours(quadrilaterals_img, [polygon], -1, (0, 0, 255), 3)
                        for point in polygon:
                            cv.circle(quadrilaterals_img, tuple(point[0]), 6, (0, 255, 255), -1)
                        cv.putText(quadrilaterals_img, f"Area: {int(area)}", 
                                  (int(polygon[0][0][0]), int(polygon[0][0][1]) - 20), 
                                  cv.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2)
            
            cv.imshow(quadrilaterals_window, quadrilaterals_img)

            # ========== PASO 4: SELECCIONAR CUADRILÁTERO ==========
            warp_result_img = np.zeros((800, 800, 3), dtype=np.uint8)
            estimated_board_img = frame.copy()
            
            offset_actual = cv.getTrackbarPos("Offset", controls_window)
            tolerancia_px = cv.getTrackbarPos("Tolerancia (px)", controls_window)
            radio_busqueda_actual = cv.getTrackbarPos("Radio busqueda", controls_window)
            min_inliers_actual = cv.getTrackbarPos("Min Inliers", controls_window)
            num_rectas_actual = cv.getTrackbarPos("Num Rectas", controls_window)
            recycle_threshold = cv.getTrackbarPos("Tolerancia Reciclaje %", controls_window) / 100.0
            
            # Variables de estado
            encontrado_valido = False
            usando_ransac = False
            usando_reciclado = False
            
            # ===== PASO 1: BUSCAR CUADRILÁTERO CON PUNTOS DE SILLA =====
            if len(quadrilaterals) > 0 and len(saddle_points) > 0:
                mejor_quad, num_puntos = encontrar_cuadrilatero_con_mas_puntos_silla(quadrilaterals, saddle_points)
                
                if mejor_quad is not None:
                    encontrado_valido = True
                    print(f"✅ CUADRILÁTERO DETECTADO: {num_puntos} puntos silla")
                    
                    # Guardar como último válido
                    ultimo_cuadrilatero_valido = mejor_quad
                    vertices = get_vertices_as_points(mejor_quad)
                    src_points = np.array(vertices, dtype=np.float32)
                    src_points_ordenados = ordenar_puntos_para_warp(src_points)
                    
                    ultimos_puntos_ordenados = src_points_ordenados
                    ultimo_frame_warp = frame.copy()
                    area = cv.contourArea(mejor_quad)
                    ultimo_num_puntos = num_puntos
                    ultimo_area = area
                    
                    estimated_board_img, warp_result_img = dibujar_cuadrilatero_y_warp(
                        frame, src_points_ordenados, num_puntos, area, offset_actual, False, False
                    )
                    
                    ultimo_warp_valido = warp_result_img.copy()
                    
                    cv.putText(estimated_board_img, "DETECCION TRADICIONAL", 
                              (10, 150), cv.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)
                    cv.putText(warp_result_img, "DETECCION TRADICIONAL", 
                              (10, 150), cv.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)
                    
                    cv.imshow(ransac_window, np.zeros_like(frame))
                    cv.imshow(intersecciones_window, np.zeros_like(frame))
            
            # ===== PASO 2: SI NO HAY CUADRILÁTERO, USAR RANSAC =====
            if not encontrado_valido:
                if polygons_binary is not None and np.sum(polygons_binary) > 100:
                    
                    # Si RANSAC no se ha ejecutado nunca, ejecutarlo
                    if not ransac_ejecutado:
                        print(f"🔄 PRIMERA EJECUCIÓN RANSAC ({RANSAC_ITERACIONES} iteraciones)")
                        usando_ransac = True
                        
                        rectas_horizontal, rectas_vertical = encontrar_mejores_rectas_ransac(
                            polygons_binary, tolerancia_px, radio_busqueda_actual, 
                            RANSAC_ITERACIONES, min_inliers_actual, num_rectas_actual
                        )
                        
                        dibujar_rectas_ransac(frame, rectas_horizontal, rectas_vertical, polygons_binary)
                        
                        if len(rectas_horizontal) >= 2 and len(rectas_vertical) >= 2:
                            cuadrilateros_ransac = formar_cuadrilateros_desde_rectas(
                                rectas_horizontal, rectas_vertical, saddle_points, frame.shape
                            )
                        else:
                            cuadrilateros_ransac = []
                        
                        dibujar_intersecciones_y_cuadrilateros(
                            frame, rectas_horizontal, rectas_vertical, saddle_points, cuadrilateros_ransac
                        )
                        
                        if len(cuadrilateros_ransac) > 0:
                            mejor = cuadrilateros_ransac[0]
                            mejor_vertices = mejor['vertices']
                            num_puntos = mejor['num_puntos']
                            area = mejor['area']
                            
                            # Guardar como último válido
                            src_points = np.array(mejor_vertices, dtype=np.float32)
                            src_points_ordenados = ordenar_puntos_para_warp(src_points)
                            
                            ultimos_puntos_ordenados = src_points_ordenados
                            ultimo_frame_warp = frame.copy()
                            ultimo_cuadrilatero_valido = mejor_vertices
                            ultimo_num_puntos = num_puntos
                            ultimo_area = area
                            ransac_ejecutado = True
                            
                            estimated_board_img, warp_result_img = dibujar_cuadrilatero_y_warp(
                                frame, src_points_ordenados, num_puntos, area, offset_actual, True, False
                            )
                            
                            ultimo_warp_valido = warp_result_img.copy()
                            
                            cv.putText(estimated_board_img, "RANSAC (1ra vez)", 
                                      (10, 150), cv.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)
                            cv.putText(warp_result_img, "RANSAC (1ra vez)", 
                                      (10, 150), cv.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)
                            
                            print(f"✅ RANSAC exitoso: {num_puntos} puntos, área={area:.0f}")
                        else:
                            # RANSAC no encontró nada, reciclar si existe
                            if ultimo_cuadrilatero_valido is not None:
                                usando_reciclado = True
                                print(f"♻️ RECICLANDO (RANSAC sin resultados)")
                                src_points_ordenados = ultimos_puntos_ordenados
                                num_puntos = ultimo_num_puntos
                                area = ultimo_area
                                
                                estimated_board_img, warp_result_img = dibujar_cuadrilatero_y_warp(
                                    frame, src_points_ordenados, num_puntos, area, offset_actual, False, True
                                )
                                mostrar_mensaje_ransac(frame, "RECICLADO (sin RANSAC)", (0, 255, 255))
                            else:
                                mostrar_mensaje_ransac(frame, "No se encontraron cuadriláteros", (0, 0, 255))
                                cv.putText(estimated_board_img, "SIN DETECCION", 
                                          (10, 30), cv.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
                                cv.putText(warp_result_img, "SIN DETECCION", 
                                          (10, 30), cv.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
                    
                    # Si RANSAC ya se ejecutó, reciclar con tolerancia
                    else:
                        # Contar puntos dentro del último cuadrilátero
                        poly_contour = ultimo_cuadrilatero_valido.astype(np.int32).reshape(-1, 1, 2)
                        puntos_actuales = contar_puntos_silla_en_poligono(poly_contour, saddle_points)
                        
                        if ultimo_num_puntos > 0:
                            perdida = (ultimo_num_puntos - puntos_actuales) / ultimo_num_puntos
                            
                            # Si la pérdida es menor al umbral, reciclar
                            if perdida < recycle_threshold:
                                usando_reciclado = True
                                print(f"♻️ RECICLANDO: {puntos_actuales} pts (pérdida {perdida*100:.1f}% < {recycle_threshold*100:.0f}%)")
                                
                                src_points_ordenados = ultimos_puntos_ordenados
                                num_puntos = puntos_actuales
                                area = ultimo_area
                                
                                estimated_board_img, warp_result_img = dibujar_cuadrilatero_y_warp(
                                    frame, src_points_ordenados, num_puntos, area, offset_actual, False, True
                                )
                                
                                # Actualizar el número de puntos para el siguiente frame
                                ultimo_num_puntos = puntos_actuales
                                
                                cv.imshow(ransac_window, np.zeros_like(frame))
                                cv.imshow(intersecciones_window, np.zeros_like(frame))
                            else:
                                # Pérdida grande, ejecutar RANSAC de nuevo
                                print(f"🔄 PÉRDIDA GRANDE: {puntos_actuales} pts (pérdida {perdida*100:.1f}% >= {recycle_threshold*100:.0f}%)")
                                ransac_ejecutado = False  # Forzar nueva ejecución
                                
                                # Ejecutar RANSAC nuevamente (se ejecutará en la siguiente iteración)
                                # Para este frame, reciclamos con pérdida grande
                                usando_reciclado = True
                                src_points_ordenados = ultimos_puntos_ordenados
                                num_puntos = puntos_actuales
                                area = ultimo_area
                                
                                estimated_board_img, warp_result_img = dibujar_cuadrilatero_y_warp(
                                    frame, src_points_ordenados, num_puntos, area, offset_actual, False, True
                                )
                                
                                cv.putText(estimated_board_img, "RECICLADO (forzando RANSAC)", 
                                          (10, 150), cv.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2)
                                cv.putText(warp_result_img, "RECICLADO (forzando RANSAC)", 
                                          (10, 150), cv.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2)
                else:
                    # Sin máscara binaria, reciclar si es posible
                    if ultimo_cuadrilatero_valido is not None:
                        usando_reciclado = True
                        print(f"♻️ RECICLANDO (sin máscara binaria)")
                        src_points_ordenados = ultimos_puntos_ordenados
                        num_puntos = ultimo_num_puntos
                        area = ultimo_area
                        
                        estimated_board_img, warp_result_img = dibujar_cuadrilatero_y_warp(
                            frame, src_points_ordenados, num_puntos, area, offset_actual, False, True
                        )
                        mostrar_mensaje_ransac(frame, "RECICLADO (sin máscara)", (0, 255, 255))
                    else:
                        cv.putText(estimated_board_img, "SIN DETECCION", 
                                  (10, 30), cv.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
                        cv.putText(warp_result_img, "SIN DETECCION", 
                                  (10, 30), cv.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
                        mostrar_mensaje_ransac(frame, "SIN DETECCION", (0, 0, 255))
            
            cv.imshow(estimated_board_window, estimated_board_img)
            cv.imshow(warp_window, warp_result_img)
            cv.imshow(window_name, frame_vis)

            last_frame = current_frame

    key = cv.waitKey(20) & 0xFF

    if key == 27:
        break

cap.release()
cv.destroyAllWindows()

Frames totales: 430
FPS: 29.97
✅ CUADRILÁTERO DETECTADO: 168 puntos silla
✅ CUADRILÁTERO DETECTADO: 173 puntos silla
✅ CUADRILÁTERO DETECTADO: 171 puntos silla
🔄 PRIMERA EJECUCIÓN RANSAC (50 iteraciones)
✅ RANSAC exitoso: 153 puntos, área=92906
🔄 PÉRDIDA GRANDE: 151 pts (pérdida 1.3% >= 1%)


In [ ]:
# PRUEBA CON WEBCAM CON RANSAC

import cv2 as cv
import numpy as np
from collections import deque
import random
from copy import copy
from numpy.random import default_rng
import itertools

rng = default_rng()

# Abrir la webcam
cap = cv.VideoCapture(0)  # 0 es la cámara por defecto

if not cap.isOpened():
    print("Error: No se pudo abrir la cámara")
    exit()

# Obtener información de la cámara
fps = cap.get(cv.CAP_PROP_FPS)
if fps <= 0:
    fps = 30

print(f"Cámara abierta - FPS estimado: {fps:.2f}")

# Crear ventanas
window_name = "Video"
sobel_window = "Sobel"
canny_window = "Canny"
canny_dilated_window = "Canny Dilatado"
hough_filtered_window = "Saddle Points"
contour_window = "Contornos"
approx_window = "Polígonos aproximados"
quadrilaterals_window = "Cuadriláteros detectados"
estimated_board_window = "Cuadrilátero con más puntos silla"
warp_window = "Warp final"
controls_window = "Controles"
ransac_window = "RANSAC - Rectas ajustadas"
polygons_binary_window = "Máscara binaria de polígonos"
intersecciones_window = "Intersecciones - Puntos y Cuadriláteros"

cv.namedWindow(window_name, cv.WINDOW_NORMAL)
cv.namedWindow(sobel_window, cv.WINDOW_NORMAL)
cv.namedWindow(canny_window, cv.WINDOW_NORMAL)
cv.namedWindow(canny_dilated_window, cv.WINDOW_NORMAL)
cv.namedWindow(hough_filtered_window, cv.WINDOW_NORMAL)
cv.namedWindow(contour_window, cv.WINDOW_NORMAL)
cv.namedWindow(approx_window, cv.WINDOW_NORMAL)
cv.namedWindow(quadrilaterals_window, cv.WINDOW_NORMAL)
cv.namedWindow(estimated_board_window, cv.WINDOW_NORMAL)
cv.namedWindow(warp_window, cv.WINDOW_NORMAL)
cv.namedWindow(controls_window, cv.WINDOW_NORMAL)
cv.namedWindow(ransac_window, cv.WINDOW_NORMAL)
cv.namedWindow(polygons_binary_window, cv.WINDOW_NORMAL)
cv.namedWindow(intersecciones_window, cv.WINDOW_NORMAL)

# Variable para el offset
offset_pixels = 0

def on_offset_trackbar(pos):
    global offset_pixels
    offset_pixels = pos

# Crear trackbar para el offset (0-50 píxeles)
cv.createTrackbar(
    "Offset",
    controls_window,
    15,
    50,
    on_offset_trackbar
)

# Variables para RANSAC
RANSAC_TOLERANCIA_PX = 2
RANSAC_ITERACIONES = 50  # Solo 50 iteraciones para RANSAC
RANSAC_RADIO_BUSQUEDA = 5
RANSAC_MIN_INLIERS = 20
RANSAC_NUM_RECTAS = 6

# Porcentaje de tolerancia para reciclaje (10%)
RECYCLE_THRESHOLD_PERCENT = 50

def on_tol_px_trackbar(pos):
    global RANSAC_TOLERANCIA_PX
    RANSAC_TOLERANCIA_PX = max(1, pos)

cv.createTrackbar(
    "Tolerancia (px)",
    controls_window,
    5,
    50,
    on_tol_px_trackbar
)

def on_radio_busqueda_trackbar(pos):
    global RANSAC_RADIO_BUSQUEDA
    RANSAC_RADIO_BUSQUEDA = max(1, pos)

cv.createTrackbar(
    "Radio busqueda",
    controls_window,
    1,
    200,
    on_radio_busqueda_trackbar
)

def on_min_inliers_trackbar(pos):
    global RANSAC_MIN_INLIERS
    RANSAC_MIN_INLIERS = max(1, pos)

cv.createTrackbar(
    "Min Inliers",
    controls_window,
    1,
    50,
    on_min_inliers_trackbar
)

def on_num_rectas_trackbar(pos):
    global RANSAC_NUM_RECTAS
    RANSAC_NUM_RECTAS = max(4, pos)

cv.createTrackbar(
    "Num Rectas",
    controls_window,
    4,
    12,
    on_num_rectas_trackbar
)

def on_recycle_threshold_trackbar(pos):
    global RECYCLE_THRESHOLD_PERCENT
    RECYCLE_THRESHOLD_PERCENT = max(1, pos)

cv.createTrackbar(
    "Tolerancia Reciclaje %",
    controls_window,
    50,
    100,
    on_recycle_threshold_trackbar
)

# Parámetros
TOLERANCE = 5
MIN_AREA = 10
MIN_PUNTOS_SILLA = 50

# Variables para almacenar el último cuadrilátero válido
ultimo_cuadrilatero_valido = None
ultimo_warp_valido = None
ultimo_frame_warp = None
ultimos_puntos_ordenados = None
ultimo_num_puntos = 0
ultimo_area = 0

# Flag para saber si ya se ejecutó RANSAC
ransac_ejecutado = False

# ========== FUNCIONES ==========
def get_vertices_as_points(polygon):
    vertices = []
    for point in polygon:
        vertices.append((int(point[0][0]), int(point[0][1])))
    return vertices

def apply_warp(frame, src_points, dst_size=(800, 800)):
    dst_points = np.array([
        [0, 0],
        [dst_size[0]-1, 0],
        [dst_size[0]-1, dst_size[1]-1],
        [0, dst_size[1]-1]
    ], dtype=np.float32)
    
    src_points = np.array(src_points, dtype=np.float32)
    M = cv.getPerspectiveTransform(src_points, dst_points)
    warped = cv.warpPerspective(frame, M, dst_size)
    return warped

def aplicar_offset_a_puntos(puntos, offset, centro):
    if offset == 0:
        return puntos
    
    puntos_con_offset = []
    for punto in puntos:
        vector = centro - punto
        norm = np.linalg.norm(vector)
        if norm > 0:
            nuevo_punto = punto + (vector / norm) * offset
        else:
            nuevo_punto = punto
        puntos_con_offset.append(nuevo_punto)
    
    return np.array(puntos_con_offset, dtype=np.float32)

def obtener_puntos_de_imagen_binaria(imagen_binaria):
    pts = np.argwhere(imagen_binaria > 0)
    puntos = []
    for p in pts:
        y, x = p
        puntos.append((x, y))
    return puntos

def encontrar_punto_cercano(puntos, punto_origen, radio_busqueda):
    puntos_cercanos = []
    for p in puntos:
        dist = np.sqrt((p[0] - punto_origen[0])**2 + (p[1] - punto_origen[1])**2)
        if dist <= radio_busqueda and dist > 0:
            puntos_cercanos.append(p)
    
    if len(puntos_cercanos) == 0:
        return None
    
    return random.choice(puntos_cercanos)

def ajustar_recta_ransac_simple(puntos, tolerancia_px, radio_busqueda, iteraciones):
    if len(puntos) < 2:
        return None, [], 0
    
    mejor_recta = None
    mejores_inliers = []
    max_inliers = 0
    
    puntos_array = np.array(puntos)
    
    for _ in range(iteraciones):
        if len(puntos_array) < 2:
            break
        
        idx1 = random.randint(0, len(puntos_array) - 1)
        p1 = puntos_array[idx1]
        
        p2 = encontrar_punto_cercano(puntos_array.tolist(), p1, radio_busqueda)
        if p2 is None:
            continue
        
        x1, y1 = p1[0], p1[1]
        x2, y2 = p2[0], p2[1]
        
        if abs(x2 - x1) < 1e-6:
            continue
        
        m = (y2 - y1) / (x2 - x1)
        b = y1 - m * x1
        recta = (m, b)
        
        inliers = []
        for punto in puntos_array:
            x, y_p = punto[0], punto[1]
            y_pred = m * x + b
            dist = abs(y_p - y_pred) / np.sqrt(m**2 + 1)
            if dist <= tolerancia_px:
                inliers.append(punto)
        
        if len(inliers) > max_inliers:
            max_inliers = len(inliers)
            mejores_inliers = inliers
            mejor_recta = recta
    
    if mejor_recta is not None and len(mejores_inliers) >= 2:
        xs = [p[0] for p in mejores_inliers]
        ys = [p[1] for p in mejores_inliers]
        x = np.array(xs)
        y = np.array(ys)
        A = np.vstack([x, np.ones(len(x))]).T
        m, b = np.linalg.lstsq(A, y, rcond=None)[0]
        mejor_recta = (m, b)
        
        inliers_finales = []
        m, b = mejor_recta
        for punto in puntos:
            x, y_p = punto[0], punto[1]
            y_pred = m * x + b
            dist = abs(y_p - y_pred) / np.sqrt(m**2 + 1)
            if dist <= tolerancia_px:
                inliers_finales.append(punto)
        return mejor_recta, inliers_finales, len(inliers_finales)
    
    return mejor_recta, mejores_inliers, max_inliers

def encontrar_mejores_rectas_ransac(imagen_binaria, tolerancia_px, radio_busqueda, iteraciones, min_inliers, num_rectas=8):
    if imagen_binaria is None:
        return None, None
    
    img_temp = imagen_binaria.copy()
    rectas_horizontal = []
    
    for i in range(num_rectas // 2 + 2):
        puntos = obtener_puntos_de_imagen_binaria(img_temp)
        if len(puntos) < min_inliers:
            break
        
        recta, inliers, num_inliers = ajustar_recta_ransac_simple(
            puntos, tolerancia_px, radio_busqueda, iteraciones
        )
        
        if recta is None or num_inliers < min_inliers:
            break
        
        m, b = recta
        rectas_horizontal.append({
            'm': m,
            'b': b,
            'num_inliers': num_inliers,
            'inliers': inliers,
            'origen': 'horizontal'
        })
        
        for punto in inliers:
            x, y = int(punto[0]), int(punto[1])
            if 0 <= y < img_temp.shape[0] and 0 <= x < img_temp.shape[1]:
                img_temp[y, x] = 0
    
    imagen_transpuesta = imagen_binaria.T
    img_temp_vert = imagen_transpuesta.copy()
    rectas_vertical = []
    
    for i in range(num_rectas // 2 + 2):
        puntos = obtener_puntos_de_imagen_binaria(img_temp_vert)
        if len(puntos) < min_inliers:
            break
        
        recta, inliers, num_inliers = ajustar_recta_ransac_simple(
            puntos, tolerancia_px, radio_busqueda, iteraciones
        )
        
        if recta is None or num_inliers < min_inliers:
            break
        
        m_orig, b_orig = recta
        if abs(m_orig) > 1e-6:
            m_conv = 1.0 / m_orig
            b_conv = -b_orig / m_orig
        else:
            m_conv = 0
            b_conv = b_orig
        
        rectas_vertical.append({
            'm': m_conv,
            'b': b_conv,
            'num_inliers': num_inliers,
            'inliers': inliers,
            'origen': 'vertical'
        })
        
        for punto in inliers:
            x, y = int(punto[0]), int(punto[1])
            if 0 <= y < img_temp_vert.shape[0] and 0 <= x < img_temp_vert.shape[1]:
                img_temp_vert[y, x] = 0
    
    return rectas_horizontal, rectas_vertical

def calcular_interseccion(m1, b1, m2, b2):
    if abs(m1 - m2) < 1e-6:
        return None
    x = (b2 - b1) / (m1 - m2)
    y = m1 * x + b1
    return (int(x), int(y))

def ordenar_vertices_horario(vertices):
    if len(vertices) != 4:
        return None
    
    center = np.mean(vertices, axis=0)
    angles = np.arctan2(vertices[:, 1] - center[1], vertices[:, 0] - center[0])
    sorted_indices = np.argsort(angles)
    vertices_ordenados = vertices[sorted_indices]
    
    vertices_ordenados = vertices_ordenados[::-1]
    
    min_sum_idx = np.argmin(vertices_ordenados[:, 0] + vertices_ordenados[:, 1])
    vertices_ordenados = np.roll(vertices_ordenados, -min_sum_idx, axis=0)
    
    return vertices_ordenados

def formar_cuadrilateros_desde_rectas(rectas_horizontal, rectas_vertical, saddle_points, frame_shape):
    h, w = frame_shape[:2]
    cuadrilateros = []
    
    for combo_h in itertools.combinations(range(len(rectas_horizontal)), 2):
        for combo_v in itertools.combinations(range(len(rectas_vertical)), 2):
            h1 = rectas_horizontal[combo_h[0]]
            h2 = rectas_horizontal[combo_h[1]]
            v1 = rectas_vertical[combo_v[0]]
            v2 = rectas_vertical[combo_v[1]]
            
            h1_m, h1_b = h1['m'], h1['b']
            h2_m, h2_b = h2['m'], h2['b']
            v1_m, v1_b = v1['m'], v1['b']
            v2_m, v2_b = v2['m'], v2['b']
            
            inter1 = calcular_interseccion(h1_m, h1_b, v1_m, v1_b)
            inter2 = calcular_interseccion(h1_m, h1_b, v2_m, v2_b)
            inter3 = calcular_interseccion(h2_m, h2_b, v2_m, v2_b)
            inter4 = calcular_interseccion(h2_m, h2_b, v1_m, v1_b)
            
            if None in [inter1, inter2, inter3, inter4]:
                continue
            
            vertices = [inter1, inter2, inter3, inter4]
            
            dentro = True
            for v in vertices:
                if v[0] < 0 or v[0] >= w or v[1] < 0 or v[1] >= h:
                    dentro = False
                    break
            
            if not dentro:
                continue
            
            vertices_array = np.array(vertices, dtype=np.float32)
            vertices_ordenados = ordenar_vertices_horario(vertices_array)
            
            if vertices_ordenados is None:
                continue
            
            area = cv.contourArea(vertices_ordenados.astype(np.int32))
            if area < 100:
                continue
            
            poly_contour = vertices_ordenados.astype(np.int32).reshape(-1, 1, 2)
            if not cv.isContourConvex(poly_contour):
                continue
            
            num_puntos = contar_puntos_silla_en_poligono(poly_contour, saddle_points)
            
            if num_puntos >= MIN_PUNTOS_SILLA:
                ratio = num_puntos**2 / np.sqrt(area) if area > 0 else 0
                cuadrilateros.append({
                    'vertices': vertices_ordenados,
                    'num_puntos': num_puntos,
                    'area': area,
                    'ratio': ratio,
                    'h1': h1,
                    'h2': h2,
                    'v1': v1,
                    'v2': v2
                })
    
    cuadrilateros.sort(key=lambda x: x['ratio'], reverse=True)
    return cuadrilateros

def dibujar_intersecciones_y_cuadrilateros(frame, rectas_horizontal, rectas_vertical, saddle_points, cuadrilateros):
    img = frame.copy()
    h, w = img.shape[:2]
    
    for punto in saddle_points:
        cv.circle(img, (int(punto[0]), int(punto[1])), 2, (0, 255, 255), -1)
    
    for recta in rectas_horizontal:
        m, b = recta['m'], recta['b']
        x1 = 0
        y1 = int(m * x1 + b)
        x2 = w
        y2 = int(m * x2 + b)
        cv.line(img, (x1, y1), (x2, y2), (255, 150, 0), 1)
    
    for recta in rectas_vertical:
        m, b = recta['m'], recta['b']
        x1 = 0
        y1 = int(m * x1 + b)
        x2 = w
        y2 = int(m * x2 + b)
        cv.line(img, (x1, y1), (x2, y2), (0, 200, 100), 1)
    
    for h_rect in rectas_horizontal:
        h_m, h_b = h_rect['m'], h_rect['b']
        for v_rect in rectas_vertical:
            v_m, v_b = v_rect['m'], v_rect['b']
            inter = calcular_interseccion(h_m, h_b, v_m, v_b)
            if inter is not None:
                x, y = inter
                if 0 <= x < w and 0 <= y < h:
                    cv.circle(img, (x, y), 4, (0, 255, 0), -1)
    
    y_offset = 30
    cv.putText(img, f"Cuadrilateros validos (min {MIN_PUNTOS_SILLA} pts): {len(cuadrilateros)}", 
              (10, y_offset), cv.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
    y_offset += 25
    
    colores = [
        (255, 0, 0), (0, 255, 0), (0, 0, 255), (255, 255, 0),
        (255, 0, 255), (0, 255, 255), (128, 0, 255), (255, 128, 0)
    ]
    
    for idx, cuad in enumerate(cuadrilateros[:8]):
        vertices = cuad['vertices']
        color = colores[idx % len(colores)]
        pts = vertices.astype(np.int32)
        cv.polylines(img, [pts], True, color, 2)
        
        centro = np.mean(vertices, axis=0).astype(int)
        cv.putText(img, f"{cuad['ratio']:.4f}", (centro[0]-20, centro[1]), 
                  cv.FONT_HERSHEY_SIMPLEX, 0.4, color, 1)
        
        if idx < 6:
            texto = f"#{idx+1}: {cuad['num_puntos']}pts/{cuad['area']:.0f}px = {cuad['ratio']:.4f}"
            cv.putText(img, texto, (10, y_offset), 
                      cv.FONT_HERSHEY_SIMPLEX, 0.4, color, 1)
            y_offset += 18
    
    if len(cuadrilateros) > 0:
        mejor = cuadrilateros[0]
        mejor_vertices = mejor['vertices']
        for v in mejor_vertices:
            cv.circle(img, (int(v[0]), int(v[1])), 8, (0, 255, 255), -1)
            cv.circle(img, (int(v[0]), int(v[1])), 10, (255, 255, 255), 1)
        
        cv.putText(img, f"MEJOR: {mejor['num_puntos']}pts / {mejor['area']:.0f}px = {mejor['ratio']:.4f}", 
                  (10, y_offset + 10), cv.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255), 2)
    
    cv.imshow(intersecciones_window, img)
    return img

def dibujar_rectas_ransac(frame, rectas_horizontal, rectas_vertical, imagen_binaria):
    img = frame.copy()
    h, w = img.shape[:2]
    
    puntos = obtener_puntos_de_imagen_binaria(imagen_binaria)
    puntos_muestra = random.sample(puntos, min(len(puntos), 500)) if len(puntos) > 500 else puntos
    
    for punto in puntos_muestra:
        cv.circle(img, (int(punto[0]), int(punto[1])), 2, (0, 255, 0), -1)
    
    colores_h = [(255, 150, 0), (255, 100, 0), (200, 80, 0), (150, 50, 0)]
    colores_v = [(0, 200, 100), (0, 150, 80), (0, 100, 60), (0, 80, 50)]
    
    for i, recta in enumerate(rectas_horizontal):
        m, b = recta['m'], recta['b']
        num_inliers = recta['num_inliers']
        x1 = 0
        y1 = int(m * x1 + b)
        x2 = w
        y2 = int(m * x2 + b)
        color = colores_h[i % len(colores_h)]
        cv.line(img, (x1, y1), (x2, y2), color, 2)
        cv.putText(img, f"H{i+1}: {num_inliers}", (20, 30 + i * 25), 
                  cv.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)
    
    for i, recta in enumerate(rectas_vertical):
        m, b = recta['m'], recta['b']
        num_inliers = recta['num_inliers']
        x1 = 0
        y1 = int(m * x1 + b)
        x2 = w
        y2 = int(m * x2 + b)
        color = colores_v[i % len(colores_v)]
        cv.line(img, (x1, y1), (x2, y2), color, 2)
        cv.putText(img, f"V{i+1}: {num_inliers}", (200, 30 + i * 25), 
                  cv.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)
    
    cv.putText(img, f"Grupo Horizontal: {len(rectas_horizontal)}, Grupo Vertical: {len(rectas_vertical)}", 
              (10, h - 20), cv.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
    
    cv.imshow(ransac_window, img)
    return img

def contar_puntos_silla_en_poligono(polygon, saddle_points):
    if polygon is None or len(saddle_points) == 0:
        return 0
    
    count = 0
    for point in saddle_points:
        distance = cv.pointPolygonTest(polygon, point, True)
        if distance >= 0:
            count += 1
    
    return count

def encontrar_cuadrilatero_con_mas_puntos_silla(cuadrilateros, saddle_points):
    if len(cuadrilateros) == 0:
        return None, 0
    
    mejor_cuadrilatero = None
    max_puntos = 0
    
    for quad in cuadrilateros:
        num_puntos = contar_puntos_silla_en_poligono(quad, saddle_points)
        if num_puntos > max_puntos and num_puntos >= MIN_PUNTOS_SILLA:
            max_puntos = num_puntos
            mejor_cuadrilatero = quad
    
    return mejor_cuadrilatero, max_puntos

def ordenar_puntos_para_warp(puntos):
    center = np.mean(puntos, axis=0)
    angles = np.arctan2(puntos[:, 1] - center[1], puntos[:, 0] - center[0])
    sorted_indices = np.argsort(angles)
    puntos_ordenados = puntos[sorted_indices]
    
    puntos_ordenados = puntos_ordenados[::-1]
    
    min_sum_idx = np.argmin(puntos_ordenados[:, 0] + puntos_ordenados[:, 1])
    puntos_ordenados = np.roll(puntos_ordenados, -min_sum_idx, axis=0)
    
    return puntos_ordenados

def dibujar_cuadrilatero_y_warp(frame, src_points_ordenados, num_puntos, area, offset=0, es_ransac=False, es_reciclado=False):
    estimated_board_img = frame.copy()
    warp_result_img = np.zeros((800, 800, 3), dtype=np.uint8)
    
    centro = np.mean(src_points_ordenados, axis=0)
    
    if offset > 0:
        puntos_con_offset = aplicar_offset_a_puntos(src_points_ordenados, offset, centro)
    else:
        puntos_con_offset = src_points_ordenados
    
    pts = puntos_con_offset.astype(np.int32)
    cv.polylines(estimated_board_img, [pts], True, (0, 255, 0), 4)
    for point in pts:
        cv.circle(estimated_board_img, tuple(point), 10, (0, 255, 255), -1)
    
    if es_reciclado:
        metodo = "RECICLADO"
        color_texto = (0, 255, 255)
    elif es_ransac:
        metodo = "RANSAC"
        color_texto = (0, 255, 255)
    else:
        metodo = "Detección"
        color_texto = (255, 255, 0)
    
    cv.putText(estimated_board_img, f"Metodo: {metodo}", 
              (10, 30), cv.FONT_HERSHEY_SIMPLEX, 0.6, color_texto, 2)
    cv.putText(estimated_board_img, f"Puntos silla: {num_puntos}", 
              (10, 60), cv.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
    cv.putText(estimated_board_img, f"Area: {int(area)}", 
              (10, 90), cv.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
    cv.putText(estimated_board_img, f"Offset: {offset}px", 
              (10, 120), cv.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)
    
    if es_reciclado:
        cv.putText(estimated_board_img, "RECICLADO (tolerancia)", 
                  (10, 150), cv.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)
    
    warped = apply_warp(frame, puntos_con_offset)
    warp_result_img = warped
    
    if es_reciclado:
        cv.putText(warp_result_img, "RECICLADO", 
                  (10, 30), cv.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)
    
    cv.putText(warp_result_img, f"Puntos silla: {num_puntos}", 
              (10, 60), cv.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
    cv.putText(warp_result_img, f"Area: {int(area)}", 
              (10, 90), cv.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
    cv.putText(warp_result_img, f"Offset: {offset}px", 
              (10, 120), cv.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)
    
    h, w = warp_result_img.shape[:2]
    cell_h = h // 8
    cell_w = w // 8
    for i in range(9):
        y = i * cell_h
        cv.line(warp_result_img, (0, y), (w, y), (0, 255, 255), 1)
        x = i * cell_w
        cv.line(warp_result_img, (x, 0), (x, h), (0, 255, 255), 1)
    
    return estimated_board_img, warp_result_img

def mostrar_mensaje_ransac(frame, mensaje, color=(0, 0, 255)):
    img = frame.copy()
    h, w = img.shape[:2]
    cv.putText(img, mensaje, (w//2 - 150, h//2), 
              cv.FONT_HERSHEY_SIMPLEX, 1, color, 2)
    cv.imshow(ransac_window, img)
    return img

# ========== LOOP PRINCIPAL ==========
print("\n=== INSTRUCCIONES ===")
print("1. Ajusta los parámetros con los sliders en 'Controles'")
print("2. Apunta la cámara a un tablero de ajedrez")
print("3. Presiona 'ESC' para salir")
print("4. Presiona 'r' para resetear la detección")
print("=====================\n")

while True:
    ret, frame = cap.read()
    
    if not ret:
        print("Error: No se pudo leer el frame de la cámara")
        break
    
    # Redimensionar para mejor rendimiento (opcional)
    # frame = cv.resize(frame, (640, 480))
    
    frame_vis = frame.copy()
    gray = cv.cvtColor(frame, cv.COLOR_BGR2GRAY)

    # ========== PASO 1: SOBEL Y CANNY ==========
    sobelx = cv.Sobel(gray, cv.CV_64F, 1, 0, ksize=3)
    sobely = cv.Sobel(gray, cv.CV_64F, 0, 1, ksize=3)
    sobel_magnitude = np.sqrt(sobelx**2 + sobely**2)
    sobel_magnitude = np.uint8(np.clip(sobel_magnitude, 0, 255))

    cv.imshow(sobel_window, sobel_magnitude)

    edges = cv.Canny(sobel_magnitude, 7000, 7050, apertureSize=5)
    cv.imshow(canny_window, edges)

    # ========== PASO 2: DILATACIÓN ==========
    kernel = np.ones((3,3), np.uint8)
    edges_dilated = cv.dilate(edges, kernel, iterations=2)
    cv.imshow(canny_dilated_window, edges_dilated)

    # ========== PASO 3: BÚSQUEDA DE CUADRILÁTERO + PUNTOS SILLA ==========
    Ixx = cv.Sobel(gray, cv.CV_32F, 2, 0, ksize=3)
    Iyy = cv.Sobel(gray, cv.CV_32F, 0, 2, ksize=3)
    Ixy = cv.Sobel(gray, cv.CV_32F, 1, 1, ksize=3)

    response = -(Ixx*Iyy - Ixy*Ixy)
    response = cv.GaussianBlur(response, (15,15), 0)

    mx = cv.dilate(response, np.ones((7,7), np.uint8))
    th = 0.15 * response.max()

    pts = np.where((response == mx) & (response > th))
    points = np.column_stack((pts[1], pts[0]))
    saddle_points = [(int(p[0]), int(p[1])) for p in points]

    saddle_points_img = frame.copy()
    if len(points) > 0:
        for point in points:
            cv.circle(saddle_points_img, (int(point[0]), int(point[1])), 4, (0, 255, 0), -1)
    cv.imshow(hough_filtered_window, saddle_points_img)

    contours, hierarchy = cv.findContours(edges_dilated, cv.RETR_EXTERNAL, cv.CHAIN_APPROX_SIMPLE)

    contour_img = frame.copy()
    cv.drawContours(contour_img, contours, -1, (0, 255, 0), 2)
    cv.imshow(contour_window, contour_img)

    # ========== CREAR IMAGEN BINARIA ==========
    polygons_binary = np.zeros_like(gray)
    all_polygons = []
    contour_approx_img = frame.copy()
    saddle_polygons = []
    
    for contour in contours:
        epsilon = 0.01 * cv.arcLength(contour, True)
        approx = cv.approxPolyDP(contour, epsilon, True)
        all_polygons.append(approx)
        cv.drawContours(contour_approx_img, [approx], -1, (255, 0, 0), 3)
        
        cv.drawContours(polygons_binary, [approx], -1, 255, 1)
        
        if len(approx) == 4:
            contains_saddle = False
            for point in saddle_points:
                distance = cv.pointPolygonTest(approx, point, True)
                if distance >= -TOLERANCE:
                    contains_saddle = True
                    break
            if contains_saddle:
                saddle_polygons.append(approx)

    cv.imshow(approx_window, contour_approx_img)
    cv.imshow(polygons_binary_window, polygons_binary)

    quadrilaterals_img = frame.copy()
    quadrilaterals = []
    
    for polygon in saddle_polygons:
        if len(polygon) == 4:
            area = cv.contourArea(polygon)
            if cv.isContourConvex(polygon) and area >= MIN_AREA:
                quadrilaterals.append(polygon)
                cv.drawContours(quadrilaterals_img, [polygon], -1, (0, 0, 255), 3)
                for point in polygon:
                    cv.circle(quadrilaterals_img, tuple(point[0]), 6, (0, 255, 255), -1)
                cv.putText(quadrilaterals_img, f"Area: {int(area)}", 
                          (int(polygon[0][0][0]), int(polygon[0][0][1]) - 20), 
                          cv.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2)
    
    cv.imshow(quadrilaterals_window, quadrilaterals_img)

    # ========== PASO 4: SELECCIONAR CUADRILÁTERO ==========
    warp_result_img = np.zeros((800, 800, 3), dtype=np.uint8)
    estimated_board_img = frame.copy()
    
    offset_actual = cv.getTrackbarPos("Offset", controls_window)
    tolerancia_px = cv.getTrackbarPos("Tolerancia (px)", controls_window)
    radio_busqueda_actual = cv.getTrackbarPos("Radio busqueda", controls_window)
    min_inliers_actual = cv.getTrackbarPos("Min Inliers", controls_window)
    num_rectas_actual = cv.getTrackbarPos("Num Rectas", controls_window)
    recycle_threshold = cv.getTrackbarPos("Tolerancia Reciclaje %", controls_window) / 100.0
    
    # Variables de estado
    encontrado_valido = False
    usando_ransac = False
    usando_reciclado = False
    
    # ===== PASO 1: BUSCAR CUADRILÁTERO CON PUNTOS DE SILLA =====
    if len(quadrilaterals) > 0 and len(saddle_points) > 0:
        mejor_quad, num_puntos = encontrar_cuadrilatero_con_mas_puntos_silla(quadrilaterals, saddle_points)
        
        if mejor_quad is not None:
            encontrado_valido = True
            print(f"✅ CUADRILÁTERO DETECTADO: {num_puntos} puntos silla")
            
            # Guardar como último válido
            ultimo_cuadrilatero_valido = mejor_quad
            vertices = get_vertices_as_points(mejor_quad)
            src_points = np.array(vertices, dtype=np.float32)
            src_points_ordenados = ordenar_puntos_para_warp(src_points)
            
            ultimos_puntos_ordenados = src_points_ordenados
            ultimo_frame_warp = frame.copy()
            area = cv.contourArea(mejor_quad)
            ultimo_num_puntos = num_puntos
            ultimo_area = area
            
            estimated_board_img, warp_result_img = dibujar_cuadrilatero_y_warp(
                frame, src_points_ordenados, num_puntos, area, offset_actual, False, False
            )
            
            ultimo_warp_valido = warp_result_img.copy()
            
            cv.putText(estimated_board_img, "DETECCION TRADICIONAL", 
                      (10, 150), cv.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)
            cv.putText(warp_result_img, "DETECCION TRADICIONAL", 
                      (10, 150), cv.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)
            
            cv.imshow(ransac_window, np.zeros_like(frame))
            cv.imshow(intersecciones_window, np.zeros_like(frame))
    
    # ===== PASO 2: SI NO HAY CUADRILÁTERO, USAR RANSAC =====
    if not encontrado_valido:
        if polygons_binary is not None and np.sum(polygons_binary) > 100:
            
            # Si RANSAC no se ha ejecutado nunca, ejecutarlo
            if not ransac_ejecutado:
                print(f"🔄 PRIMERA EJECUCIÓN RANSAC ({RANSAC_ITERACIONES} iteraciones)")
                usando_ransac = True
                
                rectas_horizontal, rectas_vertical = encontrar_mejores_rectas_ransac(
                    polygons_binary, tolerancia_px, radio_busqueda_actual, 
                    RANSAC_ITERACIONES, min_inliers_actual, num_rectas_actual
                )
                
                dibujar_rectas_ransac(frame, rectas_horizontal, rectas_vertical, polygons_binary)
                
                if len(rectas_horizontal) >= 2 and len(rectas_vertical) >= 2:
                    cuadrilateros_ransac = formar_cuadrilateros_desde_rectas(
                        rectas_horizontal, rectas_vertical, saddle_points, frame.shape
                    )
                else:
                    cuadrilateros_ransac = []
                
                dibujar_intersecciones_y_cuadrilateros(
                    frame, rectas_horizontal, rectas_vertical, saddle_points, cuadrilateros_ransac
                )
                
                if len(cuadrilateros_ransac) > 0:
                    mejor = cuadrilateros_ransac[0]
                    mejor_vertices = mejor['vertices']
                    num_puntos = mejor['num_puntos']
                    area = mejor['area']
                    
                    # Guardar como último válido
                    src_points = np.array(mejor_vertices, dtype=np.float32)
                    src_points_ordenados = ordenar_puntos_para_warp(src_points)
                    
                    ultimos_puntos_ordenados = src_points_ordenados
                    ultimo_frame_warp = frame.copy()
                    ultimo_cuadrilatero_valido = mejor_vertices
                    ultimo_num_puntos = num_puntos
                    ultimo_area = area
                    ransac_ejecutado = True
                    
                    estimated_board_img, warp_result_img = dibujar_cuadrilatero_y_warp(
                        frame, src_points_ordenados, num_puntos, area, offset_actual, True, False
                    )
                    
                    ultimo_warp_valido = warp_result_img.copy()
                    
                    cv.putText(estimated_board_img, "RANSAC (1ra vez)", 
                              (10, 150), cv.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)
                    cv.putText(warp_result_img, "RANSAC (1ra vez)", 
                              (10, 150), cv.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)
                    
                    print(f"✅ RANSAC exitoso: {num_puntos} puntos, área={area:.0f}")
                else:
                    # RANSAC no encontró nada, reciclar si existe
                    if ultimo_cuadrilatero_valido is not None:
                        usando_reciclado = True
                        print(f"♻️ RECICLANDO (RANSAC sin resultados)")
                        src_points_ordenados = ultimos_puntos_ordenados
                        num_puntos = ultimo_num_puntos
                        area = ultimo_area
                        
                        estimated_board_img, warp_result_img = dibujar_cuadrilatero_y_warp(
                            frame, src_points_ordenados, num_puntos, area, offset_actual, False, True
                        )
                        mostrar_mensaje_ransac(frame, "RECICLADO (sin RANSAC)", (0, 255, 255))
                    else:
                        mostrar_mensaje_ransac(frame, "No se encontraron cuadriláteros", (0, 0, 255))
                        cv.putText(estimated_board_img, "SIN DETECCION", 
                                  (10, 30), cv.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
                        cv.putText(warp_result_img, "SIN DETECCION", 
                                  (10, 30), cv.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
            
            # Si RANSAC ya se ejecutó, reciclar con tolerancia
            else:
                # Contar puntos dentro del último cuadrilátero
                poly_contour = ultimo_cuadrilatero_valido.astype(np.int32).reshape(-1, 1, 2)
                puntos_actuales = contar_puntos_silla_en_poligono(poly_contour, saddle_points)
                
                if ultimo_num_puntos > 0:
                    perdida = (ultimo_num_puntos - puntos_actuales) / ultimo_num_puntos
                    
                    # Si la pérdida es menor al umbral, reciclar
                    if perdida < recycle_threshold:
                        usando_reciclado = True
                        print(f"♻️ RECICLANDO: {puntos_actuales} pts (pérdida {perdida*100:.1f}% < {recycle_threshold*100:.0f}%)")
                        
                        src_points_ordenados = ultimos_puntos_ordenados
                        num_puntos = puntos_actuales
                        area = ultimo_area
                        
                        estimated_board_img, warp_result_img = dibujar_cuadrilatero_y_warp(
                            frame, src_points_ordenados, num_puntos, area, offset_actual, False, True
                        )
                        
                        # Actualizar el número de puntos para el siguiente frame
                        ultimo_num_puntos = puntos_actuales
                        
                        cv.imshow(ransac_window, np.zeros_like(frame))
                        cv.imshow(intersecciones_window, np.zeros_like(frame))
                    else:
                        # Pérdida grande, ejecutar RANSAC de nuevo
                        print(f"🔄 PÉRDIDA GRANDE: {puntos_actuales} pts (pérdida {perdida*100:.1f}% >= {recycle_threshold*100:.0f}%)")
                        ransac_ejecutado = False  # Forzar nueva ejecución
                        
                        # Ejecutar RANSAC nuevamente (se ejecutará en la siguiente iteración)
                        # Para este frame, reciclamos con pérdida grande
                        usando_reciclado = True
                        src_points_ordenados = ultimos_puntos_ordenados
                        num_puntos = puntos_actuales
                        area = ultimo_area
                        
                        estimated_board_img, warp_result_img = dibujar_cuadrilatero_y_warp(
                            frame, src_points_ordenados, num_puntos, area, offset_actual, False, True
                        )
                        
                        cv.putText(estimated_board_img, "RECICLADO (forzando RANSAC)", 
                                  (10, 150), cv.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2)
                        cv.putText(warp_result_img, "RECICLADO (forzando RANSAC)", 
                                  (10, 150), cv.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2)
        else:
            # Sin máscara binaria, reciclar si es posible
            if ultimo_cuadrilatero_valido is not None:
                usando_reciclado = True
                print(f"♻️ RECICLANDO (sin máscara binaria)")
                src_points_ordenados = ultimos_puntos_ordenados
                num_puntos = ultimo_num_puntos
                area = ultimo_area
                
                estimated_board_img, warp_result_img = dibujar_cuadrilatero_y_warp(
                    frame, src_points_ordenados, num_puntos, area, offset_actual, False, True
                )
                mostrar_mensaje_ransac(frame, "RECICLADO (sin máscara)", (0, 255, 255))
            else:
                cv.putText(estimated_board_img, "SIN DETECCION", 
                          (10, 30), cv.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
                cv.putText(warp_result_img, "SIN DETECCION", 
                          (10, 30), cv.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
                mostrar_mensaje_ransac(frame, "SIN DETECCION", (0, 0, 255))
    
    cv.imshow(estimated_board_window, estimated_board_img)
    cv.imshow(warp_window, warp_result_img)
    cv.imshow(window_name, frame_vis)

    key = cv.waitKey(1) & 0xFF
    
    if key == 27:  # ESC para salir
        break
    elif key == ord('r'):  # Resetear detección
        ultimo_cuadrilatero_valido = None
        ultimo_warp_valido = None
        ultimo_frame_warp = None
        ultimos_puntos_ordenados = None
        ultimo_num_puntos = 0
        ultimo_area = 0
        ransac_ejecutado = False
        print("Detección reiniciada")

cap.release()
cv.destroyAllWindows()

Cámara abierta - FPS estimado: 30.00

=== INSTRUCCIONES ===
1. Ajusta los parámetros con los sliders en 'Controles'
2. Apunta la cámara a un tablero de ajedrez
3. Presiona 'ESC' para salir
4. Presiona 'r' para resetear la detección

🔄 PRIMERA EJECUCIÓN RANSAC (50 iteraciones)
🔄 PRIMERA EJECUCIÓN RANSAC (50 iteraciones)
🔄 PRIMERA EJECUCIÓN RANSAC (50 iteraciones)
🔄 PRIMERA EJECUCIÓN RANSAC (50 iteraciones)
✅ RANSAC exitoso: 50 puntos, área=40752
♻️ RECICLANDO: 48 pts (pérdida 4.0% < 50%)
♻️ RECICLANDO: 44 pts (pérdida 8.3% < 50%)
♻️ RECICLANDO: 44 pts (pérdida 0.0% < 50%)
♻️ RECICLANDO: 44 pts (pérdida 0.0% < 50%)
♻️ RECICLANDO: 44 pts (pérdida 0.0% < 50%)
♻️ RECICLANDO: 49 pts (pérdida -11.4% < 50%)
♻️ RECICLANDO: 49 pts (pérdida 0.0% < 50%)
♻️ RECICLANDO: 43 pts (pérdida 12.2% < 50%)
♻️ RECICLANDO: 43 pts (pérdida 0.0% < 50%)
♻️ RECICLANDO: 43 pts (pérdida 0.0% < 50%)
♻️ RECICLANDO: 42 pts (pérdida 2.3% < 50%)
♻️ RECICLANDO: 42 pts (pérdida 0.0% < 50%)
♻️ RECICLANDO: 47 pts (pérdida -

In [ ]:
# Prueba con video guardado SIN RANSAC
import cv2 as cv
import numpy as np
from collections import deque

# Ruta al video
video_path = "../../../data/raw/Prueba2.mp4"

# Abrir el video
cap = cv.VideoCapture(video_path)

if not cap.isOpened():
    raise IOError(f"No se pudo abrir el video: {video_path}")

# Obtener información del video
total_frames = int(cap.get(cv.CAP_PROP_FRAME_COUNT))
fps = cap.get(cv.CAP_PROP_FPS)

print(f"Frames totales: {total_frames}")
print(f"FPS: {fps:.2f}")

# Crear ventanas
window_name = "Video"
sobel_window = "Sobel"
canny_window = "Canny"
canny_dilated_window = "Canny Dilatado"
hough_filtered_window = "Saddle Points"
contour_window = "Contornos"
approx_window = "Polígonos aproximados"
quadrilaterals_window = "Cuadriláteros detectados"
estimated_board_window = "Cuadrilátero con más puntos silla"
warp_window = "Warp final"
controls_window = "Controles"

cv.namedWindow(window_name, cv.WINDOW_NORMAL)
cv.namedWindow(sobel_window, cv.WINDOW_NORMAL)
cv.namedWindow(canny_window, cv.WINDOW_NORMAL)
cv.namedWindow(canny_dilated_window, cv.WINDOW_NORMAL)
cv.namedWindow(hough_filtered_window, cv.WINDOW_NORMAL)
cv.namedWindow(contour_window, cv.WINDOW_NORMAL)
cv.namedWindow(approx_window, cv.WINDOW_NORMAL)
cv.namedWindow(quadrilaterals_window, cv.WINDOW_NORMAL)
cv.namedWindow(estimated_board_window, cv.WINDOW_NORMAL)
cv.namedWindow(warp_window, cv.WINDOW_NORMAL)
cv.namedWindow(controls_window, cv.WINDOW_NORMAL)

# Variable para detectar cambios del slider
current_frame = 0

def on_trackbar(pos):
    global current_frame
    current_frame = pos

# Crear la trackbar para el frame
cv.createTrackbar(
    "Frame",
    window_name,
    0,
    total_frames - 1,
    on_trackbar
)

# Variable para el offset
offset_pixels = 0

def on_offset_trackbar(pos):
    global offset_pixels
    offset_pixels = pos

# Crear trackbar para el offset (0-50 píxeles)
cv.createTrackbar(
    "Offset",
    controls_window,
    0,
    50,
    on_offset_trackbar
)

last_frame = -1

# Parámetros
TOLERANCE = 5
MIN_AREA = 10
MIN_PUNTOS_SILLA = 50  # Mínimo de puntos de silla dentro del cuadrilátero

# Variables para almacenar el último cuadrilátero válido
ultimo_cuadrilatero_valido = None
ultimo_warp_valido = None
ultimo_frame_warp = None
ultimos_puntos_ordenados = None

def get_vertices_as_points(polygon):
    """Convierte un polígono a lista de puntos (x,y) en orden"""
    vertices = []
    for point in polygon:
        vertices.append((int(point[0][0]), int(point[0][1])))
    return vertices

def apply_warp(frame, src_points, dst_size=(800, 800)):
    """Aplica warp perspective a la imagen"""
    dst_points = np.array([
        [0, 0],
        [dst_size[0]-1, 0],
        [dst_size[0]-1, dst_size[1]-1],
        [0, dst_size[1]-1]
    ], dtype=np.float32)
    
    src_points = np.array(src_points, dtype=np.float32)
    M = cv.getPerspectiveTransform(src_points, dst_points)
    warped = cv.warpPerspective(frame, M, dst_size)
    return warped

def aplicar_offset_a_puntos(puntos, offset, centro):
    """
    Aplica un offset a los puntos moviéndolos hacia el centro.
    offset: número de píxeles a mover hacia el centro (0 = sin cambio)
    """
    if offset == 0:
        return puntos
    
    puntos_con_offset = []
    for punto in puntos:
        # Vector desde el punto al centro
        vector = centro - punto
        # Normalizar el vector
        norm = np.linalg.norm(vector)
        if norm > 0:
            # Mover el punto hacia el centro
            nuevo_punto = punto + (vector / norm) * offset
        else:
            nuevo_punto = punto
        puntos_con_offset.append(nuevo_punto)
    
    return np.array(puntos_con_offset, dtype=np.float32)

def contar_puntos_silla_en_poligono(polygon, saddle_points):
    """
    Cuenta cuántos puntos de silla están dentro de un polígono.
    """
    if polygon is None or len(saddle_points) == 0:
        return 0
    
    count = 0
    for point in saddle_points:
        distance = cv.pointPolygonTest(polygon, point, True)
        if distance >= 0:  # Dentro o en el borde
            count += 1
    
    return count

def encontrar_cuadrilatero_con_mas_puntos_silla(cuadrilateros, saddle_points):
    """
    Encuentra el cuadrilátero que contiene más puntos de silla dentro.
    Retorna el cuadrilátero y la cantidad de puntos.
    """
    if len(cuadrilateros) == 0:
        return None, 0
    
    mejor_cuadrilatero = None
    max_puntos = 0
    
    for quad in cuadrilateros:
        num_puntos = contar_puntos_silla_en_poligono(quad, saddle_points)
        if num_puntos > max_puntos and num_puntos >= MIN_PUNTOS_SILLA:
            max_puntos = num_puntos
            mejor_cuadrilatero = quad
    
    return mejor_cuadrilatero, max_puntos

def ordenar_puntos_para_warp(puntos):
    """Ordena los 4 puntos para warp: top-left, top-right, bottom-right, bottom-left"""
    center = np.mean(puntos, axis=0)
    angles = np.arctan2(puntos[:, 1] - center[1], puntos[:, 0] - center[0])
    sorted_indices = np.argsort(angles)
    puntos_ordenados = puntos[sorted_indices]
    
    min_sum_idx = np.argmin(puntos_ordenados[:, 0] + puntos_ordenados[:, 1])
    puntos_ordenados = np.roll(puntos_ordenados, -min_sum_idx, axis=0)
    
    area = 0
    for i in range(4):
        j = (i + 1) % 4
        area += puntos_ordenados[i, 0] * puntos_ordenados[j, 1]
        area -= puntos_ordenados[j, 0] * puntos_ordenados[i, 1]
    
    if area > 0:
        puntos_ordenados = puntos_ordenados[::-1]
        min_sum_idx = np.argmin(puntos_ordenados[:, 0] + puntos_ordenados[:, 1])
        puntos_ordenados = np.roll(puntos_ordenados, -min_sum_idx, axis=0)
    
    return puntos_ordenados

def dibujar_cuadrilatero_y_warp(frame, src_points_ordenados, num_puntos, area, offset=0):
    """Dibuja el cuadrilátero en el frame y aplica el warp con offset"""
    estimated_board_img = frame.copy()
    warp_result_img = np.zeros((800, 800, 3), dtype=np.uint8)
    
    # Calcular el centro de los puntos
    centro = np.mean(src_points_ordenados, axis=0)
    
    # Aplicar offset si es necesario
    if offset > 0:
        puntos_con_offset = aplicar_offset_a_puntos(src_points_ordenados, offset, centro)
    else:
        puntos_con_offset = src_points_ordenados
    
    # Mostrar el cuadrilátero seleccionado en el frame original
    pts = puntos_con_offset.astype(np.int32)
    cv.polylines(estimated_board_img, [pts], True, (0, 255, 0), 4)
    for point in pts:
        cv.circle(estimated_board_img, tuple(point), 10, (0, 255, 255), -1)
    
    # Mostrar el offset en la imagen
    cv.putText(estimated_board_img, f"Puntos silla: {num_puntos}", 
              (10, 30), cv.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
    cv.putText(estimated_board_img, f"Area: {int(area)}", 
              (10, 60), cv.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
    cv.putText(estimated_board_img, f"Offset: {offset}px", 
              (10, 90), cv.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)
    
    # Aplicar warp con los puntos con offset
    warped = apply_warp(frame, puntos_con_offset)
    
    # Mostrar el warp final
    warp_result_img = warped
    cv.putText(warp_result_img, f"Puntos silla: {num_puntos}", 
              (10, 30), cv.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
    cv.putText(warp_result_img, f"Area: {int(area)}", 
              (10, 60), cv.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
    cv.putText(warp_result_img, f"Offset: {offset}px", 
              (10, 90), cv.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)
    
    # Dibujar grilla 8x8 en el warp
    h, w = warp_result_img.shape[:2]
    cell_h = h // 8
    cell_w = w // 8
    for i in range(9):
        y = i * cell_h
        cv.line(warp_result_img, (0, y), (w, y), (0, 255, 255), 1)
        x = i * cell_w
        cv.line(warp_result_img, (x, 0), (x, h), (0, 255, 255), 1)
    
    return estimated_board_img, warp_result_img

while True:
    # Solo leer un nuevo frame si cambió la posición
    if current_frame != last_frame:
        cap.set(cv.CAP_PROP_POS_FRAMES, current_frame)
        ret, frame = cap.read()

        if ret:
            frame_vis = frame.copy()
            gray = cv.cvtColor(frame, cv.COLOR_BGR2GRAY)

            # ========== PASO 1: SOBEL Y CANNY ==========
            sobelx = cv.Sobel(gray, cv.CV_64F, 1, 0, ksize=3)
            sobely = cv.Sobel(gray, cv.CV_64F, 0, 1, ksize=3)
            sobel_magnitude = np.sqrt(sobelx**2 + sobely**2)
            sobel_magnitude = np.uint8(np.clip(sobel_magnitude, 0, 255))

            cv.imshow(sobel_window, sobel_magnitude)

            edges = cv.Canny(sobel_magnitude, 7000, 7050, apertureSize=5)
            cv.imshow(canny_window, edges)

            # ========== PASO 2: DILATACIÓN ==========
            kernel = np.ones((3,3), np.uint8)
            edges_dilated = cv.dilate(edges, kernel, iterations=2)
            cv.imshow(canny_dilated_window, edges_dilated)

            # ========== PASO 3: BÚSQUEDA DE CUADRILÁTERO + PUNTOS SILLA ==========
            # Calcular puntos de silla
            Ixx = cv.Sobel(gray, cv.CV_32F, 2, 0, ksize=3)
            Iyy = cv.Sobel(gray, cv.CV_32F, 0, 2, ksize=3)
            Ixy = cv.Sobel(gray, cv.CV_32F, 1, 1, ksize=3)

            response = -(Ixx*Iyy - Ixy*Ixy)
            response = cv.GaussianBlur(response, (15,15), 0)

            mx = cv.dilate(response, np.ones((7,7), np.uint8))
            th = 0.15 * response.max()

            pts = np.where((response == mx) & (response > th))
            points = np.column_stack((pts[1], pts[0]))
            saddle_points = [(int(p[0]), int(p[1])) for p in points]

            # Mostrar puntos de silla
            saddle_points_img = frame.copy()
            if len(points) > 0:
                for point in points:
                    cv.circle(saddle_points_img, (int(point[0]), int(point[1])), 4, (0, 255, 0), -1)
            cv.imshow(hough_filtered_window, saddle_points_img)

            # Encontrar contornos
            contours, hierarchy = cv.findContours(edges_dilated, cv.RETR_EXTERNAL, cv.CHAIN_APPROX_SIMPLE)

            contour_img = frame.copy()
            cv.drawContours(contour_img, contours, -1, (0, 255, 0), 2)
            cv.imshow(contour_window, contour_img)

            # Aproximar polígonos y filtrar por puntos de silla
            all_polygons = []
            contour_approx_img = frame.copy()
            saddle_polygons = []
            
            for contour in contours:
                epsilon = 0.01 * cv.arcLength(contour, True)
                approx = cv.approxPolyDP(contour, epsilon, True)
                all_polygons.append(approx)
                cv.drawContours(contour_approx_img, [approx], -1, (255, 0, 0), 3)
                
                # Filtrar por puntos de silla
                if len(approx) == 4:
                    contains_saddle = False
                    for point in saddle_points:
                        distance = cv.pointPolygonTest(approx, point, True)
                        if distance >= -TOLERANCE:
                            contains_saddle = True
                            break
                    if contains_saddle:
                        saddle_polygons.append(approx)

            cv.imshow(approx_window, contour_approx_img)

            # Detectar cuadriláteros
            quadrilaterals_img = frame.copy()
            quadrilaterals = []
            
            for polygon in saddle_polygons:
                if len(polygon) == 4:
                    area = cv.contourArea(polygon)
                    if cv.isContourConvex(polygon) and area >= MIN_AREA:
                        quadrilaterals.append(polygon)
                        cv.drawContours(quadrilaterals_img, [polygon], -1, (0, 0, 255), 3)
                        for point in polygon:
                            cv.circle(quadrilaterals_img, tuple(point[0]), 6, (0, 255, 255), -1)
                        cv.putText(quadrilaterals_img, f"Area: {int(area)}", 
                                  (int(polygon[0][0][0]), int(polygon[0][0][1]) - 20), 
                                  cv.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2)
            
            cv.imshow(quadrilaterals_window, quadrilaterals_img)

            # ========== PASO 4: SELECCIONAR CUADRILÁTERO CON MÁS PUNTOS SILLA ==========
            warp_result_img = np.zeros((800, 800, 3), dtype=np.uint8)
            estimated_board_img = frame.copy()
            
            # Variable para saber si se encontró un cuadrilátero válido
            encontrado_valido = False
            
            # Obtener el valor actual del offset
            offset_actual = cv.getTrackbarPos("Offset", controls_window)
            
            if len(quadrilaterals) > 0 and len(saddle_points) > 0:
                # Encontrar el cuadrilátero con más puntos de silla (mínimo 50)
                mejor_quad, num_puntos = encontrar_cuadrilatero_con_mas_puntos_silla(quadrilaterals, saddle_points)
                
                if mejor_quad is not None:
                    # Es válido porque tiene al menos MIN_PUNTOS_SILLA puntos
                    encontrado_valido = True
                    
                    # Guardar como último válido
                    ultimo_cuadrilatero_valido = mejor_quad
                    
                    # Obtener los 4 vértices y ordenarlos para warp
                    vertices = get_vertices_as_points(mejor_quad)
                    src_points = np.array(vertices, dtype=np.float32)
                    src_points_ordenados = ordenar_puntos_para_warp(src_points)
                    
                    # Guardar los puntos ordenados para posible reciclaje
                    ultimos_puntos_ordenados = src_points_ordenados
                    
                    # Guardar el frame actual para el warp
                    ultimo_frame_warp = frame.copy()
                    
                    # Aplicar warp y mostrar con offset
                    area = cv.contourArea(mejor_quad)
                    estimated_board_img, warp_result_img = dibujar_cuadrilatero_y_warp(
                        frame, src_points_ordenados, num_puntos, area, offset_actual
                    )
                    
                    # Guardar el warp válido
                    ultimo_warp_valido = warp_result_img.copy()
                    
                    # Añadir indicador de "NUEVO DETECTADO"
                    cv.putText(estimated_board_img, "NUEVO DETECTADO", 
                              (10, 120), cv.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
                    cv.putText(warp_result_img, "NUEVO DETECTADO", 
                              (10, 120), cv.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
            
            # Si no se encontró un cuadrilátero válido, usar el último guardado
            if not encontrado_valido:
                if ultimo_cuadrilatero_valido is not None and ultimo_frame_warp is not None:
                    # Reciclar el último cuadrilátero válido
                    print(f"Reciclando último cuadrilátero válido (Frame {current_frame})")
                    
                    # Usar los últimos puntos ordenados guardados
                    src_points_ordenados = ultimos_puntos_ordenados
                    
                    # Aplicar warp con el frame actual pero usando los puntos guardados
                    area = cv.contourArea(ultimo_cuadrilatero_valido)
                    estimated_board_img, warp_result_img = dibujar_cuadrilatero_y_warp(
                        frame, src_points_ordenados, 
                        contar_puntos_silla_en_poligono(ultimo_cuadrilatero_valido, saddle_points), 
                        area, offset_actual
                    )
                    
                    # Añadir indicador de "RECICLADO"
                    cv.putText(estimated_board_img, "RECICLADO (sin deteccion)", 
                              (10, 120), cv.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)
                    cv.putText(warp_result_img, "RECICLADO (sin deteccion)", 
                              (10, 120), cv.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)
                    
                    # Si tenemos un warp válido guardado, mostrarlo también
                    if ultimo_warp_valido is not None:
                        pass
                else:
                    # No hay ningún cuadrilátero válido guardado
                    cv.putText(estimated_board_img, "SIN DETECCION", 
                              (10, 30), cv.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
                    cv.putText(warp_result_img, "SIN DETECCION", 
                              (10, 30), cv.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
            
            cv.imshow(estimated_board_window, estimated_board_img)
            cv.imshow(warp_window, warp_result_img)
            cv.imshow(window_name, frame_vis)

            last_frame = current_frame

    key = cv.waitKey(20) & 0xFF

    if key == 27:      # ESC para salir
        break

cap.release()
cv.destroyAllWindows()

In [45]:
# PRUEBA CON VIDEO CON HOUGP (CORREGIDO)

import cv2 as cv
import numpy as np
from collections import deque
import random
from copy import copy
from numpy.random import default_rng
import itertools
import math

rng = default_rng()

# Ruta al video
video_path = "../../../data/raw/Prueba2.mp4"

# Abrir el video
cap = cv.VideoCapture(video_path)

if not cap.isOpened():
    raise IOError(f"No se pudo abrir el video: {video_path}")

# Obtener información del video
total_frames = int(cap.get(cv.CAP_PROP_FRAME_COUNT))
fps = cap.get(cv.CAP_PROP_FPS)

print(f"Frames totales: {total_frames}")
print(f"FPS: {fps:.2f}")

# Crear ventanas
window_name = "Video"
sobel_window = "Sobel"
canny_window = "Canny"
canny_dilated_window = "Canny Dilatado"
hough_filtered_window = "Saddle Points"
contour_window = "Contornos"
approx_window = "Polígonos aproximados"
quadrilaterals_window = "Cuadriláteros detectados"
estimated_board_window = "Cuadrilátero con más puntos silla"
warp_window = "Warp final"
controls_window = "Controles"
hough_window = "HoughP - Líneas detectadas"
polygons_binary_window = "Máscara binaria de polígonos"
intersecciones_window = "Intersecciones - Puntos y Cuadriláteros"
orientation_window = "Orientación del Tablero"
stability_window = "Estabilidad de Vértices"

cv.namedWindow(window_name, cv.WINDOW_NORMAL)
cv.namedWindow(sobel_window, cv.WINDOW_NORMAL)
cv.namedWindow(canny_window, cv.WINDOW_NORMAL)
cv.namedWindow(canny_dilated_window, cv.WINDOW_NORMAL)
cv.namedWindow(hough_filtered_window, cv.WINDOW_NORMAL)
cv.namedWindow(contour_window, cv.WINDOW_NORMAL)
cv.namedWindow(approx_window, cv.WINDOW_NORMAL)
cv.namedWindow(quadrilaterals_window, cv.WINDOW_NORMAL)
cv.namedWindow(estimated_board_window, cv.WINDOW_NORMAL)
cv.namedWindow(warp_window, cv.WINDOW_NORMAL)
cv.namedWindow(controls_window, cv.WINDOW_NORMAL)
cv.namedWindow(hough_window, cv.WINDOW_NORMAL)
cv.namedWindow(polygons_binary_window, cv.WINDOW_NORMAL)
cv.namedWindow(intersecciones_window, cv.WINDOW_NORMAL)
cv.namedWindow(orientation_window, cv.WINDOW_NORMAL)
cv.namedWindow(stability_window, cv.WINDOW_NORMAL)

# Variable para detectar cambios del slider
current_frame = 0

def on_trackbar(pos):
    global current_frame
    current_frame = pos

# Crear la trackbar para el frame
cv.createTrackbar(
    "Frame",
    window_name,
    0,
    total_frames - 1,
    on_trackbar
)

# Variable para el offset
offset_pixels = 0

def on_offset_trackbar(pos):
    global offset_pixels
    offset_pixels = pos

# Crear trackbar para el offset (0-50 píxeles)
cv.createTrackbar(
    "Offset",
    controls_window,
    0,
    50,
    on_offset_trackbar
)

# Variables para HoughP
HOUGH_DISTANCE_RESOLUTION = 1  # Resolución de distancia en píxeles
HOUGH_ANGLE_RESOLUTION = np.pi / 180  # Resolución angular en radianes
HOUGH_THRESHOLD = 50  # Umbral de acumulador
HOUGH_MIN_LINE_LENGTH = 50  # Longitud mínima de línea
HOUGH_MAX_LINE_GAP = 10  # Gap máximo entre segmentos

# Porcentaje de tolerancia para reciclaje (10%)
RECYCLE_THRESHOLD_PERCENT = 10

def on_hough_threshold_trackbar(pos):
    global HOUGH_THRESHOLD
    HOUGH_THRESHOLD = max(1, pos)

cv.createTrackbar(
    "Umbral Hough",
    controls_window,
    50,
    200,
    on_hough_threshold_trackbar
)

def on_hough_min_length_trackbar(pos):
    global HOUGH_MIN_LINE_LENGTH
    HOUGH_MIN_LINE_LENGTH = max(5, pos)

cv.createTrackbar(
    "Longitud mínima",
    controls_window,
    50,
    200,
    on_hough_min_length_trackbar
)

def on_hough_max_gap_trackbar(pos):
    global HOUGH_MAX_LINE_GAP
    HOUGH_MAX_LINE_GAP = max(1, pos)

cv.createTrackbar(
    "Gap máximo",
    controls_window,
    10,
    100,
    on_hough_max_gap_trackbar
)

def on_recycle_threshold_trackbar(pos):
    global RECYCLE_THRESHOLD_PERCENT
    RECYCLE_THRESHOLD_PERCENT = max(1, pos)

cv.createTrackbar(
    "Tolerancia Reciclaje %",
    controls_window,
    1,
    100,
    on_recycle_threshold_trackbar
)

# Parámetros de estabilidad
STABILITY_CHECK_FRAMES = 4  # Número de frames a verificar
STABILITY_RADIUS_PX = 30  # Radio máximo de movimiento permitido
stability_buffer = deque(maxlen=STABILITY_CHECK_FRAMES)  # Buffer para almacenar vértices recientes

last_frame = -1

# Parámetros
TOLERANCE = 5
MIN_AREA = 10
MIN_PUNTOS_SILLA = 50

# Variables para almacenar el último cuadrilátero válido
ultimo_cuadrilatero_valido = None
ultimo_warp_valido = None
ultimo_frame_warp = None
ultimos_puntos_ordenados = None
ultimo_num_puntos = 0
ultimo_area = 0
ultimo_offset_usado = 0  # Guardar el offset usado en el último warp

# Flag para saber si ya se ejecutó Hough
hough_ejecutado = False

# Variables para orientación del tablero
primer_warp_realizado = False
esquinas_referencia = None  # Almacena las esquinas en orden: [superior-izquierda, superior-derecha, inferior-derecha, inferior-izquierda]
esquinas_actuales = None

# Umbral para considerar diferencia significativa
DIFERENCIA_ENERGIA_UMBRAL = 0.05  # 5% de diferencia

def get_vertices_as_points(polygon):
    vertices = []
    for point in polygon:
        vertices.append((int(point[0][0]), int(point[0][1])))
    return vertices

def apply_warp(frame, src_points, dst_size=(800, 800)):
    dst_points = np.array([
        [0, 0],
        [dst_size[0]-1, 0],
        [dst_size[0]-1, dst_size[1]-1],
        [0, dst_size[1]-1]
    ], dtype=np.float32)
    
    src_points = np.array(src_points, dtype=np.float32)
    M = cv.getPerspectiveTransform(src_points, dst_points)
    warped = cv.warpPerspective(frame, M, dst_size)
    return warped, M

def aplicar_offset_a_puntos(puntos, offset, centro):
    """
    Aplica offset a los puntos moviéndolos hacia afuera desde el centro
    """
    if offset == 0:
        return puntos
    
    puntos_con_offset = []
    for punto in puntos:
        vector = centro - punto
        norm = np.linalg.norm(vector)
        if norm > 0:
            nuevo_punto = punto + (vector / norm) * offset
        else:
            nuevo_punto = punto
        puntos_con_offset.append(nuevo_punto)
    
    return np.array(puntos_con_offset, dtype=np.float32)

def aplicar_warp_con_offset(frame, src_points, offset_px, dst_size=(800, 800)):
    """
    Aplica warp con offset a los puntos
    """
    # Calcular centro
    centro = np.mean(src_points, axis=0)
    
    # Aplicar offset si es necesario
    if offset_px > 0:
        puntos_offset = aplicar_offset_a_puntos(src_points, offset_px, centro)
    else:
        puntos_offset = src_points
    
    # Aplicar warp
    warped, M = apply_warp(frame, puntos_offset, dst_size)
    
    return warped, M, puntos_offset

# ========== FUNCIONES PARA VERIFICACIÓN DE ESTABILIDAD ==========

def verificar_estabilidad_vertices(vertices_actuales, buffer_vertices, radio_maximo):
    """
    Verifica si los vértices actuales son estables comparándolos con los vértices anteriores
    Retorna: (es_estable, puntaje_estabilidad, distancias_por_vertice)
    """
    if len(buffer_vertices) < STABILITY_CHECK_FRAMES:
        # No hay suficientes datos para verificar
        return True, 1.0, [0.0, 0.0, 0.0, 0.0]
    
    # Convertir a arrays numpy
    vertices_actuales = np.array(vertices_actuales)
    
    # Calcular la distancia promedio de cada vértice con respecto a los frames anteriores
    distancias_por_vertice = []
    
    for i in range(4):  # 4 vértices
        distancias = []
        for vertices_anteriores in buffer_vertices:
            vertices_anteriores = np.array(vertices_anteriores)
            dist = np.linalg.norm(vertices_actuales[i] - vertices_anteriores[i])
            distancias.append(dist)
        
        # Promedio de distancias para este vértice
        dist_promedio = np.mean(distancias)
        distancias_por_vertice.append(dist_promedio)
    
    # Verificar si todos los vértices están dentro del radio máximo
    todos_dentro = all(dist < radio_maximo for dist in distancias_por_vertice)
    
    # Calcular puntaje de estabilidad (0-1)
    # 1 = perfectamente estable, 0 = inestable
    puntaje = 1.0 - (np.mean(distancias_por_vertice) / radio_maximo)
    puntaje = max(0, min(1, puntaje))
    
    return todos_dentro, puntaje, distancias_por_vertice

def dibujar_estabilidad(frame, vertices_actuales, distancias, puntaje, es_estable):
    """
    Dibuja la información de estabilidad en el frame
    """
    img = frame.copy()
    h, w = img.shape[:2]
    
    # Mostrar información de estabilidad
    y_offset = 30
    cv.putText(img, "ESTABILIDAD DE VERTICES", (10, y_offset), 
              cv.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
    y_offset += 30
    
    # Mostrar puntaje
    color = (0, 255, 0) if es_estable else (0, 0, 255)
    cv.putText(img, f"Puntaje: {puntaje:.2f}", (10, y_offset), 
              cv.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
    y_offset += 30
    
    # Mostrar estado
    estado = "ESTABLE ✓" if es_estable else "INESTABLE ✗"
    cv.putText(img, f"Estado: {estado}", (10, y_offset), 
              cv.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
    y_offset += 30
    
    # Mostrar distancias por vértice
    cv.putText(img, "Distancias por vertice:", (10, y_offset), 
              cv.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
    y_offset += 25
    
    for i, dist in enumerate(distancias):
        etiquetas = ["TL", "TR", "BR", "BL"]
        color_vertice = (0, 255, 0) if dist < STABILITY_RADIUS_PX else (0, 0, 255)
        cv.putText(img, f"  {etiquetas[i]}: {dist:.1f}px", (10, y_offset), 
                  cv.FONT_HERSHEY_SIMPLEX, 0.5, color_vertice, 1)
        y_offset += 20
    
    # Dibujar los vértices con colores según estabilidad
    if vertices_actuales is not None:
        for i, punto in enumerate(vertices_actuales):
            x, y = int(punto[0]), int(punto[1])
            # Si el vértice está dentro del radio, verde, sino rojo
            if i < len(distancias) and distancias[i] < STABILITY_RADIUS_PX:
                cv.circle(img, (x, y), 10, (0, 255, 0), -1)
            else:
                cv.circle(img, (x, y), 10, (0, 0, 255), -1)
            etiquetas = ["TL", "TR", "BR", "BL"]
            cv.putText(img, etiquetas[i], (x-15, y-15), 
                      cv.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2)
    
    cv.imshow(stability_window, img)
    return img

# ========== FIN FUNCIONES DE ESTABILIDAD ==========

# ========== FUNCIONES PARA ORIENTACIÓN DEL TABLERO ==========

def calcular_energia_region(imagen, y1, y2, x1, x2):
    """
    Calcula la energía promedio de una región de la imagen
    """
    region = imagen[y1:y2, x1:x2]
    if len(region.shape) == 3:
        energia = np.mean(cv.cvtColor(region, cv.COLOR_BGR2GRAY))
    else:
        energia = np.mean(region)
    return energia

def analizar_filas_y_columnas(warped_img):
    """
    Analiza las filas y columnas del tablero para determinar la orientación correcta
    Retorna: (usar_filas, energia_diff_filas, energia_diff_columnas, 
              necesita_voltear_filas, necesita_voltear_columnas)
    """
    h, w = warped_img.shape[:2]
    cell_h = h // 8
    cell_w = w // 8
    
    # === ANÁLISIS DE FILAS ===
    # Energía de las primeras 2 filas
    energia_filas_sup = 0
    for row in range(2):
        y1 = row * cell_h
        y2 = (row + 1) * cell_h
        energia_filas_sup += calcular_energia_region(warped_img, y1, y2, 0, w)
    energia_filas_sup /= 2
    
    # Energía de las últimas 2 filas
    energia_filas_inf = 0
    for row in range(6, 8):
        y1 = row * cell_h
        y2 = (row + 1) * cell_h
        energia_filas_inf += calcular_energia_region(warped_img, y1, y2, 0, w)
    energia_filas_inf /= 2
    
    diff_filas = abs(energia_filas_sup - energia_filas_inf)
    necesita_voltear_filas = energia_filas_sup > energia_filas_inf
    
    # === ANÁLISIS DE COLUMNAS ===
    # Energía de las primeras 2 columnas
    energia_cols_izq = 0
    for col in range(2):
        x1 = col * cell_w
        x2 = (col + 1) * cell_w
        energia_cols_izq += calcular_energia_region(warped_img, 0, h, x1, x2)
    energia_cols_izq /= 2
    
    # Energía de las últimas 2 columnas
    energia_cols_der = 0
    for col in range(6, 8):
        x1 = col * cell_w
        x2 = (col + 1) * cell_w
        energia_cols_der += calcular_energia_region(warped_img, 0, h, x1, x2)
    energia_cols_der /= 2
    
    diff_columnas = abs(energia_cols_izq - energia_cols_der)
    necesita_voltear_columnas = energia_cols_izq > energia_cols_der
    
    # Decidir si usar filas o columnas (mayor diferencia)
    usar_filas = diff_filas > diff_columnas
    
    return (usar_filas, diff_filas, diff_columnas, 
            necesita_voltear_filas, necesita_voltear_columnas,
            energia_filas_sup, energia_filas_inf,
            energia_cols_izq, energia_cols_der)

def corregir_orientacion_tablero(warped_img, usar_filas, necesita_voltear_filas, necesita_voltear_columnas):
    """
    Corrige la orientación del tablero basado en el análisis de filas/columnas
    """
    img_corregida = warped_img.copy()
    transformaciones = []
    
    if usar_filas:
        # Usar orientación basada en filas
        if necesita_voltear_filas:
            img_corregida = cv.flip(img_corregida, 0)  # Volteo vertical
            transformaciones.append("Volteo Vertical (Filas)")
            print("   ✓ Aplicado volteo vertical basado en filas")
        else:
            print("   ✓ Orientación de filas correcta")
    else:
        # Usar orientación basada en columnas
        if necesita_voltear_columnas:
            # Para columnas, necesitamos trasponer y luego posiblemente voltear
            img_corregida = cv.transpose(img_corregida)
            transformaciones.append("Transposición (Columnas)")
            print("   ✓ Aplicada transposición basada en columnas")
            
            # Después de transponer, las filas ahora son columnas
            # Verificar si necesitamos voltear verticalmente
            if necesita_voltear_columnas:
                img_corregida = cv.flip(img_corregida, 0)
                transformaciones.append("Volteo Vertical")
                print("   ✓ Aplicado volteo vertical después de transposición")
        else:
            print("   ✓ Orientación de columnas correcta")
    
    return img_corregida, transformaciones

def asignar_esquinas_a_referencia(puntos_actuales, puntos_referencia):
    """
    Asigna cada punto actual a la esquina de referencia más cercana
    Retorna los puntos reordenados según la referencia
    """
    if puntos_referencia is None or len(puntos_referencia) != 4:
        return puntos_actuales
    
    puntos_actuales = np.array(puntos_actuales)
    puntos_referencia = np.array(puntos_referencia)
    
    # Para cada punto actual, encontrar la referencia más cercana
    puntos_reordenados = np.zeros_like(puntos_actuales)
    usados = [False] * 4
    
    for i in range(4):
        distancias = []
        for j in range(4):
            if not usados[j]:
                dist = np.linalg.norm(puntos_actuales[i] - puntos_referencia[j])
                distancias.append((dist, j))
        
        if distancias:
            distancias.sort()
            idx_ref = distancias[0][1]
            puntos_reordenados[idx_ref] = puntos_actuales[i]
            usados[idx_ref] = True
    
    return puntos_reordenados

def dibujar_orientacion(frame, warped_img, esquinas, info_text, transformaciones=None):
    """
    Dibuja información de orientación en el frame
    """
    img = frame.copy()
    h, w = img.shape[:2]
    
    # Dibujar las esquinas
    if esquinas is not None:
        for i, punto in enumerate(esquinas):
            x, y = int(punto[0]), int(punto[1])
            colores = [(0, 0, 255), (0, 255, 0), (255, 0, 0), (255, 255, 0)]
            cv.circle(img, (x, y), 8, colores[i % len(colores)], -1)
            etiquetas = ["TL", "TR", "BR", "BL"]  # Top-Left, Top-Right, Bottom-Right, Bottom-Left
            cv.putText(img, etiquetas[i], (x-15, y-15), 
                      cv.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2)
    
    # Mostrar información
    y_offset = 30
    cv.putText(img, "ORIENTACION DEL TABLERO", (10, y_offset), 
              cv.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
    y_offset += 30
    
    cv.putText(img, info_text, (10, y_offset), 
              cv.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
    y_offset += 25
    
    if transformaciones:
        for trans in transformaciones:
            cv.putText(img, f"  -> {trans}", (10, y_offset), 
                      cv.FONT_HERSHEY_SIMPLEX, 0.4, (0, 255, 0), 1)
            y_offset += 20
    
    cv.imshow(orientation_window, img)
    return img

# ========== FIN FUNCIONES ORIENTACIÓN ==========

# ========== FUNCIONES PARA HOUGH PROBABILÍSTICO ==========

def obtener_rectas_hough(imagen_binaria, threshold, min_line_length, max_line_gap):
    """
    Obtiene líneas usando Hough Probabilístico
    Retorna una lista de líneas en formato polar (rho, theta)
    """
    if imagen_binaria is None:
        return []
    
    # Aplicar HoughP
    lines = cv.HoughLinesP(
        imagen_binaria,
        HOUGH_DISTANCE_RESOLUTION,
        HOUGH_ANGLE_RESOLUTION,
        threshold,
        minLineLength=min_line_length,
        maxLineGap=max_line_gap
    )
    
    if lines is None:
        return []
    
    # Convertir a formato polar
    lineas_polar = []
    for line in lines:
        x1, y1, x2, y2 = line[0]
        
        # Calcular parámetros polares (rho, theta)
        dx = x2 - x1
        dy = y2 - y1
        
        # Calcular theta (ángulo de la normal)
        if abs(dx) < 1e-6:
            theta = np.pi / 2 if dy > 0 else -np.pi / 2
        else:
            theta = np.arctan2(dx, -dy)
        
        # Calcular rho
        rho = x1 * np.cos(theta) + y1 * np.sin(theta)
        
        # Calcular orientación
        theta_normalized = theta % np.pi
        es_horizontal = theta_normalized < np.pi/4 or theta_normalized > 3*np.pi/4
        
        lineas_polar.append({
            'rho': rho,
            'theta': theta,
            'puntos': [(x1, y1), (x2, y2)],
            'origen': 'horizontal' if es_horizontal else 'vertical'
        })
    
    return lineas_polar

def agrupar_lineas_por_orientacion(lineas):
    """
    Agrupa las líneas por orientación (horizontal/vertical)
    """
    horizontales = []
    verticales = []
    
    for linea in lineas:
        if linea['origen'] == 'horizontal':
            horizontales.append(linea)
        else:
            verticales.append(linea)
    
    return horizontales, verticales

def fusionar_lineas_cercanas(lineas, umbral_rho=20, umbral_theta=0.1):
    """
    Fusiona líneas que son cercanas entre sí
    """
    if len(lineas) < 2:
        return lineas
    
    fusionadas = []
    usadas = [False] * len(lineas)
    
    for i in range(len(lineas)):
        if usadas[i]:
            continue
        
        grupo = [i]
        usadas[i] = True
        
        for j in range(i + 1, len(lineas)):
            if usadas[j]:
                continue
            
            rho1, theta1 = lineas[i]['rho'], lineas[i]['theta']
            rho2, theta2 = lineas[j]['rho'], lineas[j]['theta']
            
            # Normalizar theta
            theta1_norm = theta1 % np.pi
            theta2_norm = theta2 % np.pi
            
            diff_rho = abs(rho1 - rho2)
            diff_theta = abs(theta1_norm - theta2_norm)
            diff_theta = min(diff_theta, np.pi - diff_theta)
            
            if diff_rho < umbral_rho and diff_theta < umbral_theta:
                grupo.append(j)
                usadas[j] = True
        
        # Promediar las líneas del grupo
        if len(grupo) > 0:
            rho_prom = np.mean([lineas[idx]['rho'] for idx in grupo])
            theta_prom = np.mean([lineas[idx]['theta'] for idx in grupo])
            
            # Usar los puntos de la línea con más inliers (si existiera)
            # O simplemente usar el primero
            fusionadas.append({
                'rho': rho_prom,
                'theta': theta_prom,
                'puntos': lineas[grupo[0]]['puntos'],
                'origen': lineas[grupo[0]]['origen']
            })
    
    return fusionadas

def recta_polar_a_puntos(recta_polar, ancho, alto):
    """
    Convierte una recta en representación polar (rho, theta) a dos puntos
    para dibujarla en una imagen de dimensiones (ancho, alto)
    """
    rho, theta = recta_polar
    a = np.cos(theta)
    b = np.sin(theta)
    
    # Calcular puntos extremos
    if abs(a) > 0.01:  # No es vertical
        x1 = 0
        y1 = int((rho - x1 * a) / b) if abs(b) > 1e-6 else 0
        x2 = ancho
        y2 = int((rho - x2 * a) / b) if abs(b) > 1e-6 else 0
    else:  # Línea vertical
        y1 = 0
        x1 = int(rho / a) if abs(a) > 1e-6 else 0
        y2 = alto
        x2 = int(rho / a) if abs(a) > 1e-6 else 0
    
    # Asegurar que los puntos estén dentro de la imagen
    x1 = max(0, min(ancho, x1))
    x2 = max(0, min(ancho, x2))
    y1 = max(0, min(alto, y1))
    y2 = max(0, min(alto, y2))
    
    return (x1, y1), (x2, y2)

def calcular_interseccion_polar(recta1, recta2):
    """
    Calcula la intersección de dos rectas en representación polar (rho, theta)
    Retorna el punto (x, y) o None si son paralelas
    """
    rho1, theta1 = recta1
    rho2, theta2 = recta2
    
    # Si las rectas son paralelas (theta iguales o diferencia de 180 grados)
    if abs(theta1 - theta2) < 1e-6 or abs(abs(theta1 - theta2) - np.pi) < 1e-6:
        return None
    
    # Resolver sistema de ecuaciones:
    # x*cos(theta1) + y*sin(theta1) = rho1
    # x*cos(theta2) + y*sin(theta2) = rho2
    
    A = np.array([[np.cos(theta1), np.sin(theta1)],
                  [np.cos(theta2), np.sin(theta2)]])
    b = np.array([rho1, rho2])
    
    try:
        x, y = np.linalg.solve(A, b)
        return (int(x), int(y))
    except np.linalg.LinAlgError:
        return None

def formar_cuadrilateros_desde_rectas(rectas_horizontal, rectas_vertical, saddle_points, frame_shape):
    """
    Forma cuadriláteros a partir de las intersecciones de rectas horizontales y verticales
    """
    h, w = frame_shape[:2]
    cuadrilateros = []
    
    # Tomar las mejores líneas (más largas/confiables)
    # Ordenar por longitud de segmento
    def longitud_linea(linea):
        p1, p2 = linea['puntos']
        return np.sqrt((p2[0]-p1[0])**2 + (p2[1]-p1[1])**2)
    
    rectas_horizontal.sort(key=longitud_linea, reverse=True)
    rectas_vertical.sort(key=longitud_linea, reverse=True)
    
    # Tomar las primeras N líneas
    max_lineas = min(8, len(rectas_horizontal))
    rectas_horizontal = rectas_horizontal[:max_lineas]
    max_lineas = min(8, len(rectas_vertical))
    rectas_vertical = rectas_vertical[:max_lineas]
    
    for combo_h in itertools.combinations(range(len(rectas_horizontal)), 2):
        for combo_v in itertools.combinations(range(len(rectas_vertical)), 2):
            h1 = rectas_horizontal[combo_h[0]]
            h2 = rectas_horizontal[combo_h[1]]
            v1 = rectas_vertical[combo_v[0]]
            v2 = rectas_vertical[combo_v[1]]
            
            # Obtener parámetros polares
            h1_recta = (h1['rho'], h1['theta'])
            h2_recta = (h2['rho'], h2['theta'])
            v1_recta = (v1['rho'], v1['theta'])
            v2_recta = (v2['rho'], v2['theta'])
            
            # Calcular intersecciones
            inter1 = calcular_interseccion_polar(h1_recta, v1_recta)
            inter2 = calcular_interseccion_polar(h1_recta, v2_recta)
            inter3 = calcular_interseccion_polar(h2_recta, v2_recta)
            inter4 = calcular_interseccion_polar(h2_recta, v1_recta)
            
            if None in [inter1, inter2, inter3, inter4]:
                continue
            
            vertices = [inter1, inter2, inter3, inter4]
            
            # Verificar que los vértices estén dentro de la imagen
            dentro = True
            for v in vertices:
                if v[0] < 0 or v[0] >= w or v[1] < 0 or v[1] >= h:
                    dentro = False
                    break
            
            if not dentro:
                continue
            
            vertices_array = np.array(vertices, dtype=np.float32)
            vertices_ordenados = ordenar_vertices_horario(vertices_array)
            
            if vertices_ordenados is None:
                continue
            
            area = cv.contourArea(vertices_ordenados.astype(np.int32))
            if area < 100:
                continue
            
            poly_contour = vertices_ordenados.astype(np.int32).reshape(-1, 1, 2)
            if not cv.isContourConvex(poly_contour):
                continue
            
            num_puntos = contar_puntos_silla_en_poligono(poly_contour, saddle_points)
            
            if num_puntos >= MIN_PUNTOS_SILLA:
                ratio = num_puntos / np.sqrt(area) if area > 0 else 0
                cuadrilateros.append({
                    'vertices': vertices_ordenados,
                    'num_puntos': num_puntos,
                    'area': area,
                    'ratio': ratio,
                    'h1': h1,
                    'h2': h2,
                    'v1': v1,
                    'v2': v2
                })
    
    cuadrilateros.sort(key=lambda x: x['ratio'], reverse=True)
    return cuadrilateros

def dibujar_intersecciones_y_cuadrilateros(frame, rectas_horizontal, rectas_vertical, saddle_points, cuadrilateros):
    """
    Dibuja las rectas, intersecciones y cuadriláteros
    """
    img = frame.copy()
    h, w = img.shape[:2]
    
    for punto in saddle_points:
        cv.circle(img, (int(punto[0]), int(punto[1])), 2, (0, 255, 255), -1)
    
    # Dibujar rectas horizontales
    for recta in rectas_horizontal:
        rho, theta = recta['rho'], recta['theta']
        p1, p2 = recta_polar_a_puntos((rho, theta), w, h)
        cv.line(img, p1, p2, (255, 150, 0), 1)
    
    # Dibujar rectas verticales
    for recta in rectas_vertical:
        rho, theta = recta['rho'], recta['theta']
        p1, p2 = recta_polar_a_puntos((rho, theta), w, h)
        cv.line(img, p1, p2, (0, 200, 100), 1)
    
    # Dibujar intersecciones
    for h_rect in rectas_horizontal:
        h_polar = (h_rect['rho'], h_rect['theta'])
        for v_rect in rectas_vertical:
            v_polar = (v_rect['rho'], v_rect['theta'])
            inter = calcular_interseccion_polar(h_polar, v_polar)
            if inter is not None:
                x, y = inter
                if 0 <= x < w and 0 <= y < h:
                    cv.circle(img, (x, y), 4, (0, 255, 0), -1)
    
    y_offset = 30
    cv.putText(img, f"Cuadrilateros validos (min {MIN_PUNTOS_SILLA} pts): {len(cuadrilateros)}", 
              (10, y_offset), cv.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
    y_offset += 25
    
    colores = [
        (255, 0, 0), (0, 255, 0), (0, 0, 255), (255, 255, 0),
        (255, 0, 255), (0, 255, 255), (128, 0, 255), (255, 128, 0)
    ]
    
    for idx, cuad in enumerate(cuadrilateros[:8]):
        vertices = cuad['vertices']
        color = colores[idx % len(colores)]
        pts = vertices.astype(np.int32)
        cv.polylines(img, [pts], True, color, 2)
        
        centro = np.mean(vertices, axis=0).astype(int)
        cv.putText(img, f"{cuad['ratio']:.4f}", (centro[0]-20, centro[1]), 
                  cv.FONT_HERSHEY_SIMPLEX, 0.4, color, 1)
        
        if idx < 6:
            texto = f"#{idx+1}: {cuad['num_puntos']}pts/{cuad['area']:.0f}px = {cuad['ratio']:.4f}"
            cv.putText(img, texto, (10, y_offset), 
                      cv.FONT_HERSHEY_SIMPLEX, 0.4, color, 1)
            y_offset += 18
    
    if len(cuadrilateros) > 0:
        mejor = cuadrilateros[0]
        mejor_vertices = mejor['vertices']
        for v in mejor_vertices:
            cv.circle(img, (int(v[0]), int(v[1])), 8, (0, 255, 255), -1)
            cv.circle(img, (int(v[0]), int(v[1])), 10, (255, 255, 255), 1)
        
        cv.putText(img, f"MEJOR: {mejor['num_puntos']}pts / {mejor['area']:.0f}px = {mejor['ratio']:.4f}", 
                  (10, y_offset + 10), cv.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255), 2)
    
    cv.imshow(intersecciones_window, img)
    return img

def dibujar_lineas_hough(frame, lineas_horizontal, lineas_vertical):
    """
    Dibuja las líneas encontradas por HoughP
    """
    img = frame.copy()
    h, w = img.shape[:2]
    
    colores_h = [(255, 150, 0), (255, 100, 0), (200, 80, 0), (150, 50, 0)]
    colores_v = [(0, 200, 100), (0, 150, 80), (0, 100, 60), (0, 80, 50)]
    
    # Dibujar líneas horizontales con sus segmentos originales
    for i, recta in enumerate(lineas_horizontal):
        # Dibujar el segmento original
        p1, p2 = recta['puntos']
        color = colores_h[i % len(colores_h)]
        cv.line(img, p1, p2, color, 2)
        
        # Dibujar la línea extendida
        rho, theta = recta['rho'], recta['theta']
        p1_ext, p2_ext = recta_polar_a_puntos((rho, theta), w, h)
        cv.line(img, p1_ext, p2_ext, color, 1)
        
        cv.putText(img, f"H{i+1}", (p1[0], p1[1] - 10), 
                  cv.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)
    
    # Dibujar líneas verticales
    for i, recta in enumerate(lineas_vertical):
        # Dibujar el segmento original
        p1, p2 = recta['puntos']
        color = colores_v[i % len(colores_v)]
        cv.line(img, p1, p2, color, 2)
        
        # Dibujar la línea extendida
        rho, theta = recta['rho'], recta['theta']
        p1_ext, p2_ext = recta_polar_a_puntos((rho, theta), w, h)
        cv.line(img, p1_ext, p2_ext, color, 1)
        
        cv.putText(img, f"V{i+1}", (p1[0], p1[1] - 10), 
                  cv.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)
    
    cv.putText(img, f"Lineas H: {len(lineas_horizontal)}, Lineas V: {len(lineas_vertical)}", 
              (10, 30), cv.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
    
    cv.imshow(hough_window, img)
    return img

# ========== FIN DE FUNCIONES PARA HOUGH ==========

def contar_puntos_silla_en_poligono(polygon, saddle_points):
    if polygon is None or len(saddle_points) == 0:
        return 0
    
    count = 0
    for point in saddle_points:
        distance = cv.pointPolygonTest(polygon, point, True)
        if distance >= 0:
            count += 1
    
    return count

def encontrar_cuadrilatero_con_mas_puntos_silla(cuadrilateros, saddle_points):
    if len(cuadrilateros) == 0:
        return None, 0
    
    mejor_cuadrilatero = None
    max_puntos = 0
    
    for quad in cuadrilateros:
        num_puntos = contar_puntos_silla_en_poligono(quad, saddle_points)
        if num_puntos > max_puntos and num_puntos >= MIN_PUNTOS_SILLA:
            max_puntos = num_puntos
            mejor_cuadrilatero = quad
    
    return mejor_cuadrilatero, max_puntos

def ordenar_vertices_horario(vertices):
    if len(vertices) != 4:
        return None
    
    center = np.mean(vertices, axis=0)
    angles = np.arctan2(vertices[:, 1] - center[1], vertices[:, 0] - center[0])
    sorted_indices = np.argsort(angles)
    vertices_ordenados = vertices[sorted_indices]
    
    vertices_ordenados = vertices_ordenados[::-1]
    
    min_sum_idx = np.argmin(vertices_ordenados[:, 0] + vertices_ordenados[:, 1])
    vertices_ordenados = np.roll(vertices_ordenados, -min_sum_idx, axis=0)
    
    return vertices_ordenados

def ordenar_puntos_para_warp(puntos):
    center = np.mean(puntos, axis=0)
    angles = np.arctan2(puntos[:, 1] - center[1], puntos[:, 0] - center[0])
    sorted_indices = np.argsort(angles)
    puntos_ordenados = puntos[sorted_indices]
    
    puntos_ordenados = puntos_ordenados[::-1]
    
    min_sum_idx = np.argmin(puntos_ordenados[:, 0] + puntos_ordenados[:, 1])
    puntos_ordenados = np.roll(puntos_ordenados, -min_sum_idx, axis=0)
    
    return puntos_ordenados

def dibujar_cuadrilatero_y_warp(frame, src_points_ordenados, num_puntos, area, offset=0, 
                                es_hough=False, es_reciclado=False, warp_corregido=None, 
                                puntos_con_offset=None):
    """
    Dibuja el cuadrilátero y el warp con offset aplicado
    """
    estimated_board_img = frame.copy()
    
    # Si tenemos warp corregido, usarlo
    if warp_corregido is not None:
        warp_result_img = warp_corregido
    else:
        warp_result_img = np.zeros((800, 800, 3), dtype=np.uint8)
    
    # Si no se proporcionaron puntos con offset, calcularlos
    if puntos_con_offset is None:
        centro = np.mean(src_points_ordenados, axis=0)
        puntos_con_offset = aplicar_offset_a_puntos(src_points_ordenados, offset, centro)
    
    pts = puntos_con_offset.astype(np.int32)
    cv.polylines(estimated_board_img, [pts], True, (0, 255, 0), 4)
    for point in pts:
        cv.circle(estimated_board_img, tuple(point), 10, (0, 255, 255), -1)
    
    if es_reciclado:
        metodo = "RECICLADO"
        color_texto = (0, 255, 255)
    elif es_hough:
        metodo = "HOUGH"
        color_texto = (0, 255, 255)
    else:
        metodo = "Detección"
        color_texto = (255, 255, 0)
    
    cv.putText(estimated_board_img, f"Metodo: {metodo}", 
              (10, 30), cv.FONT_HERSHEY_SIMPLEX, 0.6, color_texto, 2)
    cv.putText(estimated_board_img, f"Puntos silla: {num_puntos}", 
              (10, 60), cv.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
    cv.putText(estimated_board_img, f"Area: {int(area)}", 
              (10, 90), cv.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
    cv.putText(estimated_board_img, f"Offset: {offset}px", 
              (10, 120), cv.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)
    
    if es_reciclado:
        cv.putText(estimated_board_img, "RECICLADO (tolerancia)", 
                  (10, 150), cv.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)
    
    # Si el warp fue corregido, mostrar información
    if warp_corregido is not None:
        cv.putText(estimated_board_img, "ORIENTACION CORREGIDA", 
                  (10, 180), cv.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
        cv.putText(estimated_board_img, f"Offset aplicado: {offset}px", 
                  (10, 210), cv.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 0), 1)
    
    if es_reciclado:
        cv.putText(warp_result_img, "RECICLADO", 
                  (10, 30), cv.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)
    
    cv.putText(warp_result_img, f"Puntos silla: {num_puntos}", 
              (10, 60), cv.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
    cv.putText(warp_result_img, f"Area: {int(area)}", 
              (10, 90), cv.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
    cv.putText(warp_result_img, f"Offset: {offset}px", 
              (10, 120), cv.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)
    
    # Dibujar grid
    h, w = warp_result_img.shape[:2]
    cell_h = h // 8
    cell_w = w // 8
    for i in range(9):
        y = i * cell_h
        cv.line(warp_result_img, (0, y), (w, y), (0, 255, 255), 1)
        x = i * cell_w
        cv.line(warp_result_img, (x, 0), (x, h), (0, 255, 255), 1)
    
    # Marcar la orientación en el warp
    cv.putText(warp_result_img, "NEGRAS", (10, 30), 
              cv.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 2)
    cv.putText(warp_result_img, "BLANCAS", (10, h - 10), 
              cv.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2)
    
    return estimated_board_img, warp_result_img

def mostrar_mensaje_hough(frame, mensaje, color=(0, 0, 255)):
    img = frame.copy()
    h, w = img.shape[:2]
    cv.putText(img, mensaje, (w//2 - 150, h//2), 
              cv.FONT_HERSHEY_SIMPLEX, 1, color, 2)
    cv.imshow(hough_window, img)
    return img

while True:
    if current_frame != last_frame:
        cap.set(cv.CAP_PROP_POS_FRAMES, current_frame)
        ret, frame = cap.read()

        if ret:
            frame_vis = frame.copy()
            gray = cv.cvtColor(frame, cv.COLOR_BGR2GRAY)

            # ========== PASO 1: SOBEL Y CANNY ==========
            sobelx = cv.Sobel(gray, cv.CV_64F, 1, 0, ksize=3)
            sobely = cv.Sobel(gray, cv.CV_64F, 0, 1, ksize=3)
            sobel_magnitude = np.sqrt(sobelx**2 + sobely**2)
            sobel_magnitude = np.uint8(np.clip(sobel_magnitude, 0, 255))

            cv.imshow(sobel_window, sobel_magnitude)

            edges = cv.Canny(sobel_magnitude, 7000, 7050, apertureSize=5)
            cv.imshow(canny_window, edges)

            # ========== PASO 2: DILATACIÓN ==========
            kernel = np.ones((3,3), np.uint8)
            edges_dilated = cv.dilate(edges, kernel, iterations=2)
            cv.imshow(canny_dilated_window, edges_dilated)

            # ========== PASO 3: BÚSQUEDA DE CUADRILÁTERO + PUNTOS SILLA ==========
            Ixx = cv.Sobel(gray, cv.CV_32F, 2, 0, ksize=3)
            Iyy = cv.Sobel(gray, cv.CV_32F, 0, 2, ksize=3)
            Ixy = cv.Sobel(gray, cv.CV_32F, 1, 1, ksize=3)

            response = -(Ixx*Iyy - Ixy*Ixy)
            response = cv.GaussianBlur(response, (15,15), 0)

            mx = cv.dilate(response, np.ones((7,7), np.uint8))
            th = 0.15 * response.max()

            pts = np.where((response == mx) & (response > th))
            points = np.column_stack((pts[1], pts[0]))
            saddle_points = [(int(p[0]), int(p[1])) for p in points]

            saddle_points_img = frame.copy()
            if len(points) > 0:
                for point in points:
                    cv.circle(saddle_points_img, (int(point[0]), int(point[1])), 4, (0, 255, 0), -1)
            cv.imshow(hough_filtered_window, saddle_points_img)

            contours, hierarchy = cv.findContours(edges_dilated, cv.RETR_EXTERNAL, cv.CHAIN_APPROX_SIMPLE)

            contour_img = frame.copy()
            cv.drawContours(contour_img, contours, -1, (0, 255, 0), 2)
            cv.imshow(contour_window, contour_img)

            # ========== CREAR IMAGEN BINARIA ==========
            polygons_binary = np.zeros_like(gray)
            all_polygons = []
            contour_approx_img = frame.copy()
            saddle_polygons = []
            
            for contour in contours:
                epsilon = 0.01 * cv.arcLength(contour, True)
                approx = cv.approxPolyDP(contour, epsilon, True)
                all_polygons.append(approx)
                cv.drawContours(contour_approx_img, [approx], -1, (255, 0, 0), 3)
                
                cv.drawContours(polygons_binary, [approx], -1, 255, 1)
                
                if len(approx) == 4:
                    contains_saddle = False
                    for point in saddle_points:
                        distance = cv.pointPolygonTest(approx, point, True)
                        if distance >= -TOLERANCE:
                            contains_saddle = True
                            break
                    if contains_saddle:
                        saddle_polygons.append(approx)

            cv.imshow(approx_window, contour_approx_img)
            cv.imshow(polygons_binary_window, polygons_binary)

            quadrilaterals_img = frame.copy()
            quadrilaterals = []
            
            for polygon in saddle_polygons:
                if len(polygon) == 4:
                    area = cv.contourArea(polygon)
                    if cv.isContourConvex(polygon) and area >= MIN_AREA:
                        quadrilaterals.append(polygon)
                        cv.drawContours(quadrilaterals_img, [polygon], -1, (0, 0, 255), 3)
                        for point in polygon:
                            cv.circle(quadrilaterals_img, tuple(point[0]), 6, (0, 255, 255), -1)
                        cv.putText(quadrilaterals_img, f"Area: {int(area)}", 
                                  (int(polygon[0][0][0]), int(polygon[0][0][1]) - 20), 
                                  cv.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2)
            
            cv.imshow(quadrilaterals_window, quadrilaterals_img)

            # ========== PASO 4: SELECCIONAR CUADRILÁTERO ==========
            warp_result_img = np.zeros((800, 800, 3), dtype=np.uint8)
            estimated_board_img = frame.copy()
            
            offset_actual = cv.getTrackbarPos("Offset", controls_window)
            hough_threshold = cv.getTrackbarPos("Umbral Hough", controls_window)
            hough_min_length = cv.getTrackbarPos("Longitud mínima", controls_window)
            hough_max_gap = cv.getTrackbarPos("Gap máximo", controls_window)
            recycle_threshold = cv.getTrackbarPos("Tolerancia Reciclaje %", controls_window) / 100.0
            
            # Variables de estado
            encontrado_valido = False
            usando_hough = False
            usando_reciclado = False
            
            # Variables para warp corregido
            warp_corregido = None
            puntos_ordenados_final = None
            puntos_con_offset_final = None
            num_puntos_final = 0
            area_final = 0
            info_orientacion = ""
            transformaciones = []
            
            # Variables para estabilidad
            vertices_actuales = None
            es_estable = True
            puntaje_estabilidad = 1.0
            distancias_estabilidad = [0.0, 0.0, 0.0, 0.0]
            
            # ===== PASO 1: BUSCAR CUADRILÁTERO CON PUNTOS DE SILLA =====
            if len(quadrilaterals) > 0 and len(saddle_points) > 0:
                mejor_quad, num_puntos = encontrar_cuadrilatero_con_mas_puntos_silla(quadrilaterals, saddle_points)
                
                if mejor_quad is not None:
                    encontrado_valido = True
                    print(f"✅ CUADRILÁTERO DETECTADO: {num_puntos} puntos silla")
                    
                    # Guardar como último válido
                    ultimo_cuadrilatero_valido = mejor_quad
                    vertices = get_vertices_as_points(mejor_quad)
                    src_points = np.array(vertices, dtype=np.float32)
                    src_points_ordenados = ordenar_puntos_para_warp(src_points)
                    
                    ultimos_puntos_ordenados = src_points_ordenados
                    ultimo_frame_warp = frame.copy()
                    area = cv.contourArea(mejor_quad)
                    ultimo_num_puntos = num_puntos
                    ultimo_area = area
                    
                    # Aplicar warp con offset
                    warped, M, puntos_con_offset = aplicar_warp_con_offset(frame, src_points_ordenados, offset_actual)
                    warp_result_img = warped
                    ultimo_offset_usado = offset_actual
                    
                    # Guardar vértices actuales para estabilidad
                    vertices_actuales = src_points_ordenados.copy()
                    
                    # Verificar estabilidad
                    if len(stability_buffer) > 0:
                        es_estable, puntaje_estabilidad, distancias_estabilidad = verificar_estabilidad_vertices(
                            vertices_actuales, stability_buffer, STABILITY_RADIUS_PX
                        )
                        print(f"   Estabilidad: {'ESTABLE' if es_estable else 'INESTABLE'} (puntaje: {puntaje_estabilidad:.2f})")
                    else:
                        print("   Estabilidad: PRIMER FRAME (sin referencia)")
                    
                    # Agregar al buffer de estabilidad
                    stability_buffer.append(vertices_actuales.copy())
                    
                    # ===== CORREGIR ORIENTACIÓN =====
                    if not primer_warp_realizado:
                        print("🔄 PRIMERA VEZ - Analizando orientación...")
                        
                        # 1. Analizar filas y columnas
                        (usar_filas, diff_filas, diff_columnas, 
                         necesita_voltear_filas, necesita_voltear_columnas,
                         energia_filas_sup, energia_filas_inf,
                         energia_cols_izq, energia_cols_der) = analizar_filas_y_columnas(warped)
                        
                        print(f"   Diferencias - Filas: {diff_filas:.2f}, Columnas: {diff_columnas:.2f}")
                        print(f"   Usando: {'FILAS' if usar_filas else 'COLUMNAS'}")
                        print(f"   Energía filas - Superior: {energia_filas_sup:.2f}, Inferior: {energia_filas_inf:.2f}")
                        print(f"   Energía columnas - Izquierda: {energia_cols_izq:.2f}, Derecha: {energia_cols_der:.2f}")
                        
                        # 2. Corregir orientación
                        warp_corregido, transformaciones = corregir_orientacion_tablero(
                            warped, usar_filas, necesita_voltear_filas, necesita_voltear_columnas
                        )
                        
                        # 3. Establecer esquinas de referencia
                        esquinas_referencia = np.array([
                            [0, 0],
                            [799, 0],
                            [799, 799],
                            [0, 799]
                        ], dtype=np.float32)
                        
                        # Ajustar esquinas según transformaciones
                        if 'Transposición (Columnas)' in transformaciones:
                            # Intercambiar x e y
                            esquinas_referencia = np.array([
                                [0, 799],
                                [0, 0],
                                [799, 0],
                                [799, 799]
                            ], dtype=np.float32)
                        
                        if 'Volteo Vertical (Filas)' in transformaciones or 'Volteo Vertical' in transformaciones:
                            # Voltear verticalmente
                            esquinas_referencia = np.array([
                                [0, 799],
                                [799, 799],
                                [799, 0],
                                [0, 0]
                            ], dtype=np.float32)
                        
                        primer_warp_realizado = True
                        info_orientacion = f"Usando: {'FILAS' if usar_filas else 'COLUMNAS'}"
                        print(f"✅ Orientación corregida: {info_orientacion}")
                        
                        warp_result_img = warp_corregido
                        puntos_ordenados_final = src_points_ordenados
                        puntos_con_offset_final = puntos_con_offset
                        num_puntos_final = num_puntos
                        area_final = area
                        
                        # Dibujar información de orientación
                        dibujar_orientacion(frame, warp_corregido, esquinas_referencia, info_orientacion, transformaciones)
                        
                    else:
                        # No es la primera vez - asignar esquinas por proximidad
                        print("🔄 Asignando esquinas por proximidad...")
                        
                        # Reordenar puntos según referencia
                        puntos_reordenados = asignar_esquinas_a_referencia(src_points_ordenados, esquinas_referencia)
                        puntos_ordenados_final = puntos_reordenados
                        num_puntos_final = num_puntos
                        area_final = area
                        
                        # Aplicar warp con los puntos reordenados y offset
                        warped, M, puntos_con_offset = aplicar_warp_con_offset(frame, puntos_reordenados, offset_actual)
                        warp_result_img = warped
                        warp_corregido = warped
                        puntos_con_offset_final = puntos_con_offset
                        
                        info_orientacion = "Esquinas asignadas por proximidad"
                        print("✅ Esquinas asignadas")
                        
                        # Dibujar información de orientación
                        dibujar_orientacion(frame, warped, esquinas_referencia, info_orientacion)
                    
                    # Dibujar información de estabilidad
                    dibujar_estabilidad(frame, vertices_actuales, distancias_estabilidad, puntaje_estabilidad, es_estable)
                    
                    ultimo_warp_valido = warp_result_img.copy()
                    
                    estimated_board_img, warp_result_img = dibujar_cuadrilatero_y_warp(
                        frame, puntos_ordenados_final, num_puntos_final, area_final, offset_actual, 
                        False, False, warp_corregido, puntos_con_offset_final
                    )
                    
                    cv.putText(estimated_board_img, "DETECCION TRADICIONAL", 
                              (10, 150), cv.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)
                    
                    # Mostrar estado de estabilidad en el warp
                    if es_estable:
                        cv.putText(estimated_board_img, f"ESTABLE (puntaje: {puntaje_estabilidad:.2f})", 
                                  (10, 240), cv.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
                        cv.putText(warp_result_img, f"ESTABLE (puntaje: {puntaje_estabilidad:.2f})", 
                                  (10, 150), cv.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
                    else:
                        cv.putText(estimated_board_img, f"INESTABLE (puntaje: {puntaje_estabilidad:.2f})", 
                                  (10, 240), cv.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2)
                        cv.putText(warp_result_img, f"INESTABLE (puntaje: {puntaje_estabilidad:.2f})", 
                                  (10, 150), cv.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2)
                    
                    cv.imshow(hough_window, np.zeros_like(frame))
                    cv.imshow(intersecciones_window, np.zeros_like(frame))
            
            # ===== PASO 2: SI NO HAY CUADRILÁTERO, USAR HOUGH =====
            if not encontrado_valido:
                if polygons_binary is not None and np.sum(polygons_binary) > 100:
                    
                    # Si Hough no se ha ejecutado nunca, ejecutarlo
                    if not hough_ejecutado:
                        print(f"🔄 EJECUTANDO HOUGH PROBABILÍSTICO")
                        usando_hough = True
                        
                        # Obtener líneas con HoughP
                        lineas = obtener_rectas_hough(
                            polygons_binary, 
                            hough_threshold,
                            hough_min_length,
                            hough_max_gap
                        )
                        
                        if len(lineas) > 0:
                            # Agrupar por orientación
                            rectas_horizontal, rectas_vertical = agrupar_lineas_por_orientacion(lineas)
                            
                            # Fusionar líneas cercanas
                            rectas_horizontal = fusionar_lineas_cercanas(rectas_horizontal)
                            rectas_vertical = fusionar_lineas_cercanas(rectas_vertical)
                            
                            print(f"   Líneas horizontales: {len(rectas_horizontal)}, verticales: {len(rectas_vertical)}")
                        else:
                            rectas_horizontal = []
                            rectas_vertical = []
                        
                        dibujar_lineas_hough(frame, rectas_horizontal, rectas_vertical)
                        
                        if len(rectas_horizontal) >= 2 and len(rectas_vertical) >= 2:
                            cuadrilateros_hough = formar_cuadrilateros_desde_rectas(
                                rectas_horizontal, rectas_vertical, saddle_points, frame.shape
                            )
                        else:
                            cuadrilateros_hough = []
                        
                        dibujar_intersecciones_y_cuadrilateros(
                            frame, rectas_horizontal, rectas_vertical, saddle_points, cuadrilateros_hough
                        )
                        
                        if len(cuadrilateros_hough) > 0:
                            mejor = cuadrilateros_hough[0]
                            mejor_vertices = mejor['vertices']
                            num_puntos = mejor['num_puntos']
                            area = mejor['area']
                            
                            # Guardar como último válido
                            src_points = np.array(mejor_vertices, dtype=np.float32)
                            src_points_ordenados = ordenar_puntos_para_warp(src_points)
                            
                            # Aplicar warp con offset
                            warped, M, puntos_con_offset = aplicar_warp_con_offset(frame, src_points_ordenados, offset_actual)
                            
                            # Guardar vértices actuales para estabilidad
                            vertices_actuales = src_points_ordenados.copy()
                            
                            # Verificar estabilidad
                            if len(stability_buffer) > 0:
                                es_estable, puntaje_estabilidad, distancias_estabilidad = verificar_estabilidad_vertices(
                                    vertices_actuales, stability_buffer, STABILITY_RADIUS_PX
                                )
                                print(f"   Estabilidad: {'ESTABLE' if es_estable else 'INESTABLE'} (puntaje: {puntaje_estabilidad:.2f})")
                            else:
                                print("   Estabilidad: PRIMER FRAME (sin referencia)")
                            
                            # Agregar al buffer de estabilidad
                            stability_buffer.append(vertices_actuales.copy())
                            
                            # Corregir orientación si es primera vez
                            if not primer_warp_realizado:
                                print("🔄 PRIMERA VEZ (HOUGH) - Analizando orientación...")
                                
                                # Analizar filas y columnas
                                (usar_filas, diff_filas, diff_columnas, 
                                 necesita_voltear_filas, necesita_voltear_columnas,
                                 energia_filas_sup, energia_filas_inf,
                                 energia_cols_izq, energia_cols_der) = analizar_filas_y_columnas(warped)
                                
                                print(f"   Diferencias - Filas: {diff_filas:.2f}, Columnas: {diff_columnas:.2f}")
                                print(f"   Usando: {'FILAS' if usar_filas else 'COLUMNAS'}")
                                
                                # Corregir orientación
                                warp_corregido, transformaciones = corregir_orientacion_tablero(
                                    warped, usar_filas, necesita_voltear_filas, necesita_voltear_columnas
                                )
                                
                                # Establecer esquinas de referencia
                                esquinas_referencia = np.array([
                                    [0, 0],
                                    [799, 0],
                                    [799, 799],
                                    [0, 799]
                                ], dtype=np.float32)
                                
                                # Ajustar esquinas según transformaciones
                                if 'Transposición (Columnas)' in transformaciones:
                                    esquinas_referencia = np.array([
                                        [0, 799],
                                        [0, 0],
                                        [799, 0],
                                        [799, 799]
                                    ], dtype=np.float32)
                                
                                if 'Volteo Vertical (Filas)' in transformaciones or 'Volteo Vertical' in transformaciones:
                                    esquinas_referencia = np.array([
                                        [0, 799],
                                        [799, 799],
                                        [799, 0],
                                        [0, 0]
                                    ], dtype=np.float32)
                                
                                primer_warp_realizado = True
                                info_orientacion = f"Usando: {'FILAS' if usar_filas else 'COLUMNAS'}"
                                
                                dibujar_orientacion(frame, warp_corregido, esquinas_referencia, info_orientacion, transformaciones)
                            else:
                                # Reordenar puntos según referencia
                                puntos_reordenados = asignar_esquinas_a_referencia(src_points_ordenados, esquinas_referencia)
                                warped, M, puntos_con_offset = aplicar_warp_con_offset(frame, puntos_reordenados, offset_actual)
                                warp_corregido = warped
                                info_orientacion = "Esquinas asignadas por proximidad"
                                dibujar_orientacion(frame, warped, esquinas_referencia, info_orientacion)
                            
                            # Dibujar información de estabilidad
                            dibujar_estabilidad(frame, vertices_actuales, distancias_estabilidad, puntaje_estabilidad, es_estable)
                            
                            ultimos_puntos_ordenados = src_points_ordenados
                            ultimo_frame_warp = frame.copy()
                            ultimo_cuadrilatero_valido = mejor_vertices
                            ultimo_num_puntos = num_puntos
                            ultimo_area = area
                            hough_ejecutado = True
                            ultimo_warp_valido = warp_corregido.copy()
                            ultimo_offset_usado = offset_actual
                            
                            estimated_board_img, warp_result_img = dibujar_cuadrilatero_y_warp(
                                frame, src_points_ordenados, num_puntos, area, offset_actual, 
                                True, False, warp_corregido, puntos_con_offset
                            )
                            
                            cv.putText(estimated_board_img, "HOUGH (1ra vez)", 
                                      (10, 150), cv.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)
                            
                            # Mostrar estado de estabilidad en el warp
                            if es_estable:
                                cv.putText(estimated_board_img, f"ESTABLE (puntaje: {puntaje_estabilidad:.2f})", 
                                          (10, 240), cv.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
                                cv.putText(warp_result_img, f"ESTABLE (puntaje: {puntaje_estabilidad:.2f})", 
                                          (10, 150), cv.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
                            else:
                                cv.putText(estimated_board_img, f"INESTABLE (puntaje: {puntaje_estabilidad:.2f})", 
                                          (10, 240), cv.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2)
                                cv.putText(warp_result_img, f"INESTABLE (puntaje: {puntaje_estabilidad:.2f})", 
                                          (10, 150), cv.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2)
                            
                            print(f"✅ HOUGH exitoso: {num_puntos} puntos, área={area:.0f}")
                        else:
                            # Hough no encontró nada, reciclar si existe
                            if ultimo_cuadrilatero_valido is not None:
                                usando_reciclado = True
                                print(f"♻️ RECICLANDO (Hough sin resultados)")
                                src_points_ordenados = ultimos_puntos_ordenados
                                num_puntos = ultimo_num_puntos
                                area = ultimo_area
                                
                                # Reordenar puntos según referencia si existe
                                if esquinas_referencia is not None:
                                    puntos_reordenados = asignar_esquinas_a_referencia(src_points_ordenados, esquinas_referencia)
                                else:
                                    puntos_reordenados = src_points_ordenados
                                
                                warped, M, puntos_con_offset = aplicar_warp_con_offset(frame, puntos_reordenados, offset_actual)
                                
                                estimated_board_img, warp_result_img = dibujar_cuadrilatero_y_warp(
                                    frame, puntos_reordenados, num_puntos, area, offset_actual, 
                                    False, True, warped, puntos_con_offset
                                )
                                mostrar_mensaje_hough(frame, "RECICLADO (sin Hough)", (0, 255, 255))
                            else:
                                mostrar_mensaje_hough(frame, "No se encontraron cuadriláteros", (0, 0, 255))
                                cv.putText(estimated_board_img, "SIN DETECCION", 
                                          (10, 30), cv.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
                                cv.putText(warp_result_img, "SIN DETECCION", 
                                          (10, 30), cv.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
                    
                    # Si Hough ya se ejecutó, reciclar con tolerancia
                    else:
                        # Contar puntos dentro del último cuadrilátero
                        poly_contour = ultimo_cuadrilatero_valido.astype(np.int32).reshape(-1, 1, 2)
                        puntos_actuales = contar_puntos_silla_en_poligono(poly_contour, saddle_points)
                        
                        if ultimo_num_puntos > 0:
                            perdida = (ultimo_num_puntos - puntos_actuales) / ultimo_num_puntos
                            
                            # Si la pérdida es menor al umbral, reciclar
                            if perdida < recycle_threshold:
                                usando_reciclado = True
                                print(f"♻️ RECICLANDO: {puntos_actuales} pts (pérdida {perdida*100:.1f}% < {recycle_threshold*100:.0f}%)")
                                
                                src_points_ordenados = ultimos_puntos_ordenados
                                num_puntos = puntos_actuales
                                area = ultimo_area
                                
                                # Reordenar puntos según referencia si existe
                                if esquinas_referencia is not None:
                                    puntos_reordenados = asignar_esquinas_a_referencia(src_points_ordenados, esquinas_referencia)
                                else:
                                    puntos_reordenados = src_points_ordenados
                                
                                warped, M, puntos_con_offset = aplicar_warp_con_offset(frame, puntos_reordenados, offset_actual)
                                
                                estimated_board_img, warp_result_img = dibujar_cuadrilatero_y_warp(
                                    frame, puntos_reordenados, num_puntos, area, offset_actual, 
                                    False, True, warped, puntos_con_offset
                                )
                                
                                # Actualizar el número de puntos para el siguiente frame
                                ultimo_num_puntos = puntos_actuales
                                
                                cv.imshow(hough_window, np.zeros_like(frame))
                                cv.imshow(intersecciones_window, np.zeros_like(frame))
                            else:
                                # Pérdida grande, ejecutar Hough de nuevo
                                print(f"🔄 PÉRDIDA GRANDE: {puntos_actuales} pts (pérdida {perdida*100:.1f}% >= {recycle_threshold*100:.0f}%)")
                                hough_ejecutado = False  # Forzar nueva ejecución
                                
                                # Para este frame, reciclamos con pérdida grande
                                usando_reciclado = True
                                src_points_ordenados = ultimos_puntos_ordenados
                                num_puntos = puntos_actuales
                                area = ultimo_area
                                
                                # Reordenar puntos según referencia si existe
                                if esquinas_referencia is not None:
                                    puntos_reordenados = asignar_esquinas_a_referencia(src_points_ordenados, esquinas_referencia)
                                else:
                                    puntos_reordenados = src_points_ordenados
                                
                                warped, M, puntos_con_offset = aplicar_warp_con_offset(frame, puntos_reordenados, offset_actual)
                                
                                estimated_board_img, warp_result_img = dibujar_cuadrilatero_y_warp(
                                    frame, puntos_reordenados, num_puntos, area, offset_actual, 
                                    False, True, warped, puntos_con_offset
                                )
                                
                                cv.putText(estimated_board_img, "RECICLADO (forzando Hough)", 
                                          (10, 150), cv.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2)
                                cv.putText(warp_result_img, "RECICLADO (forzando Hough)", 
                                          (10, 150), cv.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2)
                else:
                    # Sin máscara binaria, reciclar si es posible
                    if ultimo_cuadrilatero_valido is not None:
                        usando_reciclado = True
                        print(f"♻️ RECICLANDO (sin máscara binaria)")
                        src_points_ordenados = ultimos_puntos_ordenados
                        num_puntos = ultimo_num_puntos
                        area = ultimo_area
                        
                        # Reordenar puntos según referencia si existe
                        if esquinas_referencia is not None:
                            puntos_reordenados = asignar_esquinas_a_referencia(src_points_ordenados, esquinas_referencia)
                        else:
                            puntos_reordenados = src_points_ordenados
                        
                        warped, M, puntos_con_offset = aplicar_warp_con_offset(frame, puntos_reordenados, offset_actual)
                        
                        estimated_board_img, warp_result_img = dibujar_cuadrilatero_y_warp(
                            frame, puntos_reordenados, num_puntos, area, offset_actual, 
                            False, True, warped, puntos_con_offset
                        )
                        mostrar_mensaje_hough(frame, "RECICLADO (sin máscara)", (0, 255, 255))
                    else:
                        cv.putText(estimated_board_img, "SIN DETECCION", 
                                  (10, 30), cv.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
                        cv.putText(warp_result_img, "SIN DETECCION", 
                                  (10, 30), cv.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
                        mostrar_mensaje_hough(frame, "SIN DETECCION", (0, 0, 255))
            
            cv.imshow(estimated_board_window, estimated_board_img)
            cv.imshow(warp_window, warp_result_img)
            cv.imshow(window_name, frame_vis)

            last_frame = current_frame

    key = cv.waitKey(20) & 0xFF

    if key == 27:
        break

cap.release()
cv.destroyAllWindows()

Frames totales: 430
FPS: 29.97
✅ CUADRILÁTERO DETECTADO: 170 puntos silla
   Estabilidad: PRIMER FRAME (sin referencia)
🔄 PRIMERA VEZ - Analizando orientación...
   Diferencias - Filas: 18.09, Columnas: 79.77
   Usando: COLUMNAS
   Energía filas - Superior: 140.05, Inferior: 158.14
   Energía columnas - Izquierda: 181.26, Derecha: 101.49
   ✓ Aplicada transposición basada en columnas
   ✓ Aplicado volteo vertical después de transposición
✅ Orientación corregida: Usando: COLUMNAS
✅ CUADRILÁTERO DETECTADO: 168 puntos silla
   Estabilidad: ESTABLE (puntaje: 1.00)
🔄 Asignando esquinas por proximidad...
✅ Esquinas asignadas
✅ CUADRILÁTERO DETECTADO: 170 puntos silla
   Estabilidad: ESTABLE (puntaje: 1.00)
🔄 Asignando esquinas por proximidad...
✅ Esquinas asignadas
♻️ RECICLANDO (sin máscara binaria)
✅ CUADRILÁTERO DETECTADO: 170 puntos silla
   Estabilidad: ESTABLE (puntaje: 1.00)
🔄 Asignando esquinas por proximidad...
✅ Esquinas asignadas
✅ CUADRILÁTERO DETECTADO: 169 puntos silla
   Estab

In [44]:
# PRUEBA CON WEBCAM CON HOUGH PROBABILÍSTICO CON ORIENTACIÓN CORRECTA
# CON VISUALIZACIÓN DE ENERGÍA DE FILAS Y COLUMNAS - CORREGIDO ESTABILIDAD

import cv2 as cv
import numpy as np
from collections import deque
import random
from copy import copy
from numpy.random import default_rng
import itertools
import math

rng = default_rng()

# Usar webcam (0 es la cámara predeterminada)
CAMERA_ID = 0
cap = cv.VideoCapture(CAMERA_ID)

if not cap.isOpened():
    raise IOError(f"No se pudo abrir la cámara {CAMERA_ID}")

# Obtener información de la cámara
frame_width = int(cap.get(cv.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv.CAP_PROP_FPS)

print(f"Resolución: {frame_width}x{frame_height}")
print(f"FPS: {fps:.2f}")

# Crear ventanas
window_name = "Video"
sobel_window = "Sobel"
canny_window = "Canny"
canny_dilated_window = "Canny Dilatado"
hough_filtered_window = "Saddle Points"
contour_window = "Contornos"
approx_window = "Polígonos aproximados"
quadrilaterals_window = "Cuadriláteros detectados"
estimated_board_window = "Cuadrilátero con más puntos silla"
warp_window = "Warp final"
controls_window = "Controles"
hough_window = "HoughP - Líneas detectadas"
polygons_binary_window = "Máscara binaria de polígonos"
intersecciones_window = "Intersecciones - Puntos y Cuadriláteros"
orientation_window = "Orientación del Tablero"
stability_window = "Estabilidad de Vértices"
energy_window = "Energía de Filas y Columnas"

cv.namedWindow(window_name, cv.WINDOW_NORMAL)
cv.namedWindow(sobel_window, cv.WINDOW_NORMAL)
cv.namedWindow(canny_window, cv.WINDOW_NORMAL)
cv.namedWindow(canny_dilated_window, cv.WINDOW_NORMAL)
cv.namedWindow(hough_filtered_window, cv.WINDOW_NORMAL)
cv.namedWindow(contour_window, cv.WINDOW_NORMAL)
cv.namedWindow(approx_window, cv.WINDOW_NORMAL)
cv.namedWindow(quadrilaterals_window, cv.WINDOW_NORMAL)
cv.namedWindow(estimated_board_window, cv.WINDOW_NORMAL)
cv.namedWindow(warp_window, cv.WINDOW_NORMAL)
cv.namedWindow(controls_window, cv.WINDOW_NORMAL)
cv.namedWindow(hough_window, cv.WINDOW_NORMAL)
cv.namedWindow(polygons_binary_window, cv.WINDOW_NORMAL)
cv.namedWindow(intersecciones_window, cv.WINDOW_NORMAL)
cv.namedWindow(orientation_window, cv.WINDOW_NORMAL)
cv.namedWindow(stability_window, cv.WINDOW_NORMAL)
cv.namedWindow(energy_window, cv.WINDOW_NORMAL)

# Variable para el offset
offset_pixels = 0

def on_offset_trackbar(pos):
    global offset_pixels
    offset_pixels = pos

# Crear trackbar para el offset (0-50 píxeles)
cv.createTrackbar(
    "Offset",
    controls_window,
    15,
    50,
    on_offset_trackbar
)

# Variables para HoughP
HOUGH_DISTANCE_RESOLUTION = 1  # Resolución de distancia en píxeles
HOUGH_ANGLE_RESOLUTION = np.pi / 360  # Resolución angular en radianes
HOUGH_THRESHOLD = 50  # Umbral de acumulador
HOUGH_MIN_LINE_LENGTH = 50  # Longitud mínima de línea
HOUGH_MAX_LINE_GAP = 10  # Gap máximo entre segmentos

# Porcentaje de tolerancia para reciclaje (10%)
RECYCLE_THRESHOLD_PERCENT = 10

def on_hough_threshold_trackbar(pos):
    global HOUGH_THRESHOLD
    HOUGH_THRESHOLD = max(1, pos)

cv.createTrackbar(
    "Umbral Hough",
    controls_window,
    50,
    200,
    on_hough_threshold_trackbar
)

def on_hough_min_length_trackbar(pos):
    global HOUGH_MIN_LINE_LENGTH
    HOUGH_MIN_LINE_LENGTH = max(5, pos)

cv.createTrackbar(
    "Longitud mínima",
    controls_window,
    50,
    200,
    on_hough_min_length_trackbar
)

def on_hough_max_gap_trackbar(pos):
    global HOUGH_MAX_LINE_GAP
    HOUGH_MAX_LINE_GAP = max(1, pos)

cv.createTrackbar(
    "Gap máximo",
    controls_window,
    40,
    100,
    on_hough_max_gap_trackbar
)

def on_recycle_threshold_trackbar(pos):
    global RECYCLE_THRESHOLD_PERCENT
    RECYCLE_THRESHOLD_PERCENT = max(1, pos)

cv.createTrackbar(
    "Tolerancia Reciclaje %",
    controls_window,
    10,
    100,
    on_recycle_threshold_trackbar
)

# Parámetros de estabilidad - AHORA SOLO PARA ORIENTACIÓN
STABILITY_CHECK_FRAMES = 5  # Número de frames a verificar
STABILITY_RADIUS_PX = 30  # Radio máximo de movimiento permitido
PUNTAJE_MINIMO_ESTABILIDAD = 0.6  # Puntaje mínimo para considerar estable

stability_buffer = deque(maxlen=STABILITY_CHECK_FRAMES)  # Buffer para almacenar vértices recientes

# Parámetros
TOLERANCE = 5
MIN_AREA = 10
MIN_PUNTOS_SILLA = 50

# Variables para almacenar el último cuadrilátero válido
ultimo_cuadrilatero_valido = None
ultimo_warp_valido = None
ultimo_frame_warp = None
ultimos_puntos_ordenados = None
ultimo_num_puntos = 0
ultimo_area = 0

# Flag para saber si ya se ejecutó Hough
hough_ejecutado = False

# Variables para orientación del tablero - GLOBALES
orientacion_definida = False  # Flag para saber si ya se definió la orientación
esquinas_categorizadas = None  # Almacena las esquinas categorizadas: [TL, TR, BR, BL]
warp_corregido_guardado = None  # Almacena el warp corregido
vertices_fijos = None  # Almacena los vértices fijos para orientación [TL, TR, BR, BL]
ultimo_warp_sin_corregir = None  # Almacena el warp sin corregir (para orientación)

# Variables para almacenar las transformaciones de orientación
transformaciones_guardadas = None
esquinas_categorizadas_guardadas = None

def get_vertices_as_points(polygon):
    vertices = []
    for point in polygon:
        vertices.append((int(point[0][0]), int(point[0][1])))
    return vertices

def apply_warp(frame, src_points, dst_size=(800, 800)):
    dst_points = np.array([
        [0, 0],
        [dst_size[0]-1, 0],
        [dst_size[0]-1, dst_size[1]-1],
        [0, dst_size[1]-1]
    ], dtype=np.float32)
    
    src_points = np.array(src_points, dtype=np.float32)
    M = cv.getPerspectiveTransform(src_points, dst_points)
    warped = cv.warpPerspective(frame, M, dst_size)
    return warped, M

def aplicar_offset_a_puntos(puntos, offset, centro):
    if offset == 0:
        return puntos
    
    puntos_con_offset = []
    for punto in puntos:
        vector = centro - punto
        norm = np.linalg.norm(vector)
        if norm > 0:
            nuevo_punto = punto + (vector / norm) * offset
        else:
            nuevo_punto = punto
        puntos_con_offset.append(nuevo_punto)
    
    return np.array(puntos_con_offset, dtype=np.float32)

def aplicar_warp_con_offset(frame, src_points, offset_px, dst_size=(800, 800)):
    """
    Aplica warp con offset a los puntos
    """
    # Calcular centro
    centro = np.mean(src_points, axis=0)
    
    # Aplicar offset si es necesario
    if offset_px > 0:
        puntos_offset = aplicar_offset_a_puntos(src_points, offset_px, centro)
    else:
        puntos_offset = src_points
    
    # Aplicar warp
    warped, M = apply_warp(frame, puntos_offset, dst_size)
    
    return warped, M, puntos_offset

# ========== FUNCIONES PARA VERIFICACIÓN DE ESTABILIDAD (SOLO PARA ORIENTACIÓN) ==========

def verificar_estabilidad_vertices(vertices_actuales, buffer_vertices, radio_maximo):
    """
    Verifica si los vértices actuales son estables comparándolos con los vértices anteriores
    Retorna: (es_estable, puntaje_estabilidad, distancias_por_vertice)
    """
    if len(buffer_vertices) < STABILITY_CHECK_FRAMES:
        # No hay suficientes datos para verificar
        return True, 1.0, [0.0, 0.0, 0.0, 0.0]
    
    # Convertir a arrays numpy
    vertices_actuales = np.array(vertices_actuales)
    
    # Calcular la distancia promedio de cada vértice con respecto a los frames anteriores
    distancias_por_vertice = []
    
    for i in range(4):  # 4 vértices
        distancias = []
        for vertices_anteriores in buffer_vertices:
            vertices_anteriores = np.array(vertices_anteriores)
            dist = np.linalg.norm(vertices_actuales[i] - vertices_anteriores[i])
            distancias.append(dist)
        
        # Promedio de distancias para este vértice
        dist_promedio = np.mean(distancias)
        distancias_por_vertice.append(dist_promedio)
    
    # Verificar si todos los vértices están dentro del radio máximo
    todos_dentro = all(dist < radio_maximo for dist in distancias_por_vertice)
    
    # Calcular puntaje de estabilidad (0-1)
    # 1 = perfectamente estable, 0 = inestable
    puntaje = 1.0 - (np.mean(distancias_por_vertice) / radio_maximo)
    puntaje = max(0, min(1, puntaje))
    
    return todos_dentro, puntaje, distancias_por_vertice

def dibujar_estabilidad(frame, vertices_actuales, distancias, puntaje, es_estable, orientacion_definida):
    """
    Dibuja la información de estabilidad en el frame
    """
    img = frame.copy()
    h, w = img.shape[:2]
    
    # Mostrar información de estabilidad
    y_offset = 30
    cv.putText(img, "ESTABILIDAD DE VERTICES (para orientacion)", (10, y_offset), 
              cv.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)
    y_offset += 30
    
    # Mostrar puntaje
    color = (0, 255, 0) if es_estable else (0, 0, 255)
    cv.putText(img, f"Puntaje: {puntaje:.2f}", (10, y_offset), 
              cv.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
    y_offset += 30
    
    # Mostrar estado de orientación
    if orientacion_definida:
        estado = "ORIENTACION DEFINIDA ✓"
        color_estado = (0, 255, 0)
    elif es_estable:
        estado = "ESTABLE (definiendo orientacion...)"
        color_estado = (0, 255, 255)
    else:
        estado = "INESTABLE ✗"
        color_estado = (0, 0, 255)
    
    cv.putText(img, f"Estado: {estado}", (10, y_offset), 
              cv.FONT_HERSHEY_SIMPLEX, 0.6, color_estado, 2)
    y_offset += 30
    
    # Mostrar progreso de estabilización
    cv.putText(img, f"Frames en buffer: {len(stability_buffer)}/{STABILITY_CHECK_FRAMES}", 
              (10, y_offset), cv.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
    y_offset += 30
    
    # Mostrar distancias por vértice
    cv.putText(img, "Distancias por vertice:", (10, y_offset), 
              cv.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
    y_offset += 25
    
    for i, dist in enumerate(distancias):
        etiquetas = ["TL", "TR", "BR", "BL"]
        color_vertice = (0, 255, 0) if dist < STABILITY_RADIUS_PX else (0, 0, 255)
        cv.putText(img, f"  {etiquetas[i]}: {dist:.1f}px", (10, y_offset), 
                  cv.FONT_HERSHEY_SIMPLEX, 0.5, color_vertice, 1)
        y_offset += 20
    
    # Dibujar los vértices con colores según estabilidad
    if vertices_actuales is not None:
        for i, punto in enumerate(vertices_actuales):
            x, y = int(punto[0]), int(punto[1])
            if orientacion_definida:
                cv.circle(img, (x, y), 10, (0, 255, 0), -1)
            elif i < len(distancias) and distancias[i] < STABILITY_RADIUS_PX:
                cv.circle(img, (x, y), 10, (0, 255, 0), -1)
            else:
                cv.circle(img, (x, y), 10, (0, 0, 255), -1)
            etiquetas = ["TL", "TR", "BR", "BL"]
            cv.putText(img, etiquetas[i], (x-15, y-15), 
                      cv.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2)
    
    cv.imshow(stability_window, img)
    return img

# ========== FIN FUNCIONES DE ESTABILIDAD ==========

# ========== FUNCIONES PARA ORIENTACIÓN DEL TABLERO ==========

def calcular_energia_region(imagen, y1, y2, x1, x2):
    """
    Calcula la energía promedio de una región de la imagen
    """
    region = imagen[y1:y2, x1:x2]
    if len(region.shape) == 3:
        energia = np.mean(cv.cvtColor(region, cv.COLOR_BGR2GRAY))
    else:
        energia = np.mean(region)
    return energia

def dibujar_energia_filas_columnas(warped_img):
    """
    Dibuja la visualización de energía de filas y columnas
    """
    img = warped_img.copy()
    h, w = img.shape[:2]
    cell_h = h // 8
    cell_w = w // 8
    
    # Crear imagen para mostrar energía
    energy_img = np.zeros((h, w, 3), dtype=np.uint8)
    
    # Calcular energía de cada fila y columna
    energias_filas = []
    energias_columnas = []
    
    for row in range(8):
        y1 = row * cell_h
        y2 = (row + 1) * cell_h
        energia = calcular_energia_region(img, y1, y2, 0, w)
        energias_filas.append(energia)
    
    for col in range(8):
        x1 = col * cell_w
        x2 = (col + 1) * cell_w
        energia = calcular_energia_region(img, 0, h, x1, x2)
        energias_columnas.append(energia)
    
    # Normalizar para visualización
    max_energia_filas = max(energias_filas) if energias_filas else 1
    max_energia_columnas = max(energias_columnas) if energias_columnas else 1
    
    # Dibujar barras de energía para filas (a la izquierda)
    for i, energia in enumerate(energias_filas):
        y1 = i * cell_h
        y2 = (i + 1) * cell_h
        # Barra horizontal que representa la energía
        bar_width = int((energia / max_energia_filas) * 50)
        cv.rectangle(energy_img, (0, y1), (bar_width, y2), (0, 255, 0), -1)
        # Texto con el valor
        cv.putText(energy_img, f"{energia:.1f}", (bar_width + 5, y1 + cell_h//2), 
                  cv.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 255), 1)
    
    # Dibujar barras de energía para columnas (en la parte inferior)
    for i, energia in enumerate(energias_columnas):
        x1 = i * cell_w
        x2 = (i + 1) * cell_w
        bar_height = int((energia / max_energia_columnas) * 50)
        cv.rectangle(energy_img, (x1, h - bar_height), (x2, h), (0, 0, 255), -1)
        # Texto con el valor
        cv.putText(energy_img, f"{energia:.1f}", (x1 + cell_w//4, h - bar_height - 5), 
                  cv.FONT_HERSHEY_SIMPLEX, 0.3, (255, 255, 255), 1)
    
    # Marcar las filas y columnas de interés (primeras 2 y últimas 2)
    # Filas superiores (0,1)
    for row in range(2):
        y1 = row * cell_h
        y2 = (row + 1) * cell_h
        cv.rectangle(energy_img, (0, y1), (w, y2), (255, 255, 0), 2)
    # Filas inferiores (6,7)
    for row in range(6, 8):
        y1 = row * cell_h
        y2 = (row + 1) * cell_h
        cv.rectangle(energy_img, (0, y1), (w, y2), (255, 0, 255), 2)
    # Columnas izquierdas (0,1)
    for col in range(2):
        x1 = col * cell_w
        x2 = (col + 1) * cell_w
        cv.rectangle(energy_img, (x1, 0), (x2, h), (0, 255, 255), 2)
    # Columnas derechas (6,7)
    for col in range(6, 8):
        x1 = col * cell_w
        x2 = (col + 1) * cell_w
        cv.rectangle(energy_img, (x1, 0), (x2, h), (255, 0, 0), 2)
    
    # Texto de información
    cv.putText(energy_img, "FILAS (verde)", (60, 20), 
              cv.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)
    cv.putText(energy_img, "COLUMNAS (rojo)", (60, 40), 
              cv.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1)
    
    # Mostrar estadísticas
    energia_filas_sup = (energias_filas[0] + energias_filas[1]) / 2
    energia_filas_inf = (energias_filas[6] + energias_filas[7]) / 2
    energia_cols_izq = (energias_columnas[0] + energias_columnas[1]) / 2
    energia_cols_der = (energias_columnas[6] + energias_columnas[7]) / 2
    
    info_text = f"Filas Sup: {energia_filas_sup:.1f} | Filas Inf: {energia_filas_inf:.1f} | Diff: {abs(energia_filas_sup - energia_filas_inf):.1f}"
    cv.putText(energy_img, info_text, (10, h - 80), 
              cv.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 0), 1)
    
    info_text2 = f"Cols Izq: {energia_cols_izq:.1f} | Cols Der: {energia_cols_der:.1f} | Diff: {abs(energia_cols_izq - energia_cols_der):.1f}"
    cv.putText(energy_img, info_text2, (10, h - 60), 
              cv.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255), 1)
    
    # Superponer la imagen original en la parte derecha
    img_small = cv.resize(img, (w//2, h//2))
    energy_img[0:h//2, w//2:w] = img_small
    
    cv.imshow(energy_window, energy_img)
    
    # Retornar energías para uso posterior
    return {
        'filas_sup': energia_filas_sup,
        'filas_inf': energia_filas_inf,
        'cols_izq': energia_cols_izq,
        'cols_der': energia_cols_der,
        'diff_filas': abs(energia_filas_sup - energia_filas_inf),
        'diff_columnas': abs(energia_cols_izq - energia_cols_der),
        'energias_filas': energias_filas,
        'energias_columnas': energias_columnas
    }

def analizar_y_corregir_orientacion(warped_img):
    """
    Analiza la imagen warpeada y corrige la orientación para que las negras estén arriba y blancas abajo
    Retorna: (imagen_corregida, transformaciones_aplicadas, esquinas_categorizadas, datos_energia)
    """
    img = warped_img.copy()
    h, w = img.shape[:2]
    cell_h = h // 8
    cell_w = w // 8
    
    transformaciones = []
    
    # === PASO 1: Analizar filas y columnas ===
    # Energía de las primeras 2 filas
    energia_filas_sup = 0
    for row in range(2):
        y1 = row * cell_h
        y2 = (row + 1) * cell_h
        energia_filas_sup += calcular_energia_region(img, y1, y2, 0, w)
    energia_filas_sup /= 2
    
    # Energía de las últimas 2 filas
    energia_filas_inf = 0
    for row in range(6, 8):
        y1 = row * cell_h
        y2 = (row + 1) * cell_h
        energia_filas_inf += calcular_energia_region(img, y1, y2, 0, w)
    energia_filas_inf /= 2
    
    diff_filas = abs(energia_filas_sup - energia_filas_inf)
    
    # Energía de las primeras 2 columnas
    energia_cols_izq = 0
    for col in range(2):
        x1 = col * cell_w
        x2 = (col + 1) * cell_w
        energia_cols_izq += calcular_energia_region(img, 0, h, x1, x2)
    energia_cols_izq /= 2
    
    # Energía de las últimas 2 columnas
    energia_cols_der = 0
    for col in range(6, 8):
        x1 = col * cell_w
        x2 = (col + 1) * cell_w
        energia_cols_der += calcular_energia_region(img, 0, h, x1, x2)
    energia_cols_der /= 2
    
    diff_columnas = abs(energia_cols_izq - energia_cols_der)
    
    datos_energia = {
        'filas_sup': energia_filas_sup,
        'filas_inf': energia_filas_inf,
        'cols_izq': energia_cols_izq,
        'cols_der': energia_cols_der,
        'diff_filas': diff_filas,
        'diff_columnas': diff_columnas
    }
    
    print(f"   Diferencias - Filas: {diff_filas:.2f}, Columnas: {diff_columnas:.2f}")
    print(f"   Energía filas - Superior: {energia_filas_sup:.2f}, Inferior: {energia_filas_inf:.2f}")
    print(f"   Energía columnas - Izquierda: {energia_cols_izq:.2f}, Derecha: {energia_cols_der:.2f}")
    
    # === PASO 2: Decidir si trasponer (rotar 90°) ===
    # Si las columnas tienen mayor diferencia, trasponer
    if diff_columnas > diff_filas:
        img = cv.transpose(img)
        transformaciones.append("Transposición (90°)")
        print("   ✓ Aplicada transposición (90°) - Columnas tienen mayor diferencia")
        # Recalcular energías después de transponer
        h, w = img.shape[:2]
        cell_h = h // 8
        cell_w = w // 8
        
        # Recalcular energías de filas
        energia_filas_sup = 0
        for row in range(2):
            y1 = row * cell_h
            y2 = (row + 1) * cell_h
            energia_filas_sup += calcular_energia_region(img, y1, y2, 0, w)
        energia_filas_sup /= 2
        
        energia_filas_inf = 0
        for row in range(6, 8):
            y1 = row * cell_h
            y2 = (row + 1) * cell_h
            energia_filas_inf += calcular_energia_region(img, y1, y2, 0, w)
        energia_filas_inf /= 2
        
        print(f"   Nueva energía filas - Superior: {energia_filas_sup:.2f}, Inferior: {energia_filas_inf:.2f}")
        
        # Actualizar datos de energía
        datos_energia['filas_sup'] = energia_filas_sup
        datos_energia['filas_inf'] = energia_filas_inf
    else:
        print("   ✓ No se requiere transposición - Filas tienen mayor diferencia")
    
    # === PASO 3: Verificar orientación vertical (negras arriba, blancas abajo) ===
    # Las negras tienen menor energía, las blancas mayor energía
    # Si las filas superiores tienen más energía que las inferiores, voltear 180°
    if energia_filas_sup > energia_filas_inf:
        img = cv.flip(img, 0)  # Volteo vertical
        transformaciones.append("Volteo Vertical (180°)")
        print("   ✓ Aplicado volteo vertical (180°) - Superiores tienen más energía")
    else:
        print("   ✓ Orientación vertical correcta - Inferiores tienen más energía")
    
    # === PASO 4: Categorizar esquinas ===
    # Después de todas las transformaciones, las esquinas en la imagen warpeada son:
    # TL = (0,0), TR = (w-1,0), BR = (w-1,h-1), BL = (0,h-1)
    esquinas_categorizadas = np.array([
        [0, 0],
        [w-1, 0],
        [w-1, h-1],
        [0, h-1]
    ], dtype=np.float32)
    
    # Si hubo transposición, las esquinas cambian de orden
    if 'Transposición (90°)' in transformaciones:
        esquinas_categorizadas = np.array([
            [0, h-1],
            [0, 0],
            [w-1, 0],
            [w-1, h-1]
        ], dtype=np.float32)
    
    # Si hubo volteo vertical, las esquinas cambian de orden
    if 'Volteo Vertical (180°)' in transformaciones:
        esquinas_categorizadas = np.array([
            [0, h-1],
            [w-1, h-1],
            [w-1, 0],
            [0, 0]
        ], dtype=np.float32)
    
    return img, transformaciones, esquinas_categorizadas, datos_energia

def dibujar_orientacion(frame, info_text, transformaciones=None):
    """
    Dibuja información de orientación en el frame
    """
    img = frame.copy()
    h, w = img.shape[:2]
    
    # Mostrar información
    y_offset = 30
    cv.putText(img, "ORIENTACION DEL TABLERO", (10, y_offset), 
              cv.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
    y_offset += 30
    
    cv.putText(img, info_text, (10, y_offset), 
              cv.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
    y_offset += 25
    
    if transformaciones:
        for trans in transformaciones:
            cv.putText(img, f"  -> {trans}", (10, y_offset), 
                      cv.FONT_HERSHEY_SIMPLEX, 0.4, (0, 255, 0), 1)
            y_offset += 20
    
    cv.imshow(orientation_window, img)
    return img

# ========== FIN FUNCIONES ORIENTACIÓN ==========

# ========== FUNCIONES PARA HOUGH PROBABILÍSTICO ==========

def obtener_rectas_hough(imagen_binaria, threshold, min_line_length, max_line_gap):
    """
    Obtiene líneas usando Hough Probabilístico
    Retorna una lista de líneas en formato polar (rho, theta)
    """
    if imagen_binaria is None:
        return []
    
    # Aplicar HoughP
    lines = cv.HoughLinesP(
        imagen_binaria,
        HOUGH_DISTANCE_RESOLUTION,
        HOUGH_ANGLE_RESOLUTION,
        threshold,
        minLineLength=min_line_length,
        maxLineGap=max_line_gap
    )
    
    if lines is None:
        return []
    
    # Convertir a formato polar
    lineas_polar = []
    for line in lines:
        x1, y1, x2, y2 = line[0]
        
        # Calcular parámetros polares (rho, theta)
        dx = x2 - x1
        dy = y2 - y1
        
        # Calcular theta (ángulo de la normal)
        if abs(dx) < 1e-6:
            theta = np.pi / 2 if dy > 0 else -np.pi / 2
        else:
            theta = np.arctan2(dx, -dy)
        
        # Calcular rho
        rho = x1 * np.cos(theta) + y1 * np.sin(theta)
        
        # Calcular orientación
        theta_normalized = theta % np.pi
        es_horizontal = theta_normalized < np.pi/4 or theta_normalized > 3*np.pi/4
        
        lineas_polar.append({
            'rho': rho,
            'theta': theta,
            'puntos': [(x1, y1), (x2, y2)],
            'origen': 'horizontal' if es_horizontal else 'vertical'
        })
    
    return lineas_polar

def agrupar_lineas_por_orientacion(lineas):
    """
    Agrupa las líneas por orientación (horizontal/vertical)
    """
    horizontales = []
    verticales = []
    
    for linea in lineas:
        if linea['origen'] == 'horizontal':
            horizontales.append(linea)
        else:
            verticales.append(linea)
    
    return horizontales, verticales

def fusionar_lineas_cercanas(lineas, umbral_rho=20, umbral_theta=0.1):
    """
    Fusiona líneas que son cercanas entre sí
    """
    if len(lineas) < 2:
        return lineas
    
    fusionadas = []
    usadas = [False] * len(lineas)
    
    for i in range(len(lineas)):
        if usadas[i]:
            continue
        
        grupo = [i]
        usadas[i] = True
        
        for j in range(i + 1, len(lineas)):
            if usadas[j]:
                continue
            
            rho1, theta1 = lineas[i]['rho'], lineas[i]['theta']
            rho2, theta2 = lineas[j]['rho'], lineas[j]['theta']
            
            # Normalizar theta
            theta1_norm = theta1 % np.pi
            theta2_norm = theta2 % np.pi
            
            diff_rho = abs(rho1 - rho2)
            diff_theta = abs(theta1_norm - theta2_norm)
            diff_theta = min(diff_theta, np.pi - diff_theta)
            
            if diff_rho < umbral_rho and diff_theta < umbral_theta:
                grupo.append(j)
                usadas[j] = True
        
        # Promediar las líneas del grupo
        if len(grupo) > 0:
            rho_prom = np.mean([lineas[idx]['rho'] for idx in grupo])
            theta_prom = np.mean([lineas[idx]['theta'] for idx in grupo])
            
            # Usar los puntos de la línea con más inliers (si existiera)
            # O simplemente usar el primero
            fusionadas.append({
                'rho': rho_prom,
                'theta': theta_prom,
                'puntos': lineas[grupo[0]]['puntos'],
                'origen': lineas[grupo[0]]['origen']
            })
    
    return fusionadas

def recta_polar_a_puntos(recta_polar, ancho, alto):
    """
    Convierte una recta en representación polar (rho, theta) a dos puntos
    para dibujarla en una imagen de dimensiones (ancho, alto)
    """
    rho, theta = recta_polar
    a = np.cos(theta)
    b = np.sin(theta)
    
    # Calcular puntos extremos
    if abs(a) > 0.01:  # No es vertical
        x1 = 0
        y1 = int((rho - x1 * a) / b) if abs(b) > 1e-6 else 0
        x2 = ancho
        y2 = int((rho - x2 * a) / b) if abs(b) > 1e-6 else 0
    else:  # Línea vertical
        y1 = 0
        x1 = int(rho / a) if abs(a) > 1e-6 else 0
        y2 = alto
        x2 = int(rho / a) if abs(a) > 1e-6 else 0
    
    # Asegurar que los puntos estén dentro de la imagen
    x1 = max(0, min(ancho, x1))
    x2 = max(0, min(ancho, x2))
    y1 = max(0, min(alto, y1))
    y2 = max(0, min(alto, y2))
    
    return (x1, y1), (x2, y2)

def calcular_interseccion_polar(recta1, recta2):
    """
    Calcula la intersección de dos rectas en representación polar (rho, theta)
    Retorna el punto (x, y) o None si son paralelas
    """
    rho1, theta1 = recta1
    rho2, theta2 = recta2
    
    # Si las rectas son paralelas (theta iguales o diferencia de 180 grados)
    if abs(theta1 - theta2) < 1e-6 or abs(abs(theta1 - theta2) - np.pi) < 1e-6:
        return None
    
    # Resolver sistema de ecuaciones:
    # x*cos(theta1) + y*sin(theta1) = rho1
    # x*cos(theta2) + y*sin(theta2) = rho2
    
    A = np.array([[np.cos(theta1), np.sin(theta1)],
                  [np.cos(theta2), np.sin(theta2)]])
    b = np.array([rho1, rho2])
    
    try:
        x, y = np.linalg.solve(A, b)
        return (int(x), int(y))
    except np.linalg.LinAlgError:
        return None

def formar_cuadrilateros_desde_rectas(rectas_horizontal, rectas_vertical, saddle_points, frame_shape):
    """
    Forma cuadriláteros a partir de las intersecciones de rectas horizontales y verticales
    """
    h, w = frame_shape[:2]
    cuadrilateros = []
    
    # Tomar las mejores líneas (más largas/confiables)
    # Ordenar por longitud de segmento
    def longitud_linea(linea):
        p1, p2 = linea['puntos']
        return np.sqrt((p2[0]-p1[0])**2 + (p2[1]-p1[1])**2)
    
    rectas_horizontal.sort(key=longitud_linea, reverse=True)
    rectas_vertical.sort(key=longitud_linea, reverse=True)
    
    # Tomar las primeras N líneas
    max_lineas = min(8, len(rectas_horizontal))
    rectas_horizontal = rectas_horizontal[:max_lineas]
    max_lineas = min(8, len(rectas_vertical))
    rectas_vertical = rectas_vertical[:max_lineas]
    
    for combo_h in itertools.combinations(range(len(rectas_horizontal)), 2):
        for combo_v in itertools.combinations(range(len(rectas_vertical)), 2):
            h1 = rectas_horizontal[combo_h[0]]
            h2 = rectas_horizontal[combo_h[1]]
            v1 = rectas_vertical[combo_v[0]]
            v2 = rectas_vertical[combo_v[1]]
            
            # Obtener parámetros polares
            h1_recta = (h1['rho'], h1['theta'])
            h2_recta = (h2['rho'], h2['theta'])
            v1_recta = (v1['rho'], v1['theta'])
            v2_recta = (v2['rho'], v2['theta'])
            
            # Calcular intersecciones
            inter1 = calcular_interseccion_polar(h1_recta, v1_recta)
            inter2 = calcular_interseccion_polar(h1_recta, v2_recta)
            inter3 = calcular_interseccion_polar(h2_recta, v2_recta)
            inter4 = calcular_interseccion_polar(h2_recta, v1_recta)
            
            if None in [inter1, inter2, inter3, inter4]:
                continue
            
            vertices = [inter1, inter2, inter3, inter4]
            
            # Verificar que los vértices estén dentro de la imagen
            dentro = True
            for v in vertices:
                if v[0] < 0 or v[0] >= w or v[1] < 0 or v[1] >= h:
                    dentro = False
                    break
            
            if not dentro:
                continue
            
            vertices_array = np.array(vertices, dtype=np.float32)
            vertices_ordenados = ordenar_vertices_horario(vertices_array)
            
            if vertices_ordenados is None:
                continue
            
            area = cv.contourArea(vertices_ordenados.astype(np.int32))
            if area < 100:
                continue
            
            poly_contour = vertices_ordenados.astype(np.int32).reshape(-1, 1, 2)
            if not cv.isContourConvex(poly_contour):
                continue
            
            num_puntos = contar_puntos_silla_en_poligono(poly_contour, saddle_points)
            
            if num_puntos >= MIN_PUNTOS_SILLA:
                ratio = num_puntos**2 / np.sqrt(area) if area > 0 else 0
                cuadrilateros.append({
                    'vertices': vertices_ordenados,
                    'num_puntos': num_puntos,
                    'area': area,
                    'ratio': ratio,
                    'h1': h1,
                    'h2': h2,
                    'v1': v1,
                    'v2': v2
                })
    
    cuadrilateros.sort(key=lambda x: x['ratio'], reverse=True)
    return cuadrilateros

def dibujar_intersecciones_y_cuadrilateros(frame, rectas_horizontal, rectas_vertical, saddle_points, cuadrilateros):
    """
    Dibuja las rectas, intersecciones y cuadriláteros
    """
    img = frame.copy()
    h, w = img.shape[:2]
    
    for punto in saddle_points:
        cv.circle(img, (int(punto[0]), int(punto[1])), 2, (0, 255, 255), -1)
    
    # Dibujar rectas horizontales
    for recta in rectas_horizontal:
        rho, theta = recta['rho'], recta['theta']
        p1, p2 = recta_polar_a_puntos((rho, theta), w, h)
        cv.line(img, p1, p2, (255, 150, 0), 1)
    
    # Dibujar rectas verticales
    for recta in rectas_vertical:
        rho, theta = recta['rho'], recta['theta']
        p1, p2 = recta_polar_a_puntos((rho, theta), w, h)
        cv.line(img, p1, p2, (0, 200, 100), 1)
    
    # Dibujar intersecciones
    for h_rect in rectas_horizontal:
        h_polar = (h_rect['rho'], h_rect['theta'])
        for v_rect in rectas_vertical:
            v_polar = (v_rect['rho'], v_rect['theta'])
            inter = calcular_interseccion_polar(h_polar, v_polar)
            if inter is not None:
                x, y = inter
                if 0 <= x < w and 0 <= y < h:
                    cv.circle(img, (x, y), 4, (0, 255, 0), -1)
    
    y_offset = 30
    cv.putText(img, f"Cuadrilateros validos (min {MIN_PUNTOS_SILLA} pts): {len(cuadrilateros)}", 
              (10, y_offset), cv.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
    y_offset += 25
    
    colores = [
        (255, 0, 0), (0, 255, 0), (0, 0, 255), (255, 255, 0),
        (255, 0, 255), (0, 255, 255), (128, 0, 255), (255, 128, 0)
    ]
    
    for idx, cuad in enumerate(cuadrilateros[:8]):
        vertices = cuad['vertices']
        color = colores[idx % len(colores)]
        pts = vertices.astype(np.int32)
        cv.polylines(img, [pts], True, color, 2)
        
        centro = np.mean(vertices, axis=0).astype(int)
        cv.putText(img, f"{cuad['ratio']:.4f}", (centro[0]-20, centro[1]), 
                  cv.FONT_HERSHEY_SIMPLEX, 0.4, color, 1)
        
        if idx < 6:
            texto = f"#{idx+1}: {cuad['num_puntos']}pts/{cuad['area']:.0f}px = {cuad['ratio']:.4f}"
            cv.putText(img, texto, (10, y_offset), 
                      cv.FONT_HERSHEY_SIMPLEX, 0.4, color, 1)
            y_offset += 18
    
    if len(cuadrilateros) > 0:
        mejor = cuadrilateros[0]
        mejor_vertices = mejor['vertices']
        for v in mejor_vertices:
            cv.circle(img, (int(v[0]), int(v[1])), 8, (0, 255, 255), -1)
            cv.circle(img, (int(v[0]), int(v[1])), 10, (255, 255, 255), 1)
        
        cv.putText(img, f"MEJOR: {mejor['num_puntos']}pts / {mejor['area']:.0f}px = {mejor['ratio']:.4f}", 
                  (10, y_offset + 10), cv.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255), 2)
    
    cv.imshow(intersecciones_window, img)
    return img

def dibujar_lineas_hough(frame, lineas_horizontal, lineas_vertical):
    """
    Dibuja las líneas encontradas por HoughP
    """
    img = frame.copy()
    h, w = img.shape[:2]
    
    colores_h = [(255, 150, 0), (255, 100, 0), (200, 80, 0), (150, 50, 0)]
    colores_v = [(0, 200, 100), (0, 150, 80), (0, 100, 60), (0, 80, 50)]
    
    # Dibujar líneas horizontales con sus segmentos originales
    for i, recta in enumerate(lineas_horizontal):
        # Dibujar el segmento original
        p1, p2 = recta['puntos']
        color = colores_h[i % len(colores_h)]
        cv.line(img, p1, p2, color, 2)
        
        # Dibujar la línea extendida
        rho, theta = recta['rho'], recta['theta']
        p1_ext, p2_ext = recta_polar_a_puntos((rho, theta), w, h)
        cv.line(img, p1_ext, p2_ext, color, 1)
        
        cv.putText(img, f"H{i+1}", (p1[0], p1[1] - 10), 
                  cv.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)
    
    # Dibujar líneas verticales
    for i, recta in enumerate(lineas_vertical):
        # Dibujar el segmento original
        p1, p2 = recta['puntos']
        color = colores_v[i % len(colores_v)]
        cv.line(img, p1, p2, color, 2)
        
        # Dibujar la línea extendida
        rho, theta = recta['rho'], recta['theta']
        p1_ext, p2_ext = recta_polar_a_puntos((rho, theta), w, h)
        cv.line(img, p1_ext, p2_ext, color, 1)
        
        cv.putText(img, f"V{i+1}", (p1[0], p1[1] - 10), 
                  cv.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)
    
    cv.putText(img, f"Lineas H: {len(lineas_horizontal)}, Lineas V: {len(lineas_vertical)}", 
              (10, 30), cv.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
    
    cv.imshow(hough_window, img)
    return img

# ========== FIN DE FUNCIONES PARA HOUGH ==========

def contar_puntos_silla_en_poligono(polygon, saddle_points):
    if polygon is None or len(saddle_points) == 0:
        return 0
    
    count = 0
    for point in saddle_points:
        distance = cv.pointPolygonTest(polygon, point, True)
        if distance >= 0:
            count += 1
    
    return count

def encontrar_cuadrilatero_con_mas_puntos_silla(cuadrilateros, saddle_points):
    if len(cuadrilateros) == 0:
        return None, 0
    
    mejor_cuadrilatero = None
    max_puntos = 0
    
    for quad in cuadrilateros:
        num_puntos = contar_puntos_silla_en_poligono(quad, saddle_points)
        if num_puntos > max_puntos and num_puntos >= MIN_PUNTOS_SILLA:
            max_puntos = num_puntos
            mejor_cuadrilatero = quad
    
    return mejor_cuadrilatero, max_puntos

def ordenar_vertices_horario(vertices):
    if len(vertices) != 4:
        return None
    
    center = np.mean(vertices, axis=0)
    angles = np.arctan2(vertices[:, 1] - center[1], vertices[:, 0] - center[0])
    sorted_indices = np.argsort(angles)
    vertices_ordenados = vertices[sorted_indices]
    
    vertices_ordenados = vertices_ordenados[::-1]
    
    min_sum_idx = np.argmin(vertices_ordenados[:, 0] + vertices_ordenados[:, 1])
    vertices_ordenados = np.roll(vertices_ordenados, -min_sum_idx, axis=0)
    
    return vertices_ordenados

def ordenar_puntos_para_warp(puntos):
    center = np.mean(puntos, axis=0)
    angles = np.arctan2(puntos[:, 1] - center[1], puntos[:, 0] - center[0])
    sorted_indices = np.argsort(angles)
    puntos_ordenados = puntos[sorted_indices]
    
    puntos_ordenados = puntos_ordenados[::-1]
    
    min_sum_idx = np.argmin(puntos_ordenados[:, 0] + puntos_ordenados[:, 1])
    puntos_ordenados = np.roll(puntos_ordenados, -min_sum_idx, axis=0)
    
    return puntos_ordenados

def dibujar_cuadrilatero_y_warp(frame, src_points_ordenados, num_puntos, area, offset=0, 
                                es_hough=False, es_reciclado=False, warp_corregido=None, 
                                puntos_con_offset=None, orientacion_definida=False):
    """
    Dibuja el cuadrilátero y el warp con offset aplicado
    """
    estimated_board_img = frame.copy()
    
    # Si tenemos warp corregido, usarlo
    if warp_corregido is not None:
        warp_result_img = warp_corregido
    else:
        warp_result_img = np.zeros((800, 800, 3), dtype=np.uint8)
    
    # Si no se proporcionaron puntos con offset, calcularlos
    if puntos_con_offset is None:
        centro = np.mean(src_points_ordenados, axis=0)
        puntos_con_offset = aplicar_offset_a_puntos(src_points_ordenados, offset, centro)
    
    pts = puntos_con_offset.astype(np.int32)
    
    # Color del cuadrilátero según estado
    if orientacion_definida:
        color_cuad = (0, 255, 0)  # Verde: orientación definida
    else:
        color_cuad = (0, 255, 255)  # Amarillo: sin orientación
    
    cv.polylines(estimated_board_img, [pts], True, color_cuad, 4)
    for point in pts:
        cv.circle(estimated_board_img, tuple(point), 10, color_cuad, -1)
    
    if es_reciclado:
        metodo = "RECICLADO"
        color_texto = (0, 255, 255)
    elif es_hough:
        metodo = "HOUGH"
        color_texto = (0, 255, 255)
    else:
        metodo = "Detección"
        color_texto = (255, 255, 0)
    
    cv.putText(estimated_board_img, f"Metodo: {metodo}", 
              (10, 30), cv.FONT_HERSHEY_SIMPLEX, 0.6, color_texto, 2)
    cv.putText(estimated_board_img, f"Puntos silla: {num_puntos}", 
              (10, 60), cv.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
    cv.putText(estimated_board_img, f"Area: {int(area)}", 
              (10, 90), cv.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
    cv.putText(estimated_board_img, f"Offset: {offset}px", 
              (10, 120), cv.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)
    
    # Estado de la orientación
    if orientacion_definida:
        cv.putText(estimated_board_img, "ORIENTACION DEFINIDA ✓", 
                  (10, 150), cv.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
    else:
        cv.putText(estimated_board_img, "DEFINIENDO ORIENTACION...", 
                  (10, 150), cv.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)
    
    if es_reciclado:
        cv.putText(estimated_board_img, "RECICLADO (tolerancia)", 
                  (10, 180), cv.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)
    
    # Si el warp fue corregido, mostrar información
    if warp_corregido is not None:
        cv.putText(estimated_board_img, "ORIENTACION CORREGIDA", 
                  (10, 210), cv.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
        cv.putText(estimated_board_img, f"Offset aplicado: {offset}px", 
                  (10, 240), cv.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 0), 1)
    
    if es_reciclado:
        cv.putText(warp_result_img, "RECICLADO", 
                  (10, 30), cv.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)
    
    cv.putText(warp_result_img, f"Puntos silla: {num_puntos}", 
              (10, 60), cv.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
    cv.putText(warp_result_img, f"Area: {int(area)}", 
              (10, 90), cv.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
    cv.putText(warp_result_img, f"Offset: {offset}px", 
              (10, 120), cv.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)
    
    if orientacion_definida:
        cv.putText(warp_result_img, "ORIENTACION CORRECTA ✓", 
                  (10, 150), cv.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
    else:
        cv.putText(warp_result_img, "DEFINIENDO ORIENTACION...", 
                  (10, 150), cv.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)
    
    # Dibujar grid
    h, w = warp_result_img.shape[:2]
    cell_h = h // 8
    cell_w = w // 8
    for i in range(9):
        y = i * cell_h
        cv.line(warp_result_img, (0, y), (w, y), (0, 255, 255), 1)
        x = i * cell_w
        cv.line(warp_result_img, (x, 0), (x, h), (0, 255, 255), 1)
    
    # Marcar la orientación en el warp
    cv.putText(warp_result_img, "NEGRAS", (10, 30), 
              cv.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 2)
    cv.putText(warp_result_img, "BLANCAS", (10, h - 10), 
              cv.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2)
    
    return estimated_board_img, warp_result_img

def mostrar_mensaje_hough(frame, mensaje, color=(0, 0, 255)):
    img = frame.copy()
    h, w = img.shape[:2]
    cv.putText(img, mensaje, (w//2 - 150, h//2), 
              cv.FONT_HERSHEY_SIMPLEX, 1, color, 2)
    cv.imshow(hough_window, img)
    return img

# Variables para control de FPS
frame_count = 0
start_time = cv.getTickCount()

# Función auxiliar para actualizar estabilidad (SOLO PARA ORIENTACIÓN)
def actualizar_estabilidad_orientacion(vertices_actuales):
    global stability_buffer, orientacion_definida
    
    if vertices_actuales is not None:
        # Actualizar buffer de estabilidad
        stability_buffer.append(vertices_actuales.copy())
        
        # Verificar estabilidad
        if len(stability_buffer) > 0:
            es_estable, puntaje_estabilidad, distancias_estabilidad = verificar_estabilidad_vertices(
                vertices_actuales, stability_buffer, STABILITY_RADIUS_PX
            )
            
            # Si la orientación no está definida y hay suficiente estabilidad
            if not orientacion_definida and len(stability_buffer) >= STABILITY_CHECK_FRAMES and es_estable and puntaje_estabilidad > PUNTAJE_MINIMO_ESTABILIDAD:
                print(f"   ✅ ESTABILIDAD ALCANZADA! (puntaje: {puntaje_estabilidad:.2f})")
                return True, es_estable, puntaje_estabilidad, distancias_estabilidad
            
            print(f"   ⏳ Estabilizando orientación... ({len(stability_buffer)}/{STABILITY_CHECK_FRAMES} frames, puntaje: {puntaje_estabilidad:.2f})")
            return False, es_estable, puntaje_estabilidad, distancias_estabilidad
    
    return False, False, 0.0, [0.0, 0.0, 0.0, 0.0]

while True:
    # Capturar frame de la cámara
    ret, frame = cap.read()

    if not ret:
        print("Error al capturar frame de la cámara")
        break

    # Procesar el frame
    frame_vis = frame.copy()
    gray = cv.cvtColor(frame, cv.COLOR_BGR2GRAY)

    # ========== PASO 1: SOBEL Y CANNY ==========
    sobelx = cv.Sobel(gray, cv.CV_64F, 1, 0, ksize=3)
    sobely = cv.Sobel(gray, cv.CV_64F, 0, 1, ksize=3)
    sobel_magnitude = np.sqrt(sobelx**2 + sobely**2)
    sobel_magnitude = np.uint8(np.clip(sobel_magnitude, 0, 255))

    cv.imshow(sobel_window, sobel_magnitude)

    edges = cv.Canny(sobel_magnitude, 7000, 7050, apertureSize=5)
    cv.imshow(canny_window, edges)

    # ========== PASO 2: DILATACIÓN ==========
    kernel = np.ones((3,3), np.uint8)
    edges_dilated = cv.dilate(edges, kernel, iterations=2)
    cv.imshow(canny_dilated_window, edges_dilated)

    # ========== PASO 3: BÚSQUEDA DE CUADRILÁTERO + PUNTOS SILLA ==========
    Ixx = cv.Sobel(gray, cv.CV_32F, 2, 0, ksize=3)
    Iyy = cv.Sobel(gray, cv.CV_32F, 0, 2, ksize=3)
    Ixy = cv.Sobel(gray, cv.CV_32F, 1, 1, ksize=3)

    response = -(Ixx*Iyy - Ixy*Ixy)
    response = cv.GaussianBlur(response, (15,15), 0)

    mx = cv.dilate(response, np.ones((7,7), np.uint8))
    th = 0.15 * response.max()

    pts = np.where((response == mx) & (response > th))
    points = np.column_stack((pts[1], pts[0]))
    saddle_points = [(int(p[0]), int(p[1])) for p in points]

    saddle_points_img = frame.copy()
    if len(points) > 0:
        for point in points:
            cv.circle(saddle_points_img, (int(point[0]), int(point[1])), 4, (0, 255, 0), -1)
    cv.imshow(hough_filtered_window, saddle_points_img)

    contours, hierarchy = cv.findContours(edges_dilated, cv.RETR_EXTERNAL, cv.CHAIN_APPROX_SIMPLE)

    contour_img = frame.copy()
    cv.drawContours(contour_img, contours, -1, (0, 255, 0), 2)
    cv.imshow(contour_window, contour_img)

    # ========== CREAR IMAGEN BINARIA ==========
    polygons_binary = np.zeros_like(gray)
    all_polygons = []
    contour_approx_img = frame.copy()
    saddle_polygons = []
    
    for contour in contours:
        epsilon = 0.01 * cv.arcLength(contour, True)
        approx = cv.approxPolyDP(contour, epsilon, True)
        all_polygons.append(approx)
        cv.drawContours(contour_approx_img, [approx], -1, (255, 0, 0), 3)
        
        cv.drawContours(polygons_binary, [approx], -1, 255, 1)
        
        if len(approx) == 4:
            contains_saddle = False
            for point in saddle_points:
                distance = cv.pointPolygonTest(approx, point, True)
                if distance >= -TOLERANCE:
                    contains_saddle = True
                    break
            if contains_saddle:
                saddle_polygons.append(approx)

    cv.imshow(approx_window, contour_approx_img)
    cv.imshow(polygons_binary_window, polygons_binary)

    quadrilaterals_img = frame.copy()
    quadrilaterals = []
    
    for polygon in saddle_polygons:
        if len(polygon) == 4:
            area = cv.contourArea(polygon)
            if cv.isContourConvex(polygon) and area >= MIN_AREA:
                quadrilaterals.append(polygon)
                cv.drawContours(quadrilaterals_img, [polygon], -1, (0, 0, 255), 3)
                for point in polygon:
                    cv.circle(quadrilaterals_img, tuple(point[0]), 6, (0, 255, 255), -1)
                cv.putText(quadrilaterals_img, f"Area: {int(area)}", 
                          (int(polygon[0][0][0]), int(polygon[0][0][1]) - 20), 
                          cv.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2)
    
    cv.imshow(quadrilaterals_window, quadrilaterals_img)

    # ========== PASO 4: SELECCIONAR CUADRILÁTERO (SIEMPRE EJECUTAR) ==========
    warp_result_img = np.zeros((800, 800, 3), dtype=np.uint8)
    estimated_board_img = frame.copy()
    
    offset_actual = cv.getTrackbarPos("Offset", controls_window)
    hough_threshold = cv.getTrackbarPos("Umbral Hough", controls_window)
    hough_min_length = cv.getTrackbarPos("Longitud mínima", controls_window)
    hough_max_gap = cv.getTrackbarPos("Gap máximo", controls_window)
    recycle_threshold = cv.getTrackbarPos("Tolerancia Reciclaje %", controls_window) / 100.0
    
    # Variables de estado
    encontrado_valido = False
    usando_hough = False
    usando_reciclado = False
    
    # Variables para warp corregido
    warp_corregido = None
    puntos_ordenados_final = None
    puntos_con_offset_final = None
    num_puntos_final = 0
    area_final = 0
    info_orientacion = ""
    transformaciones = []
    datos_energia = None
    
    # Variables para estabilidad (solo orientación)
    vertices_actuales = None
    es_estable = True
    puntaje_estabilidad = 1.0
    distancias_estabilidad = [0.0, 0.0, 0.0, 0.0]
    buffer_actualizado = False

    # ===== ALGORITMO DE DETECCIÓN (SIEMPRE EJECUTAR) =====
    
    # ===== PASO 1: BUSCAR CUADRILÁTERO CON PUNTOS DE SILLA =====
    if len(quadrilaterals) > 0 and len(saddle_points) > 0:
        mejor_quad, num_puntos = encontrar_cuadrilatero_con_mas_puntos_silla(quadrilaterals, saddle_points)
        
        if mejor_quad is not None:
            encontrado_valido = True
            print(f"✅ CUADRILÁTERO DETECTADO: {num_puntos} puntos silla")
            
            # Guardar como último válido
            ultimo_cuadrilatero_valido = mejor_quad
            vertices = get_vertices_as_points(mejor_quad)
            src_points = np.array(vertices, dtype=np.float32)
            src_points_ordenados = ordenar_puntos_para_warp(src_points)
            
            ultimos_puntos_ordenados = src_points_ordenados
            ultimo_frame_warp = frame.copy()
            area = cv.contourArea(mejor_quad)
            ultimo_num_puntos = num_puntos
            ultimo_area = area
            
            # Aplicar warp con offset
            warped, M, puntos_con_offset = aplicar_warp_con_offset(frame, src_points_ordenados, offset_actual)
            warp_result_img = warped
            
            # Guardar vértices actuales para estabilidad de orientación
            vertices_actuales = src_points_ordenados.copy()
            
            # Actualizar estabilidad de orientación
            orientacion_estable, es_estable, puntaje_estabilidad, distancias_estabilidad = actualizar_estabilidad_orientacion(vertices_actuales)
            
            # Si la orientación no está definida y hay estabilidad, definirla
            if not orientacion_definida and orientacion_estable:
                print("🔄 ANALIZANDO ORIENTACIÓN DEL TABLERO...")
                
                # Analizar y corregir orientación en la imagen warpeada
                warped_corregida, transformaciones, esquinas_cat, datos_energia = analizar_y_corregir_orientacion(warp_result_img)
                
                # Mostrar visualización de energía
                dibujar_energia_filas_columnas(warped_corregida)
                
                # Guardar la orientación
                orientacion_definida = True
                esquinas_categorizadas = esquinas_cat
                warp_corregido_guardado = warped_corregida.copy()
                
                info_orientacion = f"Orientación definida: {', '.join(transformaciones) if transformaciones else 'Correcta'}"
                print(f"✅ {info_orientacion}")
                
                # Dibujar información de orientación
                dibujar_orientacion(frame, info_orientacion, transformaciones)
            
            # Si la orientación ya está definida, aplicar la corrección directamente
            if orientacion_definida and warp_corregido_guardado is not None:
                # Aplicar la corrección de orientación al warp actual
                warped_corregida, transformaciones, esquinas_cat, datos_energia = analizar_y_corregir_orientacion(warp_result_img)
                warp_corregido = warped_corregida
                warp_corregido_guardado = warped_corregida.copy()
                warp_result_img = warped_corregida.copy()
                info_orientacion = "Orientación aplicada"
            else:
                warp_corregido = warp_result_img
                info_orientacion = "Esperando orientación..."
    
    # ===== PASO 2: SI NO HAY CUADRILÁTERO, USAR HOUGH =====
    if not encontrado_valido:
        if polygons_binary is not None and np.sum(polygons_binary) > 100:
            
            # Obtener líneas con HoughP (siempre ejecutar)
            lineas = obtener_rectas_hough(
                polygons_binary, 
                hough_threshold,
                hough_min_length,
                hough_max_gap
            )
            
            if len(lineas) > 0:
                # Agrupar por orientación
                rectas_horizontal, rectas_vertical = agrupar_lineas_por_orientacion(lineas)
                
                # Fusionar líneas cercanas
                rectas_horizontal = fusionar_lineas_cercanas(rectas_horizontal)
                rectas_vertical = fusionar_lineas_cercanas(rectas_vertical)
                
                print(f"   Líneas horizontales: {len(rectas_horizontal)}, verticales: {len(rectas_vertical)}")
            else:
                rectas_horizontal = []
                rectas_vertical = []
            
            dibujar_lineas_hough(frame, rectas_horizontal, rectas_vertical)
            
            if len(rectas_horizontal) >= 2 and len(rectas_vertical) >= 2:
                cuadrilateros_hough = formar_cuadrilateros_desde_rectas(
                    rectas_horizontal, rectas_vertical, saddle_points, frame.shape
                )
            else:
                cuadrilateros_hough = []
            
            dibujar_intersecciones_y_cuadrilateros(
                frame, rectas_horizontal, rectas_vertical, saddle_points, cuadrilateros_hough
            )
            
            if len(cuadrilateros_hough) > 0:
                usando_hough = True
                encontrado_valido = True
                
                mejor = cuadrilateros_hough[0]
                mejor_vertices = mejor['vertices']
                num_puntos = mejor['num_puntos']
                area = mejor['area']
                
                # Guardar como último válido
                src_points = np.array(mejor_vertices, dtype=np.float32)
                src_points_ordenados = ordenar_puntos_para_warp(src_points)
                
                # Aplicar warp con offset
                warped, M, puntos_con_offset = aplicar_warp_con_offset(frame, src_points_ordenados, offset_actual)
                
                # Guardar vértices actuales para estabilidad de orientación
                vertices_actuales = src_points_ordenados.copy()
                
                # Actualizar estabilidad de orientación
                orientacion_estable, es_estable, puntaje_estabilidad, distancias_estabilidad = actualizar_estabilidad_orientacion(vertices_actuales)
                
                ultimos_puntos_ordenados = src_points_ordenados
                ultimo_frame_warp = frame.copy()
                ultimo_cuadrilatero_valido = mejor_vertices
                ultimo_num_puntos = num_puntos
                ultimo_area = area
                ultimo_warp_valido = warp_corregido.copy() if warp_corregido is not None else warped.copy()
                
                print(f"✅ HOUGH exitoso: {num_puntos} puntos, área={area:.0f}")
                
                # Si la orientación ya está definida, aplicar la corrección
                if orientacion_definida and warp_corregido_guardado is not None:
                    warped_corregida, transformaciones, esquinas_cat, datos_energia = analizar_y_corregir_orientacion(warped)
                    warp_corregido = warped_corregida
                    warp_corregido_guardado = warped_corregida.copy()
                    warp_result_img = warped_corregida.copy()
                    info_orientacion = "Orientación aplicada (Hough)"
                else:
                    warp_corregido = warped
                    warp_result_img = warped
                    info_orientacion = "Esperando orientación (Hough)"
            else:
                # Hough no encontró nada, reciclar si existe
                if ultimo_cuadrilatero_valido is not None:
                    usando_reciclado = True
                    encontrado_valido = True
                    print(f"♻️ RECICLANDO (Hough sin resultados)")
                    src_points_ordenados = ultimos_puntos_ordenados
                    num_puntos = ultimo_num_puntos
                    area = ultimo_area
                    
                    warped, M, puntos_con_offset = aplicar_warp_con_offset(frame, src_points_ordenados, offset_actual)
                    
                    # Guardar vértices actuales para estabilidad de orientación
                    vertices_actuales = src_points_ordenados.copy()
                    
                    # Actualizar estabilidad de orientación
                    orientacion_estable, es_estable, puntaje_estabilidad, distancias_estabilidad = actualizar_estabilidad_orientacion(vertices_actuales)
                    
                    warp_corregido = warped
                    warp_result_img = warped
                    mostrar_mensaje_hough(frame, "RECICLADO (sin Hough)", (0, 255, 255))
                else:
                    mostrar_mensaje_hough(frame, "No se encontraron cuadriláteros", (0, 0, 255))
                    cv.putText(estimated_board_img, "SIN DETECCION", 
                              (10, 30), cv.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
                    cv.putText(warp_result_img, "SIN DETECCION", 
                              (10, 30), cv.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
        else:
            # Sin máscara binaria, reciclar si es posible
            if ultimo_cuadrilatero_valido is not None:
                usando_reciclado = True
                encontrado_valido = True
                print(f"♻️ RECICLANDO (sin máscara binaria)")
                src_points_ordenados = ultimos_puntos_ordenados
                num_puntos = ultimo_num_puntos
                area = ultimo_area
                
                warped, M, puntos_con_offset = aplicar_warp_con_offset(frame, src_points_ordenados, offset_actual)
                
                # Guardar vértices actuales para estabilidad de orientación
                vertices_actuales = src_points_ordenados.copy()
                
                # Actualizar estabilidad de orientación
                orientacion_estable, es_estable, puntaje_estabilidad, distancias_estabilidad = actualizar_estabilidad_orientacion(vertices_actuales)
                
                warp_corregido = warped
                warp_result_img = warped
                mostrar_mensaje_hough(frame, "RECICLADO (sin máscara)", (0, 255, 255))
            else:
                cv.putText(estimated_board_img, "SIN DETECCION", 
                          (10, 30), cv.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
                cv.putText(warp_result_img, "SIN DETECCION", 
                          (10, 30), cv.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
                mostrar_mensaje_hough(frame, "SIN DETECCION", (0, 0, 255))
    
    # ===== DIBUJAR RESULTADOS =====
    if encontrado_valido and vertices_actuales is not None:
        # Dibujar información de estabilidad (solo orientación)
        dibujar_estabilidad(frame, vertices_actuales, distancias_estabilidad, 
                           puntaje_estabilidad, es_estable, orientacion_definida)
        
        # Actualizar puntos finales para el dibujo
        if 'puntos_con_offset' in locals() and puntos_con_offset is not None:
            puntos_con_offset_final = puntos_con_offset
        else:
            puntos_con_offset_final = None
            
        if 'src_points_ordenados' in locals() and src_points_ordenados is not None:
            puntos_ordenados_final = src_points_ordenados
        else:
            puntos_ordenados_final = ultimos_puntos_ordenados if ultimos_puntos_ordenados is not None else np.zeros((4, 2), dtype=np.float32)
            
        num_puntos_final = ultimo_num_puntos
        area_final = ultimo_area
        
        estimated_board_img, warp_result_img = dibujar_cuadrilatero_y_warp(
            frame, puntos_ordenados_final, num_puntos_final, area_final, offset_actual, 
            usando_hough, usando_reciclado, warp_corregido, puntos_con_offset_final, orientacion_definida
        )
        
        if encontrado_valido:
            cv.putText(estimated_board_img, "DETECCION TRADICIONAL", 
                      (10, 180 if orientacion_definida else 210), cv.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)
        elif usando_hough:
            cv.putText(estimated_board_img, "HOUGH", 
                      (10, 180 if orientacion_definida else 210), cv.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)
        elif usando_reciclado:
            cv.putText(estimated_board_img, "RECICLADO", 
                      (10, 180 if orientacion_definida else 210), cv.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)
    
    cv.imshow(estimated_board_window, estimated_board_img)
    cv.imshow(warp_window, warp_result_img)
    cv.imshow(window_name, frame_vis)

    # Mostrar información de FPS
    frame_count += 1
    if frame_count % 30 == 0:
        end_time = cv.getTickCount()
        elapsed = (end_time - start_time) / cv.getTickFrequency()
        fps_actual = 30 / elapsed
        print(f"FPS: {fps_actual:.2f}")
        start_time = cv.getTickCount()

    # Salir con ESC
    key = cv.waitKey(1) & 0xFF
    if key == 27:
        break

cap.release()
cv.destroyAllWindows()

Resolución: 640x480
FPS: 30.00
   Líneas horizontales: 10, verticales: 4
   ⏳ Estabilizando orientación... (1/5 frames, puntaje: 1.00)
✅ HOUGH exitoso: 91 puntos, área=104512
   Líneas horizontales: 7, verticales: 4
   ⏳ Estabilizando orientación... (2/5 frames, puntaje: 1.00)
✅ HOUGH exitoso: 143 puntos, área=106497
   Líneas horizontales: 6, verticales: 5
   ⏳ Estabilizando orientación... (3/5 frames, puntaje: 1.00)
✅ HOUGH exitoso: 139 puntos, área=106370
   Líneas horizontales: 6, verticales: 5
   ⏳ Estabilizando orientación... (4/5 frames, puntaje: 1.00)
✅ HOUGH exitoso: 139 puntos, área=106370
   Líneas horizontales: 3, verticales: 4
   ✅ ESTABILIDAD ALCANZADA! (puntaje: 0.87)
✅ HOUGH exitoso: 201 puntos, área=107564
   Líneas horizontales: 3, verticales: 4
   ✅ ESTABILIDAD ALCANZADA! (puntaje: 0.93)
✅ HOUGH exitoso: 201 puntos, área=107564
   Líneas horizontales: 3, verticales: 4
   ✅ ESTABILIDAD ALCANZADA! (puntaje: 0.93)
✅ HOUGH exitoso: 200 puntos, área=108388
   Líneas horiz

In [ ]:
# =============================================================================
#  DETECCIÓN DE TABLERO DE AJEDREZ EN VIVO + CORRECCIÓN DE ORIENTACIÓN
# =============================================================================
#
#  ALGORITMO (resumen):
#
#  FASE 1 - BÚSQUEDA DEL TABLERO (se repite cada frame, salvo reciclaje):
#     1) Sobel + Canny -> dilatación -> contornos -> polígonos aproximados.
#     2) De esos polígonos, cuadriláteros convexos con área mínima y con
#        suficientes "puntos de silla" (esquinas de casillero) en su interior.
#        Si hay varios, gana el que tenga más puntos de silla.
#     3) Si no se encontró nada así, se cae a un HoughLinesP sobre la máscara
#        binaria de los polígonos aproximados. Se agrupan las rectas en
#        horizontales/verticales, se calculan todas las intersecciones y se
#        arman cuadriláteros. Gana el de mayor ratio (puntos de silla**2 / np.sqrt(área)).
#     4) OPTIMIZACIÓN DE RECICLAJE: antes de repetir todo lo anterior, se
#        cuentan los puntos de silla del frame actual que caen DENTRO del
#        último cuadrilátero conocido. Si esa cantidad no cayó más de un
#        porcentaje de tolerancia respecto al último valor bueno, se reciclan
#        directamente esas 4 esquinas (se evita todo el pipeline de arriba).
#
#  FASE 2 - CORRECCIÓN DE ORIENTACIÓN (se resuelve UNA SOLA VEZ):
#     1) No se confía en la orientación hasta que las 4 esquinas se mantengan
#        relativamente quietas durante varios frames (estabilidad).
#     2) Una vez estables, se analiza la energía de las 2 primeras/últimas
#        filas y columnas del warp crudo para decidir si hay que transponer
#        (90°) y/o voltear verticalmente (180°) para que los jugadores queden
#        enfrentados arriba/abajo y las blancas terminen abajo.
#     3) Con esas dos decisiones se calcula, de forma puramente geométrica,
#        qué esquina cruda (de las 4 detectadas) corresponde a cada etiqueta
#        final TL/TR/BR/BL, y se guardan esas 4 esquinas como "referencia".
#     4) De ahí en adelante, el warp es trivial: cada frame se emparejan las
#        4 esquinas recién detectadas con las 4 de referencia (la asignación
#        que minimiza la distancia total), se actualiza la posición de la
#        referencia, y se warpea directamente en el orden TL,TR,BR,BL. No se
#        vuelve a analizar energía nunca más.
#
# =============================================================================

import cv2 as cv
import numpy as np
from collections import deque
import itertools

# ============================== CÁMARA ======================================

CAMERA_ID = 0
cap = cv.VideoCapture(CAMERA_ID)

if not cap.isOpened():
    raise IOError(f"No se pudo abrir la cámara {CAMERA_ID}")

frame_width = int(cap.get(cv.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv.CAP_PROP_FPS)
print(f"Resolución: {frame_width}x{frame_height}")
print(f"FPS: {fps:.2f}")

# ============================== VENTANAS ====================================

window_name = "Video"
sobel_window = "Sobel"
canny_window = "Canny"
canny_dilated_window = "Canny Dilatado"
saddle_window = "Puntos de Silla"
contour_window = "Contornos"
approx_window = "Polígonos aproximados"
quadrilaterals_window = "Cuadriláteros candidatos"
estimated_board_window = "Cuadrilátero elegido"
warp_window = "Warp final"
controls_window = "Controles"
hough_window = "HoughP - Líneas detectadas"
polygons_binary_window = "Máscara binaria de polígonos"
intersecciones_window = "Intersecciones y cuadriláteros (Hough)"
orientation_window = "Orientación del tablero"
stability_window = "Estabilidad / Tracking de esquinas"
energy_window = "Energía de filas y columnas"

for w in [window_name, sobel_window, canny_window, canny_dilated_window,
          saddle_window, contour_window, approx_window, quadrilaterals_window,
          estimated_board_window, warp_window, controls_window, hough_window,
          polygons_binary_window, intersecciones_window, orientation_window,
          stability_window, energy_window]:
    cv.namedWindow(w, cv.WINDOW_NORMAL)

# ============================== CONTROLES ===================================

offset_pixels = 15
HOUGH_THRESHOLD = 50
HOUGH_MIN_LINE_LENGTH = 50
HOUGH_MAX_LINE_GAP = 40
RECYCLE_THRESHOLD_PERCENT = 5  # % de tolerancia de caída de puntos de silla

def _set_offset(pos):
    global offset_pixels
    offset_pixels = pos

def _set_hough_threshold(pos):
    global HOUGH_THRESHOLD
    HOUGH_THRESHOLD = max(1, pos)

def _set_hough_min_length(pos):
    global HOUGH_MIN_LINE_LENGTH
    HOUGH_MIN_LINE_LENGTH = max(5, pos)

def _set_hough_max_gap(pos):
    global HOUGH_MAX_LINE_GAP
    HOUGH_MAX_LINE_GAP = max(1, pos)

def _set_recycle_threshold(pos):
    global RECYCLE_THRESHOLD_PERCENT
    RECYCLE_THRESHOLD_PERCENT = max(1, pos)

cv.createTrackbar("Offset", controls_window, offset_pixels, 50, _set_offset)
cv.createTrackbar("Umbral Hough", controls_window, HOUGH_THRESHOLD, 200, _set_hough_threshold)
cv.createTrackbar("Longitud minima", controls_window, HOUGH_MIN_LINE_LENGTH, 200, _set_hough_min_length)
cv.createTrackbar("Gap maximo", controls_window, HOUGH_MAX_LINE_GAP, 100, _set_hough_max_gap)
cv.createTrackbar("Tolerancia Reciclaje %", controls_window, RECYCLE_THRESHOLD_PERCENT, 100, _set_recycle_threshold)

HOUGH_DISTANCE_RESOLUTION = 1
HOUGH_ANGLE_RESOLUTION = np.pi / 360

# ============================== PARÁMETROS ==================================

TOLERANCE = 5            # margen (px) al contar puntos de silla dentro de un polígono
MIN_AREA = 10             # área mínima para un cuadrilátero por contornos
MIN_AREA_HOUGH = 100      # área mínima para un cuadrilátero armado por Hough
MIN_PUNTOS_SILLA = 50     # puntos de silla mínimos para aceptar un cuadrilátero

CANNY_THRESH1 = 7000
CANNY_THRESH2 = 7050

STABILITY_CHECK_FRAMES = 64      # frames a acumular antes de confiar en la estabilidad
STABILITY_RADIUS_PX = 15        # movimiento máximo promedio permitido (px)
PUNTAJE_MINIMO_ESTABILIDAD = 0.7

DST_SIZE = (800, 800)
ETIQUETAS = ["TL", "TR", "BR", "BL"]

# ============================== ESTADO GLOBAL ===============================

# Último cuadrilátero conocido (esquinas "crudas", en orden cíclico p0..p3,
# en coordenadas del frame). Sirve tanto de respaldo como de base del chequeo
# de reciclaje.
ultimos_corners_ciclicos = None
ultimo_num_puntos_silla = 0
ultimo_area = 0

# Buffer de estabilidad (solo se usa mientras la orientación no está definida)
stability_buffer = deque(maxlen=STABILITY_CHECK_FRAMES)

# Una vez resuelta la orientación (una sola vez), queda esto fijo y se va
# actualizando por tracking de vecino más cercano:
orientacion_definida = False
corners_referencia = None  # dict {'TL':(x,y), 'TR':(x,y), 'BR':(x,y), 'BL':(x,y)}

frame_count = 0
start_time = cv.getTickCount()

# =============================================================================
#  UTILIDADES GEOMÉTRICAS Y DE WARP
# =============================================================================

def get_vertices_as_points(polygon):
    return [(int(p[0][0]), int(p[0][1])) for p in polygon]

def ordenar_puntos_para_warp(puntos):
    """
    Ordena 4 puntos en un orden cíclico consistente (sentido horario,
    empezando por el que tenga menor x+y). Esto NO garantiza que el punto 0
    sea siempre la esquina "arriba-izquierda" real del tablero (eso se
    resuelve en la Fase 2); solo garantiza que la función, aplicada siempre
    de la misma manera, entrega los 4 puntos en el mismo orden relativo.
    """
    centro = np.mean(puntos, axis=0)
    angulos = np.arctan2(puntos[:, 1] - centro[1], puntos[:, 0] - centro[0])
    orden = np.argsort(angulos)[::-1]
    ordenados = puntos[orden]
    idx_min = np.argmin(ordenados[:, 0] + ordenados[:, 1])
    return np.roll(ordenados, -idx_min, axis=0)

def apply_warp(frame, src_points, dst_size=DST_SIZE):
    dst_points = np.array([
        [0, 0],
        [dst_size[0] - 1, 0],
        [dst_size[0] - 1, dst_size[1] - 1],
        [0, dst_size[1] - 1]
    ], dtype=np.float32)
    src_points = np.array(src_points, dtype=np.float32)
    M = cv.getPerspectiveTransform(src_points, dst_points)
    return cv.warpPerspective(frame, M, dst_size), M

def aplicar_offset_a_puntos(puntos, offset, centro):
    if offset == 0:
        return puntos
    salida = []
    for punto in puntos:
        vector = centro - punto
        norm = np.linalg.norm(vector)
        salida.append(punto + (vector / norm) * offset if norm > 0 else punto)
    return np.array(salida, dtype=np.float32)

def aplicar_warp_con_offset(frame, src_points, offset_px, dst_size=DST_SIZE):
    centro = np.mean(src_points, axis=0)
    puntos_offset = aplicar_offset_a_puntos(src_points, offset_px, centro) if offset_px > 0 else src_points
    warped, M = apply_warp(frame, puntos_offset, dst_size)
    return warped, M, puntos_offset

def contar_puntos_silla_en_poligono(polygon, saddle_points, tolerancia=TOLERANCE):
    if polygon is None or len(saddle_points) == 0:
        return 0
    count = 0
    for point in saddle_points:
        if cv.pointPolygonTest(polygon, point, True) >= -tolerancia:
            count += 1
    return count

def a_poligono_int32(puntos):
    return np.array(puntos, dtype=np.int32).reshape(-1, 1, 2)

# =============================================================================
#  DETECCIÓN DE PUNTOS DE SILLA (esquinas de casillero)
# =============================================================================

def detectar_puntos_silla(gray):
    Ixx = cv.Sobel(gray, cv.CV_32F, 2, 0, ksize=3)
    Iyy = cv.Sobel(gray, cv.CV_32F, 0, 2, ksize=3)
    Ixy = cv.Sobel(gray, cv.CV_32F, 1, 1, ksize=3)

    response = -(Ixx * Iyy - Ixy * Ixy)
    response = cv.GaussianBlur(response, (15, 15), 0)

    mx = cv.dilate(response, np.ones((7, 7), np.uint8))
    th = 0.15 * response.max()

    ys, xs = np.where((response == mx) & (response > th))
    return [(int(x), int(y)) for x, y in zip(xs, ys)]

# =============================================================================
#  MÉTODO 1: CUADRILÁTERO POR CONTORNOS (Sobel + Canny + dilatación)
# =============================================================================

def detectar_bordes(gray):
    sobelx = cv.Sobel(gray, cv.CV_64F, 1, 0, ksize=3)
    sobely = cv.Sobel(gray, cv.CV_64F, 0, 1, ksize=3)
    sobel_mag = np.uint8(np.clip(np.sqrt(sobelx ** 2 + sobely ** 2), 0, 255))
    canny = cv.Canny(sobel_mag, CANNY_THRESH1, CANNY_THRESH2, apertureSize=5)
    return sobel_mag, canny

def dilatar_bordes(canny_img):
    kernel = np.ones((3, 3), np.uint8)
    return cv.dilate(canny_img, kernel, iterations=2)

def buscar_contornos_y_aproximar(edges_dilated):
    contornos, _ = cv.findContours(edges_dilated, cv.RETR_EXTERNAL, cv.CHAIN_APPROX_SIMPLE)
    poligonos_aprox = []
    mascara_binaria = np.zeros_like(edges_dilated)
    for c in contornos:
        epsilon = 0.01 * cv.arcLength(c, True)
        approx = cv.approxPolyDP(c, epsilon, True)
        poligonos_aprox.append(approx)
        cv.drawContours(mascara_binaria, [approx], -1, 255, 1)
    return contornos, poligonos_aprox, mascara_binaria

def filtrar_cuadrilateros_validos(poligonos_aprox, saddle_points):
    """Cuadriláteros convexos, con área y puntos de silla suficientes."""
    validos = []
    for approx in poligonos_aprox:
        if len(approx) != 4 or not cv.isContourConvex(approx):
            continue
        area = cv.contourArea(approx)
        if area < MIN_AREA:
            continue
        num_puntos = contar_puntos_silla_en_poligono(approx, saddle_points)
        if num_puntos < MIN_PUNTOS_SILLA:
            continue
        validos.append({"vertices": approx, "num_puntos": num_puntos, "area": area})
    return validos

def elegir_mejor_por_puntos_silla(candidatos):
    """Si hay más de un candidato válido, gana el que tenga más puntos de silla."""
    if not candidatos:
        return None
    return max(candidatos, key=lambda c: c["num_puntos"])

# =============================================================================
#  MÉTODO 2: HOUGH PROBABILÍSTICO (fallback si el método 1 no encontró nada)
# =============================================================================

def obtener_rectas_hough(imagen_binaria, threshold, min_line_length, max_line_gap):
    if imagen_binaria is None:
        return []
    lines = cv.HoughLinesP(imagen_binaria, HOUGH_DISTANCE_RESOLUTION, HOUGH_ANGLE_RESOLUTION,
                            threshold, minLineLength=min_line_length, maxLineGap=max_line_gap)
    if lines is None:
        return []

    lineas_polar = []
    for line in lines:
        x1, y1, x2, y2 = line[0]
        dx, dy = x2 - x1, y2 - y1
        theta = (np.pi / 2 if dy > 0 else -np.pi / 2) if abs(dx) < 1e-6 else np.arctan2(dx, -dy)
        rho = x1 * np.cos(theta) + y1 * np.sin(theta)
        theta_norm = theta % np.pi
        es_horizontal = theta_norm < np.pi / 4 or theta_norm > 3 * np.pi / 4
        lineas_polar.append({
            "rho": rho, "theta": theta, "puntos": [(x1, y1), (x2, y2)],
            "origen": "horizontal" if es_horizontal else "vertical"
        })
    return lineas_polar

def agrupar_lineas_por_orientacion(lineas):
    horizontales = [l for l in lineas if l["origen"] == "horizontal"]
    verticales = [l for l in lineas if l["origen"] == "vertical"]
    return horizontales, verticales

def fusionar_lineas_cercanas(lineas, umbral_rho=20, umbral_theta=0.1):
    if len(lineas) < 2:
        return lineas
    fusionadas = []
    usadas = [False] * len(lineas)
    for i in range(len(lineas)):
        if usadas[i]:
            continue
        grupo = [i]
        usadas[i] = True
        for j in range(i + 1, len(lineas)):
            if usadas[j]:
                continue
            theta1_n, theta2_n = lineas[i]["theta"] % np.pi, lineas[j]["theta"] % np.pi
            diff_rho = abs(lineas[i]["rho"] - lineas[j]["rho"])
            diff_theta = abs(theta1_n - theta2_n)
            diff_theta = min(diff_theta, np.pi - diff_theta)
            if diff_rho < umbral_rho and diff_theta < umbral_theta:
                grupo.append(j)
                usadas[j] = True
        rho_prom = np.mean([lineas[idx]["rho"] for idx in grupo])
        theta_prom = np.mean([lineas[idx]["theta"] for idx in grupo])
        fusionadas.append({
            "rho": rho_prom, "theta": theta_prom,
            "puntos": lineas[grupo[0]]["puntos"], "origen": lineas[grupo[0]]["origen"]
        })
    return fusionadas

def recta_polar_a_puntos(recta_polar, ancho, alto):
    rho, theta = recta_polar
    a, b = np.cos(theta), np.sin(theta)
    if abs(a) > 0.01:
        x1, x2 = 0, ancho
        y1 = int((rho - x1 * a) / b) if abs(b) > 1e-6 else 0
        y2 = int((rho - x2 * a) / b) if abs(b) > 1e-6 else 0
    else:
        y1, y2 = 0, alto
        x1 = x2 = int(rho / a) if abs(a) > 1e-6 else 0
    x1, x2 = max(0, min(ancho, x1)), max(0, min(ancho, x2))
    y1, y2 = max(0, min(alto, y1)), max(0, min(alto, y2))
    return (x1, y1), (x2, y2)

def calcular_interseccion_polar(recta1, recta2):
    rho1, theta1 = recta1
    rho2, theta2 = recta2
    if abs(theta1 - theta2) < 1e-6 or abs(abs(theta1 - theta2) - np.pi) < 1e-6:
        return None
    A = np.array([[np.cos(theta1), np.sin(theta1)], [np.cos(theta2), np.sin(theta2)]])
    b = np.array([rho1, rho2])
    try:
        x, y = np.linalg.solve(A, b)
        return (int(x), int(y))
    except np.linalg.LinAlgError:
        return None

def formar_cuadrilateros_desde_rectas(rectas_horizontal, rectas_vertical, saddle_points, frame_shape):
    """
    Arma cuadriláteros combinando pares de rectas horizontales con pares de
    rectas verticales. El puntaje de cada uno es el ratio puntos_silla/área
    (tal como se describe: cuadriláteros con muchos puntos de silla en poca
    área ganan por sobre cuadriláteros grandes con pocos puntos).
    """
    h, w = frame_shape[:2]
    cuadrilateros = []

    def longitud(l):
        (x1, y1), (x2, y2) = l["puntos"]
        return np.hypot(x2 - x1, y2 - y1)

    rectas_horizontal = sorted(rectas_horizontal, key=longitud, reverse=True)[:8]
    rectas_vertical = sorted(rectas_vertical, key=longitud, reverse=True)[:8]

    for i1, i2 in itertools.combinations(range(len(rectas_horizontal)), 2):
        for j1, j2 in itertools.combinations(range(len(rectas_vertical)), 2):
            h1, h2 = rectas_horizontal[i1], rectas_horizontal[i2]
            v1, v2 = rectas_vertical[j1], rectas_vertical[j2]

            inter1 = calcular_interseccion_polar((h1["rho"], h1["theta"]), (v1["rho"], v1["theta"]))
            inter2 = calcular_interseccion_polar((h1["rho"], h1["theta"]), (v2["rho"], v2["theta"]))
            inter3 = calcular_interseccion_polar((h2["rho"], h2["theta"]), (v2["rho"], v2["theta"]))
            inter4 = calcular_interseccion_polar((h2["rho"], h2["theta"]), (v1["rho"], v1["theta"]))

            if None in (inter1, inter2, inter3, inter4):
                continue
            vertices = [inter1, inter2, inter3, inter4]
            if any(v[0] < 0 or v[0] >= w or v[1] < 0 or v[1] >= h for v in vertices):
                continue

            vertices_ordenados = ordenar_puntos_para_warp(np.array(vertices, dtype=np.float32))
            poly = a_poligono_int32(vertices_ordenados)
            if not cv.isContourConvex(poly):
                continue

            area = cv.contourArea(poly)
            if area < MIN_AREA_HOUGH:
                continue

            num_puntos = contar_puntos_silla_en_poligono(poly, saddle_points)
            if num_puntos < MIN_PUNTOS_SILLA:
                continue

            ratio = num_puntos**2 / np.sqrt(area)
            cuadrilateros.append({
                "vertices": vertices_ordenados, "num_puntos": num_puntos,
                "area": area, "ratio": ratio, "h1": h1, "h2": h2, "v1": v1, "v2": v2
            })

    cuadrilateros.sort(key=lambda c: c["ratio"], reverse=True)
    return cuadrilateros

# =============================================================================
#  OPTIMIZACIÓN DE RECICLAJE
# =============================================================================

def evaluar_reciclaje(corners_previos, saddle_points, num_puntos_previo, tolerancia_pct):
    """
    Cuenta cuántos puntos de silla del frame actual caen dentro del último
    cuadrilátero conocido. Si esa cantidad no cayó más que la tolerancia
    respecto al último valor bueno, se puede reciclar (evitando repetir todo
    el pipeline de detección).
    """
    if corners_previos is None or num_puntos_previo <= 0:
        return False, 0
    poly = a_poligono_int32(corners_previos)
    num_actual = contar_puntos_silla_en_poligono(poly, saddle_points)
    umbral_minimo = num_puntos_previo * (1 - tolerancia_pct / 100.0)
    return num_actual >= umbral_minimo, num_actual

# =============================================================================
#  ESTABILIDAD DE VÉRTICES (solo para decidir CUÁNDO fijar la orientación)
# =============================================================================

def verificar_estabilidad_vertices(vertices_actuales, buffer_vertices, radio_maximo):
    if len(buffer_vertices) < STABILITY_CHECK_FRAMES:
        return True, 1.0, [0.0, 0.0, 0.0, 0.0]

    distancias = []
    for i in range(4):
        dists_i = [np.linalg.norm(vertices_actuales[i] - v_ant[i]) for v_ant in buffer_vertices]
        distancias.append(np.mean(dists_i))

    todos_dentro = all(d < radio_maximo for d in distancias)
    puntaje = max(0.0, min(1.0, 1.0 - (np.mean(distancias) / radio_maximo)))
    return todos_dentro, puntaje, distancias

# =============================================================================
#  ORIENTACIÓN (se resuelve UNA SOLA VEZ)
# =============================================================================

def calcular_energia_region(imagen, y1, y2, x1, x2):
    region = imagen[y1:y2, x1:x2]
    if len(region.shape) == 3:
        return np.mean(cv.cvtColor(region, cv.COLOR_BGR2GRAY))
    return np.mean(region)

def analizar_orientacion(warp_crudo):
    """
    Decide, a partir de la energía de bordes de filas/columnas del warp
    crudo (sin corregir), si hace falta transponer (90°) y/o voltear
    verticalmente (180°) para que los jugadores queden arriba/abajo con las
    blancas abajo. Se llama una única vez, cuando ya se confirmó estabilidad.
    """
    h, w = warp_crudo.shape[:2]
    cell_h, cell_w = h // 8, w // 8

    e_filas_sup = np.mean([calcular_energia_region(warp_crudo, r * cell_h, (r + 1) * cell_h, 0, w) for r in range(2)])
    e_filas_inf = np.mean([calcular_energia_region(warp_crudo, r * cell_h, (r + 1) * cell_h, 0, w) for r in range(6, 8)])
    e_cols_izq = np.mean([calcular_energia_region(warp_crudo, 0, h, c * cell_w, (c + 1) * cell_w) for c in range(2)])
    e_cols_der = np.mean([calcular_energia_region(warp_crudo, 0, h, c * cell_w, (c + 1) * cell_w) for c in range(6, 8)])

    diff_filas = abs(e_filas_sup - e_filas_inf)
    diff_columnas = abs(e_cols_izq - e_cols_der)

    transponer = diff_columnas > diff_filas

    # Si se transpone, lo que antes era "columna izquierda/derecha" pasa a
    # ser la "fila superior/inferior" (transponer intercambia filas y columnas).
    e_sup_final, e_inf_final = (e_cols_izq, e_cols_der) if transponer else (e_filas_sup, e_filas_inf)
    voltear = e_sup_final > e_inf_final  # las blancas (más energía) deben quedar abajo

    datos_energia = {
        "filas_sup": e_filas_sup, "filas_inf": e_filas_inf,
        "cols_izq": e_cols_izq, "cols_der": e_cols_der,
        "diff_filas": diff_filas, "diff_columnas": diff_columnas,
        "transponer": transponer, "voltear": voltear
    }
    print(f"   Diferencias -> filas: {diff_filas:.2f}, columnas: {diff_columnas:.2f}")
    print(f"   Transponer: {transponer} | Voltear 180: {voltear}")
    return transponer, voltear, datos_energia

def calcular_mapeo_etiquetas(transponer, voltear):
    """
    A partir de las dos decisiones geométricas (transponer/voltear), calcula
    qué esquina cruda (índice 0..3 del orden cíclico de ordenar_puntos_para_warp)
    corresponde a cada etiqueta final TL/TR/BR/BL.

    Sin transformar nada: TL=p0, TR=p1, BR=p2, BL=p3 (mismo orden en que
    apply_warp mapea src->dst).
    Transponer (refleja sobre la diagonal principal): TL y BR quedan fijos,
    TR y BL se intercambian.
    Voltear vertical (refleja arriba<->abajo): TL<->BL, TR<->BR.
    """
    m = {"TL": 0, "TR": 1, "BR": 2, "BL": 3}
    if transponer:
        m = {"TL": m["TL"], "TR": m["BL"], "BR": m["BR"], "BL": m["TR"]}
    if voltear:
        m = {"TL": m["BL"], "TR": m["BR"], "BR": m["TR"], "BL": m["TL"]}
    return m

def asignar_etiquetas_por_cercania(corners_ciclicos, referencia):
    """
    Empareja las 4 esquinas recién detectadas (sin etiqueta) con las 4
    esquinas de referencia guardadas, buscando la asignación que minimiza la
    distancia total (con solo 4 puntos, probar las 24 permutaciones es
    trivial y evita el problema de que dos etiquetas "compitan" por la misma
    esquina, que sí podría pasar con un vecino-más-cercano ingenuo).
    Devuelve una nueva referencia con las posiciones actualizadas.
    """
    ref_pts = np.array([referencia[e] for e in ETIQUETAS])
    mejor_perm, mejor_costo = None, float("inf")
    for perm in itertools.permutations(range(4)):
        costo = sum(np.linalg.norm(corners_ciclicos[perm[i]] - ref_pts[i]) for i in range(4))
        if costo < mejor_costo:
            mejor_costo, mejor_perm = costo, perm
    return {etiqueta: corners_ciclicos[mejor_perm[i]].copy() for i, etiqueta in enumerate(ETIQUETAS)}

def dibujar_energia_filas_columnas(warp_img):
    img = warp_img.copy()
    h, w = img.shape[:2]
    cell_h, cell_w = h // 8, w // 8
    energy_img = np.zeros((h, w, 3), dtype=np.uint8)

    energias_filas = [calcular_energia_region(img, r * cell_h, (r + 1) * cell_h, 0, w) for r in range(8)]
    energias_columnas = [calcular_energia_region(img, 0, h, c * cell_w, (c + 1) * cell_w) for c in range(8)]
    max_f = max(energias_filas) or 1
    max_c = max(energias_columnas) or 1

    for i, e in enumerate(energias_filas):
        y1, y2 = i * cell_h, (i + 1) * cell_h
        bar = int((e / max_f) * 50)
        cv.rectangle(energy_img, (0, y1), (bar, y2), (0, 255, 0), -1)
        cv.putText(energy_img, f"{e:.1f}", (bar + 5, y1 + cell_h // 2), cv.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 255), 1)

    for i, e in enumerate(energias_columnas):
        x1, x2 = i * cell_w, (i + 1) * cell_w
        bar = int((e / max_c) * 50)
        cv.rectangle(energy_img, (x1, h - bar), (x2, h), (0, 0, 255), -1)
        cv.putText(energy_img, f"{e:.1f}", (x1 + cell_w // 4, h - bar - 5), cv.FONT_HERSHEY_SIMPLEX, 0.3, (255, 255, 255), 1)

    for row in range(2):
        cv.rectangle(energy_img, (0, row * cell_h), (w, (row + 1) * cell_h), (255, 255, 0), 2)
    for row in range(6, 8):
        cv.rectangle(energy_img, (0, row * cell_h), (w, (row + 1) * cell_h), (255, 0, 255), 2)
    for col in range(2):
        cv.rectangle(energy_img, (col * cell_w, 0), ((col + 1) * cell_w, h), (0, 255, 255), 2)
    for col in range(6, 8):
        cv.rectangle(energy_img, (col * cell_w, 0), ((col + 1) * cell_w, h), (255, 0, 0), 2)

    cv.putText(energy_img, "FILAS (verde)", (60, 20), cv.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)
    cv.putText(energy_img, "COLUMNAS (rojo)", (60, 40), cv.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1)

    img_small = cv.resize(img, (w // 2, h // 2))
    energy_img[0:h // 2, w // 2:w] = img_small
    cv.imshow(energy_window, energy_img)

def dibujar_orientacion(frame, info_text, transformaciones):
    img = frame.copy()
    cv.putText(img, "ORIENTACION DEL TABLERO (definida una sola vez)", (10, 30), cv.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)
    cv.putText(img, info_text, (10, 60), cv.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
    y = 90
    for t in transformaciones:
        cv.putText(img, f"  -> {t}", (10, y), cv.FONT_HERSHEY_SIMPLEX, 0.45, (0, 255, 0), 1)
        y += 22
    cv.imshow(orientation_window, img)

# =============================================================================
#  DIBUJO DE RESULTADOS
# =============================================================================

def dibujar_estabilidad(frame, vertices, distancias, puntaje, es_estable, orientacion_definida):
    img = frame.copy()
    y = 30
    cv.putText(img, "ESTABILIDAD / TRACKING DE ESQUINAS", (10, y), cv.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2); y += 30

    if orientacion_definida:
        cv.putText(img, "Estado: ORIENTACION DEFINIDA (tracking por cercania) OK", (10, y), cv.FONT_HERSHEY_SIMPLEX, 0.55, (0, 255, 0), 2)
    else:
        color = (0, 255, 0) if es_estable else (0, 0, 255)
        cv.putText(img, f"Puntaje de estabilidad: {puntaje:.2f}", (10, y), cv.FONT_HERSHEY_SIMPLEX, 0.55, color, 2); y += 28
        estado = "ESTABLE (definiendo orientacion...)" if es_estable else "INESTABLE"
        cv.putText(img, f"Estado: {estado}", (10, y), cv.FONT_HERSHEY_SIMPLEX, 0.55, color, 2); y += 28
        cv.putText(img, f"Frames en buffer: {len(stability_buffer)}/{STABILITY_CHECK_FRAMES}", (10, y), cv.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1); y += 25
        for i, d in enumerate(distancias):
            color_v = (0, 255, 0) if d < STABILITY_RADIUS_PX else (0, 0, 255)
            cv.putText(img, f"  p{i}: {d:.1f}px", (10, y), cv.FONT_HERSHEY_SIMPLEX, 0.5, color_v, 1); y += 20

    if vertices is not None:
        etiquetas_mostrar = ETIQUETAS if orientacion_definida else [f"p{i}" for i in range(4)]
        for i, punto in enumerate(vertices):
            x, y_ = int(punto[0]), int(punto[1])
            color_pt = (0, 255, 0) if orientacion_definida or (i < len(distancias) and distancias[i] < STABILITY_RADIUS_PX) else (0, 0, 255)
            cv.circle(img, (x, y_), 10, color_pt, -1)
            cv.putText(img, etiquetas_mostrar[i], (x - 15, y_ - 15), cv.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2)

    cv.imshow(stability_window, img)

def dibujar_cuadrilatero_y_warp(frame, puntos_con_offset, num_puntos, area, offset, metodo, warp_resultado, orientacion_definida):
    estimated_board_img = frame.copy()
    pts = puntos_con_offset.astype(np.int32)
    color_cuad = (0, 255, 0) if orientacion_definida else (0, 255, 255)
    cv.polylines(estimated_board_img, [pts], True, color_cuad, 4)
    for p in pts:
        cv.circle(estimated_board_img, tuple(p), 10, color_cuad, -1)

    colores_metodo = {"contornos": (255, 255, 0), "hough": (0, 255, 255), "reciclado": (0, 200, 255), "fallback": (0, 128, 255)}
    nombres_metodo = {"contornos": "DETECCION (contornos)", "hough": "HOUGH", "reciclado": "RECICLADO (optimizacion)", "fallback": "RECICLADO (respaldo, sin deteccion)"}
    color_texto = colores_metodo.get(metodo, (255, 255, 255))

    cv.putText(estimated_board_img, f"Metodo: {nombres_metodo.get(metodo, metodo)}", (10, 30), cv.FONT_HERSHEY_SIMPLEX, 0.6, color_texto, 2)
    cv.putText(estimated_board_img, f"Puntos silla: {num_puntos}", (10, 60), cv.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
    cv.putText(estimated_board_img, f"Area: {int(area)}", (10, 90), cv.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
    cv.putText(estimated_board_img, f"Offset: {offset}px", (10, 120), cv.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)
    estado_orient = "ORIENTACION DEFINIDA OK" if orientacion_definida else "DEFINIENDO ORIENTACION..."
    color_orient = (0, 255, 0) if orientacion_definida else (0, 255, 255)
    cv.putText(estimated_board_img, estado_orient, (10, 150), cv.FONT_HERSHEY_SIMPLEX, 0.6, color_orient, 2)

    warp_result_img = warp_resultado.copy()
    h, w = warp_result_img.shape[:2]
    cell_h, cell_w = h // 8, w // 8
    for i in range(9):
        cv.line(warp_result_img, (0, i * cell_h), (w, i * cell_h), (0, 255, 255), 1)
        cv.line(warp_result_img, (i * cell_w, 0), (i * cell_w, h), (0, 255, 255), 1)
    cv.putText(warp_result_img, f"Puntos silla: {num_puntos}  |  Metodo: {nombres_metodo.get(metodo, metodo)}", (10, 25), cv.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)
    cv.putText(warp_result_img, "NEGRAS", (10, 45), cv.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 2)
    cv.putText(warp_result_img, "BLANCAS", (10, h - 10), cv.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2)

    return estimated_board_img, warp_result_img

def dibujar_lineas_hough(frame, lineas_h, lineas_v):
    img = frame.copy()
    h, w = img.shape[:2]
    colores_h = [(255, 150, 0), (255, 100, 0), (200, 80, 0), (150, 50, 0)]
    colores_v = [(0, 200, 100), (0, 150, 80), (0, 100, 60), (0, 80, 50)]
    for i, recta in enumerate(lineas_h):
        p1, p2 = recta["puntos"]
        cv.line(img, p1, p2, colores_h[i % len(colores_h)], 2)
        p1e, p2e = recta_polar_a_puntos((recta["rho"], recta["theta"]), w, h)
        cv.line(img, p1e, p2e, colores_h[i % len(colores_h)], 1)
        cv.putText(img, f"H{i+1}", (p1[0], p1[1] - 10), cv.FONT_HERSHEY_SIMPLEX, 0.5, colores_h[i % len(colores_h)], 1)
    for i, recta in enumerate(lineas_v):
        p1, p2 = recta["puntos"]
        cv.line(img, p1, p2, colores_v[i % len(colores_v)], 2)
        p1e, p2e = recta_polar_a_puntos((recta["rho"], recta["theta"]), w, h)
        cv.line(img, p1e, p2e, colores_v[i % len(colores_v)], 1)
        cv.putText(img, f"V{i+1}", (p1[0], p1[1] - 10), cv.FONT_HERSHEY_SIMPLEX, 0.5, colores_v[i % len(colores_v)], 1)
    cv.putText(img, f"Lineas H: {len(lineas_h)}, Lineas V: {len(lineas_v)}", (10, 30), cv.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
    cv.imshow(hough_window, img)

def dibujar_intersecciones_y_cuadrilateros(frame, lineas_h, lineas_v, saddle_points, cuadrilateros):
    img = frame.copy()
    h, w = img.shape[:2]
    for punto in saddle_points:
        cv.circle(img, (int(punto[0]), int(punto[1])), 2, (0, 255, 255), -1)
    for recta in lineas_h:
        p1, p2 = recta_polar_a_puntos((recta["rho"], recta["theta"]), w, h)
        cv.line(img, p1, p2, (255, 150, 0), 1)
    for recta in lineas_v:
        p1, p2 = recta_polar_a_puntos((recta["rho"], recta["theta"]), w, h)
        cv.line(img, p1, p2, (0, 200, 100), 1)
    for h_r in lineas_h:
        for v_r in lineas_v:
            inter = calcular_interseccion_polar((h_r["rho"], h_r["theta"]), (v_r["rho"], v_r["theta"]))
            if inter and 0 <= inter[0] < w and 0 <= inter[1] < h:
                cv.circle(img, inter, 4, (0, 255, 0), -1)

    y = 30
    cv.putText(img, f"Cuadrilateros validos (min {MIN_PUNTOS_SILLA} pts): {len(cuadrilateros)}", (10, y), cv.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
    y += 25
    colores = [(255, 0, 0), (0, 255, 0), (0, 0, 255), (255, 255, 0), (255, 0, 255), (0, 255, 255)]
    for idx, cuad in enumerate(cuadrilateros[:6]):
        color = colores[idx % len(colores)]
        cv.polylines(img, [cuad["vertices"].astype(np.int32)], True, color, 2)
        texto = f"#{idx+1}: {cuad['num_puntos']}pts/{cuad['area']:.0f}px = {cuad['ratio']:.5f}"
        cv.putText(img, texto, (10, y), cv.FONT_HERSHEY_SIMPLEX, 0.4, color, 1)
        y += 18

    if cuadrilateros:
        mejor = cuadrilateros[0]
        for v in mejor["vertices"]:
            cv.circle(img, (int(v[0]), int(v[1])), 8, (0, 255, 255), -1)
        cv.putText(img, f"MEJOR: {mejor['num_puntos']}pts / {mejor['area']:.0f}px = {mejor['ratio']:.5f}", (10, y + 10), cv.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255), 2)

    cv.imshow(intersecciones_window, img)

def mostrar_mensaje(ventana, frame, mensaje, color=(0, 0, 255)):
    img = frame.copy()
    h, w = img.shape[:2]
    cv.putText(img, mensaje, (w // 2 - 150, h // 2), cv.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)
    cv.imshow(ventana, img)

# =============================================================================
#  BUCLE PRINCIPAL
# =============================================================================

while True:
    ret, frame = cap.read()
    if not ret:
        print("Error al capturar frame de la cámara")
        break

    gray = cv.cvtColor(frame, cv.COLOR_BGR2GRAY)

    offset_actual = cv.getTrackbarPos("Offset", controls_window)
    hough_threshold = cv.getTrackbarPos("Umbral Hough", controls_window)
    hough_min_length = cv.getTrackbarPos("Longitud minima", controls_window)
    hough_max_gap = cv.getTrackbarPos("Gap maximo", controls_window)
    recycle_tolerancia = cv.getTrackbarPos("Tolerancia Reciclaje %", controls_window)

    # ---- Puntos de silla: se calculan siempre (son la base del reciclaje) ----
    saddle_points = detectar_puntos_silla(gray)
    saddle_img = frame.copy()
    for p in saddle_points:
        cv.circle(saddle_img, p, 4, (0, 255, 0), -1)
    cv.imshow(saddle_window, saddle_img)

    # ---- ¿Se puede reciclar el último cuadrilátero conocido? ----
    if orientacion_definida:
        corners_previos = np.array([corners_referencia[e] for e in ETIQUETAS], dtype=np.float32)
    else:
        corners_previos = ultimos_corners_ciclicos

    puede_reciclar, num_puntos_reciclaje = evaluar_reciclaje(
        corners_previos, saddle_points, ultimo_num_puntos_silla, recycle_tolerancia
    )

    metodo = None
    corners_nuevos = None
    num_puntos_actual = 0
    area_actual = 0

    if puede_reciclar:
        # === OPTIMIZACIÓN: se evita todo el pipeline de detección ===
        metodo = "reciclado"
        corners_nuevos = corners_previos
        num_puntos_actual = num_puntos_reciclaje
        area_actual = ultimo_area
        print(f"♻️  Reciclando cuadrilátero ({num_puntos_actual} pts de silla, tolerancia {recycle_tolerancia}%)")
    else:
        # === PASO 1: Sobel + Canny + dilatación ===
        sobel_img, canny_img = detectar_bordes(gray)
        cv.imshow(sobel_window, sobel_img)
        cv.imshow(canny_window, canny_img)
        canny_dilatado = dilatar_bordes(canny_img)
        cv.imshow(canny_dilated_window, canny_dilatado)

        contornos, poligonos_aprox, mascara_binaria = buscar_contornos_y_aproximar(canny_dilatado)

        contour_img = frame.copy()
        cv.drawContours(contour_img, contornos, -1, (0, 255, 0), 2)
        cv.imshow(contour_window, contour_img)

        approx_img = frame.copy()
        cv.drawContours(approx_img, poligonos_aprox, -1, (255, 0, 0), 3)
        cv.imshow(approx_window, approx_img)
        cv.imshow(polygons_binary_window, mascara_binaria)

        candidatos = filtrar_cuadrilateros_validos(poligonos_aprox, saddle_points)
        quad_img = frame.copy()
        for c in candidatos:
            cv.drawContours(quad_img, [c["vertices"]], -1, (0, 0, 255), 3)
            px, py = int(c["vertices"][0][0][0]), int(c["vertices"][0][0][1])
            cv.putText(quad_img, f"{c['num_puntos']}pts/{int(c['area'])}px",
                       (px, max(0, py - 15)), cv.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2)
        cv.imshow(quadrilaterals_window, quad_img)

        # === PASO 1 (selección): más puntos de silla gana ===
        mejor1 = elegir_mejor_por_puntos_silla(candidatos)

        if mejor1 is not None:
            metodo = "contornos"
            corners_nuevos = ordenar_puntos_para_warp(
                np.array(get_vertices_as_points(mejor1["vertices"]), dtype=np.float32)
            )
            num_puntos_actual = mejor1["num_puntos"]
            area_actual = mejor1["area"]
            print(f"✅ Cuadrilátero por contornos: {num_puntos_actual} puntos de silla")
        else:
            # === PASO 2: HoughP como respaldo ===
            lineas = obtener_rectas_hough(mascara_binaria, hough_threshold, hough_min_length, hough_max_gap)
            rectas_h, rectas_v = agrupar_lineas_por_orientacion(lineas)
            rectas_h = fusionar_lineas_cercanas(rectas_h)
            rectas_v = fusionar_lineas_cercanas(rectas_v)
            dibujar_lineas_hough(frame, rectas_h, rectas_v)

            cuad_hough = []
            if len(rectas_h) >= 2 and len(rectas_v) >= 2:
                cuad_hough = formar_cuadrilateros_desde_rectas(rectas_h, rectas_v, saddle_points, frame.shape)
            dibujar_intersecciones_y_cuadrilateros(frame, rectas_h, rectas_v, saddle_points, cuad_hough)

            if cuad_hough:
                metodo = "hough"
                mejor2 = cuad_hough[0]
                corners_nuevos = mejor2["vertices"]
                num_puntos_actual = mejor2["num_puntos"]
                area_actual = mejor2["area"]
                print(f"✅ Cuadrilátero por Hough: {num_puntos_actual} puntos de silla, ratio {mejor2['ratio']:.5f}")
            elif corners_previos is not None:
                metodo = "fallback"
                corners_nuevos = corners_previos
                num_puntos_actual = ultimo_num_puntos_silla
                area_actual = ultimo_area
                mostrar_mensaje(hough_window, frame, "SIN HOUGH - usando ultimo conocido", (0, 255, 255))
            else:
                mostrar_mensaje(hough_window, frame, "SIN DETECCION", (0, 0, 255))

    # ==========================================================================
    if metodo is not None:
        ultimos_corners_ciclicos = corners_nuevos
        ultimo_num_puntos_silla = num_puntos_actual
        ultimo_area = area_actual

        if not orientacion_definida:
            # ---- FASE 2, paso previo: verificar estabilidad ----
            stability_buffer.append(corners_nuevos.copy())
            es_estable, puntaje, distancias = verificar_estabilidad_vertices(
                corners_nuevos, stability_buffer, STABILITY_RADIUS_PX
            )

            warp_crudo, M, puntos_con_offset = aplicar_warp_con_offset(frame, corners_nuevos, offset_actual)

            if (len(stability_buffer) >= STABILITY_CHECK_FRAMES and es_estable
                    and puntaje > PUNTAJE_MINIMO_ESTABILIDAD):
                print("🔄 Estabilidad alcanzada. Analizando orientación (única vez)...")
                transponer, voltear, datos_energia = analizar_orientacion(warp_crudo)
                mapeo = calcular_mapeo_etiquetas(transponer, voltear)
                corners_referencia = {etq: corners_nuevos[idx].copy() for etq, idx in mapeo.items()}
                orientacion_definida = True

                transformaciones = []
                if transponer:
                    transformaciones.append("Transposición (90°)")
                if voltear:
                    transformaciones.append("Volteo vertical (180°)")
                info_orientacion = f"Orientación definida: {', '.join(transformaciones) if transformaciones else 'sin cambios'}"
                print(f"✅ {info_orientacion}")

                warp_confirmacion = warp_crudo.copy()
                if transponer:
                    warp_confirmacion = cv.transpose(warp_confirmacion)
                if voltear:
                    warp_confirmacion = cv.flip(warp_confirmacion, 0)
                dibujar_energia_filas_columnas(warp_confirmacion)
                dibujar_orientacion(frame, info_orientacion, transformaciones)

            dibujar_estabilidad(frame, corners_nuevos, distancias, puntaje, es_estable, False)
            warp_final = warp_crudo
            puntos_dibujo = puntos_con_offset
        else:
            # ---- FASE 2 ya resuelta: tracking trivial por vecino más cercano ----
            corners_referencia = asignar_etiquetas_por_cercania(corners_nuevos, corners_referencia)
            src_ordenados = np.array([corners_referencia[e] for e in ETIQUETAS], dtype=np.float32)
            warp_final, M, puntos_dibujo = aplicar_warp_con_offset(frame, src_ordenados, offset_actual)
            dibujar_estabilidad(frame, src_ordenados, [0, 0, 0, 0], 1.0, True, True)

        estimated_board_img, warp_result_img = dibujar_cuadrilatero_y_warp(
            frame, puntos_dibujo, num_puntos_actual, area_actual, offset_actual,
            metodo, warp_final, orientacion_definida
        )
        cv.imshow(estimated_board_window, estimated_board_img)
        cv.imshow(warp_window, warp_result_img)
    else:
        mostrar_mensaje(estimated_board_window, frame, "SIN DETECCION", (0, 0, 255))
        cv.imshow(warp_window, np.zeros((*DST_SIZE, 3), dtype=np.uint8))

    cv.imshow(window_name, frame)

    frame_count += 1
    if frame_count % 30 == 0:
        elapsed = (cv.getTickCount() - start_time) / cv.getTickFrequency()
        print(f"FPS: {30 / elapsed:.2f}")
        start_time = cv.getTickCount()

    if (cv.waitKey(1) & 0xFF) == 27:  # ESC
        break

cap.release()
cv.destroyAllWindows()

Resolución: 640x480
FPS: 30.00
✅ Cuadrilátero por Hough: 53 puntos de silla, ratio 6.69819
FPS: 8.45
♻️  Reciclando cuadrilátero (53 pts de silla, tolerancia 5%)
♻️  Reciclando cuadrilátero (53 pts de silla, tolerancia 5%)
♻️  Reciclando cuadrilátero (55 pts de silla, tolerancia 5%)
♻️  Reciclando cuadrilátero (55 pts de silla, tolerancia 5%)
♻️  Reciclando cuadrilátero (53 pts de silla, tolerancia 5%)
♻️  Reciclando cuadrilátero (51 pts de silla, tolerancia 5%)
♻️  Reciclando cuadrilátero (51 pts de silla, tolerancia 5%)
♻️  Reciclando cuadrilátero (51 pts de silla, tolerancia 5%)
♻️  Reciclando cuadrilátero (49 pts de silla, tolerancia 5%)
♻️  Reciclando cuadrilátero (55 pts de silla, tolerancia 5%)
♻️  Reciclando cuadrilátero (57 pts de silla, tolerancia 5%)
♻️  Reciclando cuadrilátero (56 pts de silla, tolerancia 5%)
♻️  Reciclando cuadrilátero (59 pts de silla, tolerancia 5%)
✅ Cuadrilátero por Hough: 50 puntos de silla, ratio 6.45132
♻️  Reciclando cuadrilátero (48 pts de silla, 

KeyboardInterrupt: 

: 